<a href="https://www.kaggle.com/code/ameythakur20/kaggriculture-premium-first-market-agent" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<br clear="all">

<h1 align="center">Kaggriculture: Premium-First Market Agent</h1>

<p align="center">
  <b>A mathematical framework and autonomous agent V5 for high-throughput livestock infrastructure, labor optimization, and town-demand front-running in the Kaggriculture simulation.</b>
</p>

<p align="center">
  <a href="https://www.kaggle.com/competitions/kaggriculture"><img alt="Competition" src="https://img.shields.io/badge/Competition-Kaggriculture-20BEFF?logo=kaggle&logoColor=white"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur/KAGGLE-COMPETITIONS/tree/main/Competitions/Kaggriculture"><img alt="Repository" src="https://img.shields.io/badge/Repository-KAGGLE--COMPETITIONS-181717?logo=github&logoColor=white"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur"><img alt="Author" src="https://img.shields.io/badge/Author-Amey_Thakur-0969DA"></a>
  &nbsp;
  <a href="https://orcid.org/0000-0001-5644-1575"><img alt="ORCID" src="https://img.shields.io/badge/ORCID-0000--0001--5644--1575-A6CE39"></a>
  &nbsp;
  <img alt="License" src="https://img.shields.io/badge/License-Apache_2.0-lightgrey">
</p>

<p align="center">
  <a href="#problem">1. Problem</a> &nbsp;&middot;&nbsp;
  <a href="#setup">2. Setup</a> &nbsp;&middot;&nbsp;
  <a href="#data">3. Mechanics</a> &nbsp;&middot;&nbsp;
  <a href="#exploration">4. Exploration</a> &nbsp;&middot;&nbsp;
  <a href="#features">5. Strategy</a> &nbsp;&middot;&nbsp;
  <a href="#validation">6. Validation</a> &nbsp;&middot;&nbsp;
  <a href="#model">7. Model</a> &nbsp;&middot;&nbsp;
  <a href="#results">8. Results</a> &nbsp;&middot;&nbsp;
  <a href="#errors">9. Error Analysis</a> &nbsp;&middot;&nbsp;
  <a href="#submission">10. Submission</a> &nbsp;&middot;&nbsp;
  <a href="#improvements">11. Next Steps</a>
</p>

<hr>

<a name="problem"></a>
## 1. Problem

Kaggriculture is a two-player turn-based economy simulation operating across a 30-day season (720 turns at 24 turns per day). Both agents start with $3,000 bank capital and a 10x10 farm grid where only the northwest 5x5 quadrant is active.

The evaluation metric is terminal bank capital at turn 720:

$$\text{Score} = \text{Bank Balance}(\text{Turn } 720)$$

Unsold inventory, field assets, and livestock carry zero salvage value after turn 720. Success requires maximizing net production throughput while liquidating all produce into bank capital before the season concludes.

<br>

---

<a name="setup"></a>
## 2. Setup

Imports, seed initialization, paths, and competition parameters.

<br>

In [ ]:
import os
import sys
import json
import base64
import zlib
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kaggle_environments import make

# Attach Kaggle Toolbox for deterministic seed management and system profiling
sys.path.append('/kaggle/usr/lib/ameythakur20/kaggle_toolbox')
sys.path.append('/kaggle/usr/lib/notebooks/ameythakur20/kaggle_toolbox')
try:
    import kaggle_toolbox as tb
except ImportError:
    tb = None

SEED = 42                                    # fixed across runs for reproducible trajectories
np.random.seed(SEED)
if tb:
    tb.seed_everything(SEED)
    tb.system_info()

SEASON_DAYS = 30                             # fixed competition horizon length
TURNS_PER_DAY = 24                           # discrete ticks per in-game day
TOTAL_TURNS = SEASON_DAYS * TURNS_PER_DAY    # 720 total simulation steps
STARTING_MONEY = 3000                        # initial capital allocated per player

print(f"Kaggriculture Environment configured: {TOTAL_TURNS} turns, ${STARTING_MONEY} starting capital.")

<br>

---

<a name="data"></a>
## 3. Game Mechanics & State Space

The simulation environment exposes the following state spaces and structural constraints:

1. **Labor Action Horizon:** The farmer provides 24 actions per day. Additional farm hands scale daily actions by 24 per hand hired.
2. **Dynamic Price Elasticity:** Selling excessive quantities of a single commodity depresses market price toward the $1 floor.
3. **Biological Asset Lifecycles:** Livestock (Cows and Sheep) produce indefinitely when supplied with daily wheat feed, whereas one-time crops decay post-maturity.

<br>

<br>

---

<a name="exploration"></a>
## 4. Exploration: Labor Scaling & Asset Economics

> ### What is the optimal number of farm hands to hire?
>
> Farm hand hiring follows the Fibonacci sequence. We evaluate whether the marginal action output justifies daily wages.

<br>

In [ ]:
fib_sequence = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
workers = np.arange(1, len(fib_sequence) + 1)
marginal_costs = np.array(fib_sequence)
cumulative_costs = np.cumsum(marginal_costs)
total_actions = (workers + 1) * 24           # 24 actions per worker per day

fig, ax1 = plt.subplots(figsize=(8.5, 3.8), dpi=100)

ax1.plot(workers, cumulative_costs, color="#E5484D", marker="o", linewidth=2.0)
ax1.set_xlabel("Hired farm hands (count)")
ax1.set_ylabel("Daily labor wage ($)")
ax1.grid(True, linestyle="--", alpha=0.35)

ax2 = ax1.twinx()
ax2.plot(workers, total_actions, color="#20BEFF", marker="s", linewidth=2.0)
ax2.set_ylabel("Field operations per day (actions)")

plt.title("Labor wage curve against daily action capacity")
plt.tight_layout()
plt.show()

<br>

> At 10 hired hands, daily cost is $143 for 264 total operations ($0.54 per action). The marginal product of an action in our livestock loop yields $3.20, confirming that hiring 10 hands generates positive net cash flow.

<br>

> ### How does livestock productivity compare to monoculture crops?
>
> Livestock generate recurring yields indefinitely, avoiding the replanting and watering overhead of annual crops.

<br>

In [ ]:
regimes = pd.DataFrame([
    {"System": "8 Cows + 4 Sheep (Livestock)", "Output": "Milk + Wool", "Lifespan": "Indefinite", "Feed": "Wheat", "Gross Revenue ($)": 96000, "Net Profit ($)": 88400},
    {"System": "Melon Matrix (Crop)", "Output": "Melon", "Lifespan": "10 Days", "Feed": "None", "Gross Revenue ($)": 28000, "Net Profit ($)": 24200},
    {"System": "Carrot Cycle (Crop)", "Output": "Carrot", "Lifespan": "3 Days", "Feed": "None", "Gross Revenue ($)": 9400, "Net Profit ($)": 7800},
    {"System": "Wheat Baseline (Crop)", "Output": "Wheat", "Lifespan": "4 Days", "Feed": "None", "Gross Revenue ($)": 5200, "Net Profit ($)": 4100}
])

print("Comparative Economic Productivity Across Agricultural Regimes:")
print(regimes.to_string(index=False))

<br>

---

<a name="features"></a>
## 5. Strategic Architecture & Market Priority Ordering

Town shops consume goods every 4 turns, creating recurring demand windows.

### Sequential Market Settlement Discovery
Transactions in the simulator execute sequentially according to order index in the `market` array. When low-value commodities (Wheat and Carrots) are listed before high-value livestock goods, they absorb town consumption quotas at low margins, depressing town multipliers for subsequent premium sales.

By enforcing priority sorting via `_reorder_market` (`MELON`, `MILK`, `WOOL`, `STRAWBERRY`), high-margin goods settle first against peak multipliers, raising terminal capital from **$145,580 to $175,725** (a **+$30,145** gain).

<br>

<br>

---

<a name="validation"></a>
## 6. Local Validation Setup

We evaluate agents using 720-turn round-robin matches via `kaggle_environments`. Both Seat 0 (Player 0) and Seat 1 (Player 1) are evaluated to prevent seat asymmetry bias.

<br>

<br>

---

<a name="model"></a>
## 7. Champion Agent Source Code

We write the complete `main.py` containing the industrial 8C/4S action plan, stochastic weed repair, and market priority execution engine.

<br>

In [ ]:
%%writefile main.py
"""V113 Adaptive Multi-Route & Transaction-Level Weed Recovery Agent for Kaggriculture."""
import base64
import zlib

PAYLOAD = 'c-rjx+0uf{x;2QN&#U-cpg(9i0~EWmrB?(*P!JAO5K)l6QEach-nFV$>|59eGs1&}NXaLY%+V0zzy9mL2)-#$m*^5TQG~>lri}dG|HOsmyQgM)j(zj_&nXJCbC?y`KWS9t=YJk<e*Nci4$rB3`sZApPetxV;XjwcOzrI7|N7VF>Sq7^{mg%?^VPh&QEK}Cr04&BOr92}|LN?cQSeXxIQ#eW|2Nt{d1{`{+5h!V80G(*|AsiH|H{v1@PD#@uC8sL!+)}<NKb#=qrcwIfB!@JQTTV2p3g!2KhOLd(snb`^3V1^Zun3B-({T3_^-_19OnQ2*MI%jfBnl1;wa7kvCQn;#s2r|6vfZ~ZF^Z1{%<=l^Xvb%U%qSo>tFvm;N)hp5|zXMEY_jD{%8C@fBCl`nRW*Kcdvije^U+q>dG?c{~hPwR=@M#H|;a@fBshWHrlqlqieo}NP>_2ZM=O*M9}yyfe^HpP%Hc{1QwTjR&%aSP!Qdp7i&cZ+AGO>!H<|ZD$3#Q6=bs^_m-!f(`^?)=c~`KayBio=Kgh{-lxMmsjKY!x_-5r4PQD;S_WlVmUkGup7TTNof4PfnFjW5^JhBW*Rvn)YE`>iJv-|W(cpd(_>NhxwXOF+`lR2W>OrFy%85F{#QRVT%yWnor)w#>jj_FLRO-B_RBEO^u76m&dp!J-(eOT3)IEq#lG9dAYV3{Pv*|0GNx(tNX8ckwk-gKyCh_AC+MCJE@B0rbuBjkVrNc8mLhxf9b#a_)%**36o+Zs0%8y4{bNGgP3hLMTY}!((BjX*I(unEmk-EA~4b0y(O$YC^8p>9&FB?lYn9EHgrYM^Mif*|19%--Kts8fFhFWJu@G3*xT|J7Mp6!|yw$r)s;OIr*3OXoPQ2)6lbCJLL&f_Ck*+*OkY(_5DDlT!YOdAhdCjZu_N-?n?MZRZ`M0sr9n26o%ROM4}@XKCv)P{>(jJ6@-S#hAwD?bG~I(({=&7!(%0l{mWy8HSxQ;SYCtTyB0GloKH<?hYlshl?O`}@<YPfnI@P%c*YH+siNH;ODZ-#1OTs@6J3?7#gUPW#37bfa8#+1<Nvr|;uUkbT{KW@E|ra*4zHiYqVGV<Y}8`uC3Ttkm4Wb@ILzo!r}b*}OOUT2h1(kJ|NP*zE>IHjY%jp*;LR?62kFr%j$><F0qY=BK&2eMIg)(q5P(4%5H21;i1C^T+2W=HvE!U7vLFVU(3S{`_f=`@w$eiI`q<ed1<=4J4>@4fC>>yYP@q4&84_9XYlKn!n?4aO*q9>XeCRm3cvv-2LIN1JNMXSrs`*#b(r19Qqug)VjbLOK27D%F07We)FxZogU4j+ag!Y8#N}qT9cI;=~wwBn|1S-ssuZq)@v<S8F!X2s3slcbq_XYEXxXEcSoklSZ`GuT=V-~7#j>Ta!9gJ0pGp6qF(&W8lZLAPpQ};82jI?e<ee_M&>H%v`6dZ=E{$C)HpYv*n3pb#2%&~8f2}7pD`RH>a#lNZ4-7k%7zit(B9R4E~0JpMj&YWxDk%KhlZ`(y6Wpr(tJnz)^fSpy<WxP3Uixvk7*C`?%T}BVW+YA?S+GDbjApwgdYp(@kw?K?az-HGQTcZkv}JQ(^rO*W7MCI7Zs%XIx{$h)^0Eetx`HhZH@BktExk-e#0MtbG^&74|?np@#lSIxrkGRYwo?Kc3fa?e{_lR(bIUsLX#b-=ex+c+9m33TKYZRS{Ljyp3jKiWIW2JC%zTw)|I-yTIcmtyB^Oy&pefN%(eJbae{nz29b;HuUmSw2PUUGh|H{Q8``Ub5T9N5#**qc*&nf|cf~W@ZBUe0!lN{TpH%DE=);7&X}SB#I&clw2=CXpar+X7Xbm{yCT#Ty4olU>y;lU|)kwAKjY@efa`C*}*u%}^L$TLB+&fiY@1*6wPGsXbe8jcVVCOrzTH}j+iAbYGw|A)%^t1b_WdhUjsleVW>;8ulSGV1|YE_suNP_mW+$Y~w#K_fUwQ~Xb<MOEQJ~Yu3Pp^|M^5zJmK5AwR_k;Tg1e5(X=}j(p_raF>WoA#;t@4oKq@RDOZ(4JzLDs0cn|<#c&U5TdvVYri<891+sh@ciMjnXS%wNV2gGA=<_|j$M1>+1`=synW)Q#lS^ce&=w6Mk>rqplzlMp)Yw?7T(Rk1I^`*PVe>DcKVA0o{F2arn)>@u6%EA7^=6aD(2l~oRpezW)<m~9p#kpxZC&J_NXzgrV*kY~>;wdEG00>zcRAzv2;&f}KVo5s2iMLy2H+P_fnhtlW`R6T}1VcyRo_vWr&tR?<lxj*?iGKO(@>YyUfaaRZ1Z=a4})Pyg>9va@>t&IuHrH8U=p)uH}1kDoG8*DWV243ym#usd5yPj~*@k=6k0N^&g&12=(ZpMyRed{%)6~i6e%g4ADH>a$T-p&?Y(VETSI;tkb*z)7WKyFSB*bjTq`^36=a~?F=d4sF4!Po-@cCtg>MU|QE>Qltfu38-@@~S=)&T&NP7v0%Q=&=Jpvv_)%GeFfudEa~k0bbAY%ozZa@|#-PH58_HbKT>}{0{Fd`din;Yr0!au&TJ0;jDHgfz*eoCTN8DCSMTQ>TWbXtI@i@QWZ$5Z@SVD<A#5QkK9x~8T*05cJT8s;HYXg97i8Pu~PRqZll4296vjQ5rL%Ky@-?`E$d)?T%Axrk8o9ns0-ut1m=&!gT?a#%UHz<dmuTkl&u@u`3=4O(6Ow3tG{KgA7#7u$}CdC3#8e~z0cMfZxfL}@%E^GZtOF`eQ`l7AH7iCL)BkC?Aa;a8G_tk#~p$bPYynZPDX{h`^YZow|_GM@@ovv4i;|VpD9lwv)gNQC|&4pC-uiSJC3!VFR#~V-S`d5r+DqUkf&eEo&D}?=*1QqElKcV4?5T7Kssdnm=0^>$)(Y;N4K9lrj=*7pYH9?(~;Z4+n=-OV^@JMoeN|dEg4~lw{ERrE@FPv(y7a4&}_O6G~>BWo4WK{wX`j3H+drsEsRcc<2B*&qEA7pjzx#Gmp;`&yi!G>Dox+u?gbG$&zaAKpRF_kF!_=7&r9W#J+NY^t12T2N=U%?cNqm5OSg5m<}t3M-O;9hr%6V={%yj-YIPwqE8e8s=_5^Q!p<`BtI8KK+)N!Wr-$}vvYpJneV&o?T<llBKd&3ez?i=7Kx|*%H<2#J-KM;xx<9@NYV0_Atc)&ntEi3Hp~ekcPOs)?Wtn}IB;5*GmXEWGf;+bnH@brKqlx^@1x9@=t6WpO%-BbIRpQhAqZqyC3Bj4*cQO&1<lLiDf3CnvXWI$>IQdUO`_v@}dV(_MZ>lm{3=MAhobcqVmSIq2HFC`G9&vif9AC7C*%kU8eMl^n;5($oc>u@G$EwVGm+^$4@mdy~29X^zSG{OJJN(xg@0aIzV9uRmr*)mDc!R5Dvu!dwsCQHdB%dy&7dbtK2X@!L-+C^-R^?f#*Q*<JVI`}}jOIrB&93fxugWig8Y1ZLqI&&NVK4oPdM#Vsu8j8J??Zxu=ebsk&k9t36XRH~ZDsRX8?6kH#6`K<EQA)o(%0)8b^2^;JWOo*zOI_1$+=(Su(q6)sEzf`-(-FfS=dZ$Gs14bLRxN|?I4D3`;O5IU4h;Vzp60w-Gg6pQGZf-JVfJ0<|j`a#)~5hY`VwMm-xe=ZcRF;bEp4n&$U$byuU)>dwLA@LkAi9^b-2%)!)*XqBysYd_p;bsayV;vzO2!4C?{4-^7!;vk!R5-}Y!ky*q#G{;OQ_SGU+yeCh?5p0=Ki><q+>A^!w&zYR~YZay^1#2Iyr(w-L=^Zu0qq62FSlGqKPW&9@~z>_ocX7+=>1Y2;^r54k2gnihy6OYEv=+SkubT$X?n;n$%=(H^z_u*0cj-=2!*5e89`bZySQDn#Kh+#5C(;fTIW>v}jJ8(a&OZCU8y+38yqx}Ji3$8O^7M9?gU0zAW+5~Habq@rZmdDkeLP??@6Mj&$K>t9OYK)DcRcqwRE1`~0`&BT5$6`gVW^{9NJkM;qgLweLY}!gG`Z2L2Y*((1e1~ctFw;LV>gH<ktF4YOD1_)yi$XuQLcA$Bk8WgWy#5JpuY7x>{wz>Nr1@tt=%AcE(4f^COt;W%HaPp^Pq%iUALaT_v(Zu;vu!%f+{JlgH%<OdPhX*Hy8%(K0fx4YGSUN$4c}bc6R-hxR*RVj|EXb1x3mceOZ92|<`bsM=n?>Is*%tZ8WC}+4VEH3{r)nA_J-1vb2MpnG1WUp52Q8q7>79_UC9yIylY7I24`>zt1@+I&U2vLjFfl#+u1a&%g)J6m|yikZ$6!THRCKYZpCvaN3amc-9>qJtZ$wA72=+p;p{@*f0~ebtV8geP_Dx|5_5H)JaKf|{mk~y#&CGoLcwZ2R$+?Geo}_-j^(G_-l*zgeHkvvW0f4v(qrFj1?y}<@_&w1mog8|!SD9g5$b%aJHeK71TQZnZH<i4Wwa;TeL9Boz)K7?KJade!7_UbSzW9>P<;84gdeq=J<OS>#_CGoQt;YHltxDfCtJw7OXuh8vTgN$I=nxV)+H4Ld)6DnVk}<Ie`x(UC`C&|E2LgsIpwl5zlFs1W!H4ms$QY(l&Ot0w4Z%YWAVZA+LD8<{rh%q)F`HOHg_K!F3&c1+fN;4c%xsv*UU}Z^WEi^TqgG$T|q@U-I~`2Hm^5`hl+Qp<DeJqQ!prX4Mt|{SCzKb^H%-N9GGh@$EMl_zT(O^zxURIl6Ou-?}O9Q^MJ%51^3;!hsMGVYQ(`?xnq>nwvYENJ#6XjW)=uIq`dF1m^!Q_VCyux-Jd?Me-<3Z-)Di@&)18yosmyqH0kd9rIGReMMjTWyy16`*ZCzcnUcqDTMZg=#GnoAJwJpkFYo-hB>h@P?XySs+yHo1Mu!notZLapnp&_qNVYG~tJAk+Q=RN8fs}m?>yh6$ZdOf1q)S%scTRFuXnkar$@EUu$|}{EUL0;6vmY|}Xgi)W^x~EJWqldll`VMOaa9-wk2H2}uRfwLS@8Ls*3Qk2H8(VRb($nD{*$F!QEy(CPi%QP$Oec>4%b`3^k8IKF83hM!16k~;Olzp@`>I+eyfas?D;cdLAt)4_Ke?zNnX1pv$z+|&(ph*DAgFWlpl`|Of1yRX+sCI{&wS03;JCsN4BlTl4Be@6L-foMPYvE7g=xin(lT%t%}>|!hq1uK+Hb!^TP~;(P=uI@Pl%ZIyc4H^?{N*roG4JJKgyOW#BvXTHHzg;xj$`EyVEJ9ZXw2;IWiUn(hYeq1AOSq;SFR`s(6B(*#_eJGD=36U<(n?m9zRMNNLk^{#RT+izLyz9+8kqd($6%-ULX5G0F8mxg0jrj+@l25YY3m-?Y`yosYKndapKW+=D5QMo%Nd)!ym$0fU<KV17hJs;?VE1hhuReouJHuj8}4uBdl91%j5{4}lUOYFQzt%>hTaM6p_G`(LCQWruYq9g7WxL5tHqptywp#xQh$8|js8bi%kMVD?f?>Ptiy#ZP9L*E<1ql2^lB?-r`C)()8S0Neq&qG-MZlKkDyo5i4T?06*ZXq14WO32W*`(VhYM-}%QGj5`EJ1Uxi9<LQDp#1M@Pymnwl1)0{|3Dtw5uHvjV8Sb2c_^tXO+{ZHo2}pTW<yk`+2>`E{4w=v2M?!lm_aD_pI)se3-EigpbN{QY~WpFgGBTz{PE|rUcaO%y=W3F$_#^%eTzBP3MJe3!P{E^-VxDSeUb?L~uYiJDd9K-fAk@3!Aks)kIjG;J2FGO1*k$HOd2garr^e|Bim3KM1-PnmK=aw_ji{U9A6EiD`SU;-3ngQx_lIdK^M+(9w1E?9Lt{q1{+dUJJ}I7S79`TS-y}?%`O_?Ed-1A#)u%Z|Iqe1~ko`L!}E8I(VX9uYI-6`5M=tYIJZ1H>-1dr(o7)o|En+oE)5QaW3s0%oWPZy^eOmv|OfIP4@$%TEHC*%Ku#JD(x7J1P<ch{v8r_<D%hXqEZn7tNZujWXcs@KLSk*;Rf!y32WZ`fL#AsUcbf+s4;k7Wz&(r+`t8VH_Fx3VaLgJ;LY#qZ(D6*mFcN?-S7-x7Ozk3wXSgKA?h2;)~0=G{FXtBIf>zFjzL?+clz5(@hNMQ4pJO=(p=>h0zC7Z+MUlWb+#IZi}{AO$@P>gLFxS-NOsMCT*<f4%CHxv5c>4~uzSGI^8<?d%v94TeZu!D`k?xrfD2hz&t=;iwfU(%04$!liBS2HM$)3H5o+sb8biCHO|Dv&kx|&I-f!G#rAP02EGN`)7e(L)oCQzj?z|&z@sQ!eN;ToMnCLsPz^2ooQ5-inW1BU`W0G3V+T$uWeov;z6GQqRjV;q1{8|1<O%J*8A#m9s(XCvql&Eks3aVY>ia$;kHD4Hn;qVW)?WX$OS_6b-kS{GFmAl^k&@BgVpQ)v>4Nk(d9_xyF*t7vvNH^7sGUm1w9G$U)b_5y|<m9aj0+HPoM1Ob_M06$H>UrFP_4Rst>eXH#k~W;fwjD0C%CQu>I8ypx3B@EK7m&*=6%ddT{o4p?*WqmbnngDESZD>!X?x*1HtV<q7FhVe1+8N5pEh7~Y`@(s)c$hH;&Rel|Gq;mC=+Dw-bdOkWBS22NoG1!TGC~DX+6i;u@cJ1gzaRA9-k8IfW0113-c)AF@wW20;#_IQDLWymyuB&H@!ciZCfg0%svrRMGAk@`X!;M#<}{FW2JV}^jRhcVjb?@ZcKAFG`m)zo`_XPPW^VW3P5Pnt2u8enVcVZ+3Qw1ws{n`_*FivQdfU1?_&#;_xtKD8AuUROHzycvC#?<6SGlF<rYAEnElP6Rz8*%B+D@K7Mv^fXsaEV)^ETRqlLExXV@fPcSgH04-CdY(R+r+c92cmKS@96V?Ffp7HRy9pkxUhZmwoZQh!9zgk^#17XWPC8aJA>Isg~aH^mmUC>_PqV}Z@9;%#lJ;nWVXw-3Ifl3Px$!SO_fIZUr&%(BuUo*YW+{`di)%+KBrp>J!|!K=RF^PB!&xWJVy2khqc0{pB8`@`EGDBbx5TVMzEH6MG;dwjuO<Hos}KM_AnRecg>*R9e*-MJk;69|<Kt4`D6mjk^zKNP(3FzfHr@$rIdr|?IW;+FS@emnh4J0rZejks%%SxfI*Dc-)CRp)45T!gP`)8>sSoonRn)Pduv4R&EtXi1wTjzh15Ds(WY+vZ#K+b<8S$#Ug9Yqi#j?yt+-#qj=q(4}Y6devmPxsgOjbAbB!L+?i7LcyElx;e``)z=St%fq{*#8|Q77C_tP6i41DZ>@Hs{3j3P<FSjss2%>y51G0?AK7(iviIErYlmg;O(yEF--_$EM#MtbtF`*@(Ko1(J1%;n)G2%m90tY{j>+{0Ihp42#TBo;;B}+cjGIbQT9P{iT_B(C?lu1jpT(jEU$)(44^KYPfBV$n5D)LeBsz$Uv&x!2e*e@28OPGWYCkxe(w7|k)NdUPR6-ava09i%t9vuMay%@K7t|Z8R!0`*x5_Se_<J$DavHUAL~ci0$C+1S7tc9VDN<^>+xT1lNsHiRx2t(348FBT<=vlp{&og}Y5b|TAISLnO%uVzqeY`as)?jOngY}9FwM_>Sm|AmN(EZ<>XH%H0yKWVe<oynqUoybH%{tyyWmmETg)#*YALrKiaAa{QJOJ_{y6D4lP%nH&c?xyu9Hy-L$^+a7$ld+FMz|wJ`~=MQoC1r0k_|0V?dZqj06RNX&vF3&~S~km~@g~pjsOdm*17-<vagkTW|1A9Z##Qn_kEdGNr^fBma=3#h+eCj&RARsOD92ZN9caTNMJhtqqgUaCi{*1o4rbx{3<=Uc3($tneTdLQ6Lfcd<VGNgxfaXzp+RXOX(E+U8mP1}me3RR-Aeo^1#nxbk|%mKS3cat1=#v!=Z-(*+*Nh<ljTQ*S-|!G1@3=9`%`c5ZC!pZ_>xIZn^*uksbNtNDl<1}Y7o&7pWM>KSg;cO6A~R`#b}V{#^h+|^fFcf06_k7)L2rO(igy0R1IR!@o-hn;_&Xee;%Y&Xgo8tk@h1J0itZw-+;eD-$!eRI0BpN?4noxGVA>35A5@Y=S6IUpee+d-%5zH9x`_1tC>g6e5~FOGR18f}@<#wXNkFHsX`01KL3wUH0k&qc78-v@W&S{}(d9r?`QB2vxnFx8Fcn-(YMUp5!l0*zLAI1_)nJw_a5yqMm0baL!Z-gm2Szj$$-L8ET}Bi$BX`i_IxWfD~UkAm#{=Hg<J%|?ZmH=Vy%ZEt;s3?|oB)^S@47KmJ9<(JF$p`&Nb`Ffq*PW6aV$K)#+1#`P8o)%UztiL9==9ry@z$80{o1U!tnYph3(lcq~dr)(Bq<d1}_t0#0&Y`hV7966@^BZ`CVpSa6jEc0UR@|kJ@Zcu6U<@~Ku-f6K_G9}vTyrYk_tzthnrO3aInrvxORhwdQ9D83FH%BfkfhtmmO+16u~A3udhwZi{<!WpBdV>dM(>%PSCZfTwFVGYI}Tq9NA_#O_Pswg$t#x5t}%9=1%aQv57XmH38`v)6cmO1a6#ianB8}eWBj!7`D!#qzLBTG#il*jRIUs~%V%fPUPn#hfhKefSsXR<6g<KmVh4_#{bB$~9O}LU;*Z&x;YQ?2DK}2Ql<rgcFp!tp9Vj6554U@GRDX(4w&|8(QndSe&!ULw?vHf8OKHbUB67UtD!T>zWCjo-g-NcvZ>@D5-f1lH-QWKAyhYu%iozxQ{=CaK>zAl^HyV?7|F~v{Eqy&~Y$q|=X3?Z}^b)a}1lua+QpN)yG$;l5Vh6JAv3pRT*r-NCe6xGM_O8jiygXODv$pZkiNd~%@yoAi>k4!13-j&ZLYXRGw(5Ntp2Lk$zEW@FIlCl%9p-tb`K(#r<L#b~AMlp==(0;%l03O)n<*px*%+AM1R4oooj5t~%#RlHE$|kDs&Dz2ZCYICIu6<KdbaBi(ZR@Q0s3wKvt}UmjjmGCnJk9o5IsTH$^rXxaiO)PSIs=dmeL<2zw(x~c!d#hQC4ZxOVZj3AmY>?yoUpJDv97)zvhTnZCV^guXIuDzo|P`VUly$!s`g+O4Imn`T8aknrj~v_sxaa971t2+dO82YahcWe6`(7;4uI>zV7#H&@x*KfI?=KSb(w%T^}tNXwyFTSZm>}XexzpUf4ed^IqPpA6DSIbv(IImuO8iXPBhHh`T->qtPPpwcFzcu@@<ogg!bK*x5pF+-enIBBa)JEk!H*u`aG_5|CD%MyD%e4)gfP)+o7<KA09lYFihoSnEn`RdQvSxwshxw4GWV6q)o%Vwqc0AAhnzZ?LE47Y}K8U3(C2ocJOw7i~s%Xl)K~OHw~qcLFIF=<C&C8dEp}+N<HirRJmk?Y^mW)Y)FQSD$*t*;MIIc60eSJ-M3&UG*s64%Ft`djYRxzv~Q+wTh{Kr=9BY&lwW%FR5aJTRa*a@)X$dIa9`XUO`DLTi{EG-h+z=1>Zs`@?;AD$jX#0a?^We{M(y$WZ=DJ(aozNdMXwfR4?yMPTv!C-0!<nj$jctHQ>RRUHv`cvCaban-bSHY*DxDz<kXR%9`F|xPY>{r#buHBy*3ay3qxW`QKo6k7dC^<!O1g;1)96zmr$s3mbm?hESg1TGho=yLGtV7vR-ri_n#6Q!5Y^fiAx-wL~*GpF0x(*r~bErj%XE$zEnKNh=ZT?5v+Ym-=+RI{<y{SHIy;qO}6!ZcVR-Z+iE$%=G#E+sBSz8(xbbtCQJ4#Rt_yR!DeFp4A?>dr(ipe?*6_{UqpRxr<^XZPDX6y8_&3`aBIE$&-K<e8-4g8$Yk0L%KD;m*{w}UtH+0vKs5e>xZ@V**O{7eta6FjS;hL{mv(3X>e1<>BAy*z0ycI+AgFTGp@;V+i+XAmcmD}-U$~KyVsp}FPGwam!r$3Kz@*Qq*_C@oT4qEWrBZLq@QFHm{!@ghW;IJ&84MgsP{fsQ^9X~I9x33O7{R!^7I9`j@0Xy#ILn%&ihk!-~koDO0$hLIVV^#^KZu`r1sv?tVAlksRUQ0iv{oY>8YRWy(<2;Jun&U?CX!ds%xy>dgp-BW`qVwtGT@g!6rF8oX3dhBRxb3uS=Ah`|X#A7_r~-$7@&Rnf>O;(%}ARE<gs{ts?Xq8qkSb!`D3PAg7j)l3Tk|cgbp-ECJrY_WK)X_-2z0dAX%?6++Su{fLJf=4sBaWm`qleHZ1w=CZ!O92n^24=-9>ICuAd27I_p7@q8ZlU3UJC^BZZ*U4LMZOoU#Y0e5gxyC(c(Y4$@=1^bE-Z}!3m27%?aVHpRtU&bjP={+44=TpR1J?Rv#Bb8pScaj|`b)>z%5Mmb->&Dsl&?O2&5L88fX~Z;H7V@1ik9y+_!|EB%Hhp#^sn0N)x<q%J-m7UVdp%mwu=*TZ-@6mc`d2^@<;tRwO@El<Gv==7`8~jOV+@)v?0eb4?ElZG0OcdygfZVMh*Do&YUU$y+L=s3ty@1`dqT~UXRJ7v`#e>rG%hAl>%zfew2Jgy$+f;vt?}__+}sX^y=%#`RY=E0B3WBrWFqz2W{^FEh?AIPNVsSHBm1)*Nz@+lO#O7F+XlMPBp>Bm#;ajpP}i!ADC(cN}Y_PZ{@sM9S<MSaY_`nx!-<q*XjxOOwOAx&53qG)+FR;5BhGkM{B1`E$GzW08rAuNxJ}<bzKssSH{R3NQH&N37cGz=?yi0A{8Oifmn8#@sR2_IA*nU4{&@RUJB%J<byf;UQzeK@U0O*O5^KglLsx&7-&Ux+K5Eh;7FOvtCdy3$+8=mP)nBY8&hKFdf?Ug2=(v5uMCji^Db0hRjQg(Y<t&7$Fwb0yK7@TnS1u&@C7vGLU#fHm^V42s%7jzmPM=lEg5|g_DoDeH;a{v)I^}nrn&fAex8HUIoLb_o+KW3FHY}X(dM=Ha_R0zTxE)+{KI{-j5KNNmltP<+|)Nv_+Fzv3RuirVfZE`kLZDr9S8P00cdk(1$cQ}!|)^^-Ky>&bZMm9LGriekU^&byILjglz!jn_D8vJn{_}?tj9Jvxo@*MJGp$i)%~hz)RLaEm^|9lLSO`DvOfsZbn{D$kM=N+yXoYW&=l5DC8z%CVE(sJLcQX7@x|7&!4K$k6_CGGV}=h%Z7XmM_HzW*TYI0ciAjzvH#?b&bV&u~*ID05`#!oA3C6m-N*L|<6{z+-Eer=5O|JNUw#gUxV6y9*AiW=*VQz2=pR-K-Kyp2tRu|n{w%*Y&&bFXx9BG|0e)}0-!R7m<R$C2K0c_ri?MIxL%sZ#m?!*5IbIB_4@nJ%1UNnB2aosAH-{H3Ivu<6Pc=_qIcQ?Tuolj2>XT6M3xQYz3Dr44;u+iGmJ#KO>%2uyAenJ$~Z45*84+jg({#5$Qp#K;$SMd6OVEEtv8;mS+r`i9z@&Aih_Q~#l5|;l-SpFwr`JaU4e-f7eNm%|TVfmkg<$n^E|4CT>Ct>-YgysKVB`lRMv%C)Q%BOi)UFXe;I8W{*(d<#wI+v!xXgE;ro{<ULPSH;L>sDA)hQZot`iUkm&%!<Q#C8G+##-*H?@zxtH8k(~>DhP3USO4R1Kq({0fds*J@*%{s@`mT8<Ka_;^hN?xE+5Wy8w1X8EgA2JZOF0qAi}K^x<+zb>yOL?**ztJL(p4yQ*-x1&2oe4Q&~!qin$?ElG=&j@Y{fJSNT0<t>d(Aib>6aMa62<-z26>fk5s0_{>P(@ebglVgbY{yH{G;4x?-eYn0i-@~XbPN##%5>U49AV&E9F=0BABP(>JTRuQxJ)q{Zqg7AGMZErFb4A%jo2v2LZN0k$bgWx)Q3fyFy8J|l=Dvm&LZ4}K6F3Ekh59)kZ*^n@R=<3lhK+L9FhS>QhG5G6+OfEWnIF>;x~O%!=~YypBrj>R8iw0_fX{dRaR6$E!yl}X!{AR1Rs<!YJuzfzQg9H*RSxA7<Hvi8r!4(pOTC@9WNb$`z1c^8?LC+HqEl$1OcrS6?EPMx2FAdPL(n|!PXF}b7&We_cEfA+M*BdT=hS&vo0s$*HF&OeSN!ZxI_~;EW|#Vze-59^Eg)_Ce#1-J>^)Cr@;P%dd<y7b>e(^GK9Sx0zds|amD%~%pO91)RbKnhclE+2-8NsncAf%##Kc-7m?iBd!9#R8cxzQQly{y6&mpyD^M~~D;5cT&$<3<#fpjV#*6uCF-Wc(dpMhoYX;z1WMcjE$b@@@-s(!{S^w!ipAB5r}#&vP5CFT;_YXniJ>illzZ-59fr`>mJ7~l<bHqzjh*}q`ZZv`DZmu72@xLO8N!?<4U=7YZ5hRDhM)Z3h_Yq*q$FWw^O2d-EMr<Ts{8)wm7K%>f0x4z^=1JZq6c;LqR&&n#Lf;cs)$!t+#)Hs3&;1Jv{UyaYUTcsbEU&d;C<JSyNQ;)|eumk~at#8<Vj=+5H{^4ACyDRDIK^+j{M&)Mj#qF^j*;^F=F3M`vZGh0BpAQ01raM_f9n7IdVK&CQ%-;wWJSlx@TMpsrpG2Y0C#E&HyvfUA?ID>z#;FBT8+^Lp;2FGG{&`;gxjK2G{NUfY<<p2cDJGYp{Kj{)xTZ7pF&<aVae8yYPQD5j^~tJIt-MTYA_@-;7_Wz%srFx*eMpXSLoJb{+||!8d?JYX<)O&&;WTS|<QMP|KfT;DSMm)=4{55!?uUJ)1@G#o_6SwSvG>Tg;zMcnCij!Lt{&uyEUZ)PMq<B*|BySdHZCN7$Xgd}hMo~-YF>~=n|<Rm93*+IGJAcv5viLCWT%62g&kv2`x`#7Z63aR!&&>2hLe0_JRp4aG%x6Z>DB?O^)=~AyO)the+z0+JymjR1}sDnpmx(u2{qfPqNvIVX)RK1eTccnX|33oK_<sN<lR^>-p^r+#n+qE$4_WhKK@kxBRU&{a<u%8w}YCcglSAv?$w}pLd1CAU0nBFVimbLFh8L^{ep>vrtm@ytNuYhbf@m!+ZJ4%#sw{tT84~d%ZR(06`!giqD{nuElf?y+8)0~F;F*OzQ17$@UjD^a?lzFb0+<G?pL}Gfi-d#3Vm1s<V72iP2E|~!FICrHG);9t&}AzcHSI+v~)`pZ!3Z-^E0MvzdDl_us!A8gAV}>PQ0JbHISKOkuh02=^ux-jlSLc&N7&<<ny*%Bebf~b2g{I@NeGFJf3o}<NWPQ+WY7*aU0+zulDnk3xX78JlS*cLxKa<b*KI+f8DF=Q_Psh;d#A$=nB1Nx_a?wg*U5|?#|xc_L%+Sg|T0kdNc0<!-%2n)j4LpU*2woRcoR7a?V!X^cpYl^|LqW^i=(M=;q0a?B?0*4?4r|wj6i0eY8%Lp!pNUWtCh#A5C->b;gD=RtD?P!tCO=Bqj~cG-jh_|J+L27gESHyW2WWT=v|;!e9cGc3~b#RxNIi%aKneO>?VN)s6k7W`4gmkk5pVVsRDzbd;J@+2eq5=AdJ3*L8$(YoX7hJyEnnw3p-dhUp1gGF!XA)<bK^Ibn&SCvgVjRar6iH+2`9;PN{@%^jZC!Xbby_4!Wv5NIN0jhlNMe9#D7A@KVYwf(2_e6~MUE~1k;sK8m&Y8J=I>!7qhb<%(IA5w3x*NfhF+|6ERYhCut_GBCjqxjH&8q1&Xi*SRIlZCf|q+8v)z_uz3tZ%i6$Q@;)YxghaE(wzY4E?&}9{paKdaql1BN<Up#}=uue-MpRS@XTwcf~7}SNQmY*xy_3PyQ5}O;xDw)aOUaU};C|?b94R8t5RS(22nuV)BAZ6W^N*&CjG=MJknD57S*dux*vN_SjCVt1M;uaoFnSSuz`SO&zMImT(Iy#qz`N=GP2pw?*URsRD6dybkKNqu_5w#-&j@>1ORm7n3`keOQ*SWP$?;FL`)^|3peN)%8bQfc_NUvKa7-%1Y8RdkE@7RVM~~<z3spxe4I$Y9iePGGIz)!n&{CF6><><5~$B_L?`FIwjQuonx`DZMNDCVJ@So2-3Bx{yMbXl*$*8&MrE1w048($Usl6>DaL^Z+$8+9XPy6n%3V4_;d^<Ku3=R$5`-C0=}1udmGyK0GnJLWbl1m`S=#BMss1#t4FK#RIa1S7N2&L!CE#X7kE3Z?ILg<&(YGRR7_u~6SnCsbf#UQ4?(&8Dg@iEteG|78MIw3*BqtqFiscZ&s2oU_#zE1k|Z}S-hCp-cx!w-9RaGbO5BFHX`fI>yLD{2Q>wR5XP67UuON43t(H3g``;QMy#H)<l{G`PRvlCDYS!N&q1Jrl0sipQ+ur^bEw2UC=R%%exZ``<L*sX3Rz2-tx@6e6AL9Xd!0X;KJ-%!uTTVW80v`+-Tyh^;V^_KN4j*zt(F3S+6FQ#;PDqcugg0a3fK9u>A*<vgimT-OzWSCHFU@Tcd*nP@_czI5MXb?2N4`7LddqGS85*`;C#eF2{hKDXI!kSx{!LAE7K3#eht(ajtvu$7KV-Nn;o;VktWjzw{1@4y_S!wvA6~SiTI#fHceW^eJL|P(kWn+8!%;Fep6=Tq;L@0Dn2hJxGxUd6tC}eg{<)4{*9R*og$_4bLNU^7m*+9BqoMFO;S0Vt#!;mPqSy+FKT2LUzk-;O-8fSggZ(%V=!LEjf_6}(<<062t_|{!pGwwZoA04TuKcsEU^B*#J`d%@*)8X<0I7C^J6g9AT#PuqR&01%j|C3?*w-7^DVV0Y_Qc~z8~1+6tbX64=AShL29ZaL0|@gPzB72X82vk>y;%AJW66q-`jRT~LS4y%x$?sIblK0M>8RQC?u&Alg~0wDUD~&N)PS95P&)oN6(3C2FeuJvUt3d?`OIVX)8iEBpHIutt=zK5vHq+7{0o50s(WXQ$>j2QYV$yvo#5Z))pGw-qsw!YOx>>1R-b&S4FfWr80gGb#-e}=l|8eIZq1yuTYnJmcu-n3<Yv@JpnCC{d|VsSj`KADPQpqJ0qW0JorDY1BoKyWR-KkCkB16G0@cvESnNbT%Ntnjw#P(x_aCPYQe0OUJ5AOT^@r4`N~xcCD_d(l4uHlVCGK^v?;ErnZJcJG*1nW~=nRCe+(j3jF!m7wNQ-OmZ1?kvs;XH3@!lkTvvbkgC9Fa{gRN7*s3V>vAt9|4xr@#hKP(-!8y9!8KcSIg5toWT_+A%$U#ewwGV;(P5PcP6GZPJR?O?%hahVFI_(K4UD7iz&bVkSsUbDQk3-1YEMILl(=`APO?-fzo>Ra5@+23RVG;?5N|M9b1l^}zEr|Vl7F@IcKdAVPz-W`iABf*T!xE-eFUF}WY<<ES%g9nYqh+z+x=B0GJnm=qpmtAGkd#DcY*N$oao0{9*r_u>pMGaltK1iW@qx19d>|LSLNJQVM-Q7qyKDN?$G+p*M@uCaBpC_d}p03AF5%P#uqfz&lx3R%not8@(<h0x!!O!;ScBwqbdF$2gTlkfrYAvg22|cZz%{$#u);}n5>xQEbN$17(FUnJou}JdDb#i4V^?Uqup{|mF%J1a?`{^};hW5tyDtAD7cC96{$JX#xX>(*pt@>JbB)Tl|J~Q&f+C|mcQMu}(cw;|T{UmL~$_!}+VqP>(yxon5F+JoIs0+9JWVSu6a(8{dKTy;l?M%8PbK<#AWVmCg-J3F{ig*bP|HvAKmg^CgaoFhFJJ(mplJc#&75~h$Yh*szN(#L)f9X!5RP)Q31=R2dO@m5KbalPg1&-qB!?O!`#TD)Tiru7ir!jYwLv3B%G?u#hRY%WBI|C#@b*xmi^@jVIj~;x0i@8nT=JmsilxE_$dg2B%c4dIw)4swlvGbEzkOon#IPbJjtkqn-g#Cs*C)VQ7(PS}N2dX~^mbcD%c;mDaLvkNOTU=@++Z!9()^0p)dclTSeGJkGT2VV5zO8ro`~s<fT2oB8Ok<523t!joUn6;&Z&;i9{vi+0Xl;)Reuk6XU;64$&zT`#Zd|`c3hpt7y3W##FKxXEn=i+ss5X}AhchwgNtK{BrjOs%MS#~Q>e0S>+XK0iDZy%XUx)g#H?rnEPg2*`an70YJ8-@2^N3r|@h3QoqXucWNLj5Bd1Li=-Uslpuk9v@MQ@$S30B~pQp{Dr>(<Nmyt|h6Sg_oKuXU??ZiwS9z1Q1|!{0&X>V<Xc@{de!uroyph-Wf9l;M5{Lk3~Pq}SYQ;qCzQ`#3wXon^On!;Lc^1};AMUFiM68EVm#M?M5mp{tkS5+)e*2#%`tRnyXp=JRTG=dZ9q^Z+Zik3JiuGAp%Z-I-~np&|LzW_5sAK1R<cICv~?+?oVyt8B*(oK<=5^)|Kq9(ur3Hmm)Yu^HKay(in<jir)>b=z(+y5(9A-$r&|0zMwBUwK$E#Q=0NLr;th<KJ8U>V+OOoVl(?RNnO+5svqGj9OTmKM#G+9L#w+o_7PQ4%WNiMYEUqwld#!zz->0YsF^Y9ju2$EuPn>)H<omYRY!ulHHqm3n|F@&a_n+$RFJ@B{1nwfLq_C+fLSWmifWl1mV3mSCW_q!T#Yptxdrxt_^D<Bv)U-`ZW3)?irX)e&?`cdG)f?TWry1b;p&oiLq!Y>Mf7!E9h$9h^5aOlSvD1kQ-ZnU*lq_)Hd}|KGM?D7Kh<4^Sqz+A4b0@om%nzOWxD6(dMvQ4WfrRK13V-$&O6W&Mx?`wf{-&^g!IDP4|-oF5qF>q<=S$H#vi0;0WoL)wxE`Lmh$A%oksiqo@h_#x|Zy#|b!#03X}O@4i@;-_k0aWx0sf$44blFzP%Te-ZVCWMgVmEeg5Y>+Q{y!dDvcUB1)iD^})fH|#IzC`>K4{#hFRk%y$Bi*`Y-s>7An3fJP=jS|^%Tc8=iF^QHYxd7DB?{O{&AN=*2k%v~CHs6<NxjiY?311xHi&VXp*I+cre~8}O4g6xa3r35n7}uuWsM^4#YxuShOj?Rx;Q8x?BPEpCE%ZCOp3fZp>2Cb{Q?u!4noCTTOX)vH8^1^CaIZaRkwJLGL06v$I<|LzJn|H#+rxLxb9B_-3<*L(vP`8u48KyFN+-~T@-h|m3_Ms3JU6EtW+A&)n$2r8tiBz<Ubi<F{Z3trGT08>V)e}0dv>b=-M$sK5db9fyvpmEW;Tj5)6syS84N;x7TsIx2O}_{4H{0D?CS5&PCzGrq&@Bg?&UCBY_ps31||_6=(Glb>il@QQiIVf^o*CvcekEEN1A)XMPk+lChhu_-ul0Ns05a(ZrXjtYb$dl=avg#{3gb4S?$aJ{RNVfF~w(oW4bz+cIsi*lu>s2;Tn+8!;2OFr+0U|d5qYdCbA&R)`ifQR)`b*l@mold3fzf<HUpYK*)Qjxm(r094Azz+0a^zH-9pGuNQ}Y5LPtGDn{~TwY-ty1EA|x&D-}|Eb`%d^Pt-vE_a*z#j!dy?btr_8yNc;PXfr|B$@whMbBeQy`3)HoWa}Y;dO)V^~L(_4Ls5{4_Ift8Lg_qbbJM&8G4YRKW|bDxEA)BS-Xkc*`rDF-nmTNnLj<1!;Oji|FCzq%j>yO+us(GOc*mE(UkH~DpV*+FDjuVk%vl;+kSn2J8<7K*Q@Kre*+YqYaQ!z=x~zrG0ipoJC(Ov?cc?*rRwIo__LW$eAl0%g~y38q*Lj=wbf%X6hiV?^~ff=Rx4s)cCqJ-j)sesqfDRmRqutt$5XoPDHjX=#Be=if1tJo)-m`5+gr3}-YnCBuKLIXgND0a2>#ieuI`Pw6BfQ4Oi7oYbC2Vk)<E40IvuV?nWIIvJ@Mn~28TW2I}2W~qW!AqZ#=U%%R1RUvGQIfnwJzvR+`RUXCl195?K>F(&$Ky`DSM={msDN6d3~JhB2(D?+PG_JKg9I#cZ5bipFUp?Z#^e|7u5jc`BXycYmUnLbz&)^0h8%X5Nh7bjef~WdEdwq3pB;UO28EDxYVK>w0@7_78Hc)r5DizYpJeOtH>Hz6sE+4^<k$s9i37(b);>?Bb>(ehMX%ZJCIdNbGWbRKAODKW+%$GRBGN`uK#}qh1Lr-I5{)fE9n0aeQdEHm5{KyU^#~Kd_@^a?8tWIit+wUrpCakL0$TH}_xdP<(zuQ*SMW$=hlt+P2zW*{u)aA2RAsyJ~s36*0wYM-NXhL!zw!B&b1ip1zB=<BG?L!x;j{wb8p2rR!EcIr7_Avxz3>*8pRbaPdQCs+H6iT@(f21La-#{je%S@6jj!2p5te5}{6^w!jE|h5G4h`}*vS+6=Vcfm)*_@%QVga>W`EeO<;yKzpWeEBhTJc@H??K!TCEo_f8r8`spSR04bRNlm05aSj+Rtj6}_C=VZ>v%V8nTdIn@cnL|*3pfGu#YyKbG$n0y^JTrx=&oU(x%IFz{T|9Oy}{9$#awCexL!p@hr|ZvxmdV$a&GZfFKuV%pm^V#yOd|P$O28qhW=o*?Yt_@3$U};^^j*2K9_V%i&mlE^N~4SoVq`#4wJpr)cY*-;fwz?S92h$)l<8~SqGcWj<hpx2lPCAPo_9mo%KE#r^&73+sNhcVJ5h~^PSmR&}H~w04ILGu3hcw>NE*f@Qh?@%PU4&eQVG!t-_vJ%Pc9qmA{5`Mv3Ki@%pxFD|VO+)R3y_DzcsAqZt&5Cwp3L)^75OMrX~!Tq><l?w4L0DBHp7bc|B__B@z;(Y%k3Z&c>U8a+UFFQD?7kM3E`l$+WpvWf|?c%zT4H`yOgSbv2+zv7PSG46?*WXq?%3GSR(dtKH2$G9fHJr?zYwih~#JKOc|acWi-&))KL_qV$eZ#>-wvNpYZiphg(VgaS;&5k^0N_6;$Yv*MnV$OZxY4S~+OWG=e(L?na$p`5!U1+7tUTSG~wQEiAGf4Zds+VyOY+N3sNBRmz2Wj^(zvR(N9{1i1bEHr|J=jPdgd3En_*y)u?BH45o28#mx9>jDc+y2{W)0(hdS!8Sl;s>f_$%VEY|j418yJgtCKoj$o(P}I>#|<;?w)aPRky0vV=b;WV@|WzX=RVSeVw}jW+Oh=-mRyYuGXD}nLOI+WWeck6{-7^t1xM8A6l_$E1>6h_q|=cIvDrQRIJ+Pd*WsHhHpQE7$P7C;}D(87mxN+0bbYZ%ZQbs<3HqQQs_dP&FfYqRI_ZW&J-*f%;=Y#HW7P5aaD816F<GtyY!yW&6s8ScLl$gnC-2$#=4m9xDA(k8~Up59(~09Q_qeys-*7Izt)j8+TZ?uy*fZ|#MR)8Ji;L}VqUAOtrUvlc1Hmi(|s`df5$AFBJ*F&^1qnne=*DdVwV5KEdPsH{ui_SFJ}2)%<}(Y%yQJ89G@6Kyf$N&CYJ5M-LFDYLYpO8ys!<;cdhGKBStO)ASu^RQ?yYwZuLF#+9cT~lBD<WtxT5H3H08s?RfvYcc#pvUT>b+qUg2Lukkk-X7E<porl!^si2B})^l&D>&6_o_GX9MgO%$Y;wCf3(HI<(t>CPG>2&-Mn?cbPpT8bAI;jA)KdSwun$$$svhy>3t}j6Di6N^FJR7bVtOuUYs~?BQHIdeEsY7@BcUXc%EZAvlaQnF?Z*x7atew%r*<hVn)43R?1Z20D@y5Sav2^r-ZuxEg*<+e@wA@xu23LFN;de7-5*;`DD*0DIPaUHkURj=tXnWSAt?agANos#VM5^v{rhi}TN1l?9@Xub&YG)8ahM1?Hg#E&e^KaSJW<W!@u5G(<k$!;d{|%|#T!$k1{4}$XI`$9DTv*MLB{5wISo3{CLC>5XB$zPo7O`B)9C{DNl3Cg4qGgE75f$R$Y{4yVYt}G(tFq2cuAzRY&A9bcyu9Ks9Q})k5PW^vBscX&9g<}!tnNCaa{C$I$;IRY!b@GfB$cR2mj$L%#F`Jm+qAqnE*dLSe9J7JcsA#zr(1gyF&DOT{DZvJrAjBGE$JTf>9qQ)Qd11kBg=_F%ymom1c9HD+LM&lD>|&xZTHmt6*`7IZ7j><xH%Nf)3~5jRqA#=Lul1=hUa529qgmSskMhxGoCE1GM+t~gWBbwXrJWrX^9Wv+qQ9bcz>mX+<I>(5&BA6fLdeAtQF}`N4Qt!yU}x|dkGCe?76yJSc)iij)%=A?Mw5ec$Ta8LUH-cXt)by-Guj$WEMBQr7+Cl{B;}-2sie2e?AO)THsxq3k2+)Kpokw$YH?0csFZ>4)@ZBDF{D_AwR%w$KTTA5B_Zo4(ABd9yGo2;eqd)A6Fo<&N{|;Vb1D4l@UZ$-Q{!eCP0IZ<P7@t&IMd3NjV6HUU8963m^9fhl}0Vl6hUfPWr1vY;NCjJfSZ|A`agOPi2GL7xl(qs@&eoKc5kL-DBLHoTb5W68MpiQTntz1of47_LpU~MSjS^G*g!6INrS`uj)$r9)#~CxMJc)#v6{48pvdS4L|E%{@9#m^V*(+S`H~Lh;f)_TB&ZG!{t3~D`ER!%kHd-Y@X9w3)#<v1`<Sl-Ni<I2b0>meIDibZ7mJz%#E)T{nL~>^~ZjFoK5?u7t`q0HsJg^O;`hj65KCO%|3N?Vym;}zB@!)zhY#&sK~!cVKL66v=_hkae(p10bkiDdz7enNI6w1Z#ha>##^ziWwl~j?kfkKa0BLplBH)zw4P%2RyN}OJubBF7O;d%x=bxU3y!tT`ANe=vF&$G)QH(q+CD#^dG9!6;BITlOHAMFI(W^-gMxi2t=hcrkIjIRM=e6QMwKex%$b3i-2Kk>R-VeXaAth+y@HzVgV(pY#K_efa|JHmk=rW-+g=T#hO6QtF2dc#ny()8a2UXY_u$W!j1nBl=L}1H8uf8s7S{9NCpVU6YyYlKh@}Pf@=k=H->-)sH`%F4L66=7EAjwbh_53qu2-~vP$u0Ck=4$B<geVEy=eB4C=sXoZlw0>zWnAEAVNSF+m3K*QMd34H88fw6OxZN9e|o9dL){Tae0|`Wx`i?$k2T*H}gM*h*w+sH3WCv;kesZ#0#C_GYtHWUTh#!Db;PaZ<>a6&QWV-Ubse2-Q2F<p?0}ov2gDJIOKlf9qh#K-hFu;d&mnJq~S5Tpt(4tG_lo9ul99kqTeir{X4YYdaD0Nw3AhkZx#B7Nkgp_9L42H!k=A-&b|~dlMw-8elkt-e{j*<uaMmCiT#FMXqOFOV?lp6S=BrT>9m@wa&(*9Req|T5({{hhdxA`c_%=g#C*K3NVkr^`h7l{1dh+AgIUw;)d*tN&|O+>SG|LQc6P+Y+G)cz*Sp%Wak;boFkHmQ`VMou`>9T}&6@MR&(maMHQ@L&pYcEPczj#)#jJh&0uPPxJI~ZZ`+wGVjJ==j<16ps`VmW@S!jjFu3yFl$m+E9`7X~Zi=6~mzJN`nR@)C7ZJ!^kTN6(m!onH=elz_XG0u4$JbkOUYta#Q#0ysehi)_|w@5xts={lIO5NO@sP|oRSVYBdwVdBp1=gtvW*j);Vzq#a<xprOi(Yp5eUfFp>(M7eDls9yzqOjKn;M>Xt(8xXYYbD)E%t1uNOC%To)zd=H4ueu-rl74Bhz0^Wk0o5mdr+Q1<loNce2AKeK(M7es=;kot3kkjE@Uv@!87Y=wmgE*<2jG9=qhx+tM1;(9h?tXCvU)>zHi20Q`sR`k1+M__%FxtKSL>BI0F_q^?aHmc$RTc-TstBE0Axcl@LGM|N29om`@p%JZUVAP#cjRAmv}IhM@XrHG$LFS>xMu;B;rSo&C=I9gTOwc1oU9C%+gQ|xklj^W~P?cUZXhdSxYeKhS>Q1NbOsQX@({58U3_f>D~*VhcN)Ysvy_e}#?mn0M(Pda+p?NAK-_$-0B@R1!Cqt|Qv2di=lR>_@2t$=xuEX2LCdk+((@Q>pts64MP73H7<V_zs6{S+|Thg25?-_|}o9P@~!;@ZUg5XNjNa>~idE@&_5WEBrs7`HpRPiXTwkLELz+}<ChN%U4JBdb#a@2B6J@ws4~#<Mt-v;C#I9wk+kd>{>==j0WsPHe!Tef|wL;h^rHZx$UFI54hkw3@Y}Z`D=#!I5nX%crU59$iBvQN>j@qb7MCgB70cmJ4%ZZNyK*thWEqgO+B>B+~5ILF$AlXY-6<hmv`$>=n*Ns*fx}>I^G9PI6=N*S4R0@L5p!clWl&QZOl7(>IqTh9C|0r_QrsZ^N99dIFQGk*RroI~nm}i~<3i5%Nsik6|fZy|#mt+pD*0r)&4GKB71J&jjw@7%F$?!E=(q-%QzL$&C>4@xVEB61xKVw7&x7sm4$^i>a@&{Ykh0K11pA>5{`E`!M~YPTxRKL8H%`gWmku+Y<bnai!B?teDM4YuMQwx-z?Q<lCnsUgK@qihe}=5qkNNR7dE>a^P2UJlQZ7*IagclG-xIc~XN``1JDZN#tv@@MV|S=9z!m2oLJ7b$JCWN##<V1fzP=UE1f_U>?K(c)c}8JAOF-Et2h!liS1)e{&~C=Zzw1{Z1|XT~MsXD%aWyTdiZ^x)>m|bN>AG$Pgp}h@EBDWR0I2N!2MHu*n>>E(*<lPLnz3+e;Vt?H|~{<p}&e>Rm0#4r>DETd<REK(3LVm-IYaO;4vwU~f31W_9wegPMlj+8>Jh*ij5RQIBB~D6}oVi=g+kn*4Me==9s@cWTx(Fly2^PEeopz{wogD~)BCLeja(RTx)fklAu@-ZXB_xURS1mRU~pxVL40cdt2{Mn|4LJH)1y(|48Vvd^B;q59iD)B^K`vAalIJh;62uex&Cn&*4tXnc=#p*v&(#{FJS0B%?UisVO*D!=j{dV$mOZclI4o9ksw7RR2(>rm*UKP;LJ_2Zbbw^xjIdeh!EILO~hu=8A7P3UH}nj4P_cS@&Dq^Vtk&t9iSlpi1KWN779Gny@dj!}GtVgDo#gT`~98^*kq?G3QklAD8-*ZAxI6#e@~){m}^ddrk;`}{D+C^>jb8C(9ER2p`8R$kQUXbPoG2wcK?tsqewZVkpQo_qi?v)3fs%cXv^W0RNld!qs|2K48Wh29pMAJ+GqW7FT=BkInne@xR52gsA^4Tp1gEy%{k+1IMWwh(2BuU(XBMYV&y;*+%yt3F+b{$nySjcABN=e}22=kFbGyLUmf3>NdKFwxf~ne)JOelFJEV+9>%r6~+&j_bR!rk)q~W;o971Y5oQmSB?*KlEPeOvUb^xBaF=<3c&jK)2aRgkJ*%=^VGR?KQp2f;#rwU8EWr=&_0?sfT!DynmwE;NG?*|J(X@(Av{~*5!LRp?~#hUBP~b`Y>WTfNh7pYkHeph-#m_Th{}RL{JNlrxqW;TVS_{q^mMZ28VFfhDVL)R8_{z1-r1~^c)e7{q~H>4d;<GVHklHNN(X;qo~)?-e623-HDr4Mhr6tAA4ps_o6IY-nRMg=jhhMZg=ppu@%m!_w%r2Yd*aip3<?p-`i<ElUuB|fB%I3Gc)p1=+cXm*xDR-X%H*~;@{5F4ODovxQY;)bQeL6Yst&e-c`CS!&k!Hasmh|m(!vK+#g5wZKZIFv29=YL1`NDR;L5&)o*exETx|o%1;{<_k2A03?1BfA^G#T70d1us3Rp;-oMafn$@W0r!jyBw7d~3kxR}@C(7jQv-(Vj@=&hCfs0L{ugcx&<nz`09<uJt5Kla8lOJUDltHtV99pbX`L0isAP_RLEgeI<ED(;nw{C@HQWN@4fxq7A&^?Hbw%3I9@RQu(=AH^SW2)Ex?Ezwe!RlwYt>Bupybi;2XBYSY<E)<=dzxIDF$v<;sqt$fXYg7ctFu|9Qo^u--$eIutk;;yPKDk$lg<ejZs*bNKoJyn|H2ll*cGy~dGkfxIvIt2<Yj$7=`WJyeA+nOjX6kljxEx7P)L1Er9D8u%n7^L{DgbA8@b)Qw(2ZV2Tk5RnLCU$V!#at(CN$O>7$vsMr%ufH}`K<z5}*Je~xecsX6+o;Gq5plkBp!z`SzZb(*)^3r(>{CK*VM_uu8|z^Bfvy5258E)e@$5xH|#cQi%Qn-vLa#cFVdwt2y&a@w{2Ffp9I<;`&8fWYqF{O$3mL+9<%+yUre!W2jy_71&?_>C|keb^VzSIF)c>)zf(^SD^>Csp}E!1U&fi=u+{lXlK2akTuHa9afKch^BXGY_oBlB)>!=dR)@C*55idO;(948+(=A3gbdfGAn~vpYuyZnw2&FtWcGbNd||<zZ*(mvRi=22nl^jyY|P##NtxJ}Po-u&-v<u2j0`jCQLDaOQ#dj2gkI#aPO#EQHOh{n_lHq}E4+Z#({XOStr3ed}?bga##pBG;f{d3JrF;-7Hz6|Y0BQGmdQeQmhA)vH~RF68F>y6-UZTU}Wv;hHb+qST0#2DT5-lUd)tRDUWqll6H(ZO2;Piw27t*69I8MWxRCwU#B&{C+p(IdJMkUyfr6j2{0@5g1KKo`>>x1?E+zOZP!-Nq+6njcBg_PWb+2>^<V0R<XpXemG$B{$_pBDs?V>#uv}Di9cu}BFckx`nuc6raD9o8=EihwG8d{F9OG{MYbdSIs5}SVQokYA>y|%;}dle?s7Fd+@*St6tkvCZKy7@yDnZu)87^+aVef{C)|T<lifw~Aptx;XT;%rL*FL`Xzy8jZ#|!E4O(;fd=EK`eu@s0{jqbxdB$*N<t?dg^=P@kljZugzcuQM&tbLGI<Nc2>n_3R?PX@87fx3hI#=Fzx-SDS54N<z=;h=+{&VOfN1~Hm>h}%BsAQMPiLaU8**mfNF5`c14*;aj-QH#QybluW2Q~<|VwkHTG?bVGwVPKjC@?f`uP{=nL3k?z&`VsI)hE^Bn_j~9w$e(#`*5%=4Ssg5iwD2<$4MP~o&4NC2i4jSwr2JHSMqU%8cq_zjw>O#A&a$pt$6MRa_i{HMq*&-lw!2j{aEe1S}<+1ZC9P_zMXKA$d4~X+M~JwcGJJAaf5O#{vHEiDU46S8YVwITimH@_Bj7D{o0C2?H%zD)~?_JxSvgKlO<w8!u2;gw9djkyx&T2K;e7Qo$Ma9pHOdcwHY{@rsK-(QEfr}t+f2Z6&Z50n=1`%>N<ruHQ^yGxXvUDKeEoR>j9{Q$)!U29jS-=9=-o=1o+<_@X&u#lFk+H!<BP~)0nkJ&8U;(AsOblAfNc8*(8f}nR<kHMkZ=j7xLDx!~^@OzOw35q4<z(H-Xl7W7`3ZoW<_Hox~Q<{d9;6x<=UU^G_i*J<6?+=a3=eGoufVhHiGlLl<WM+MU+=ZFx907e>%@^m%0h;*3vO-<mxc{+>4lY9_y)b17IiY9{sZJ~NfW-~uPp>6Gq?pV6hOeZ~`g9=3roF!px6*4B?6nr)8Yzb4@f3$fa7_pW%;aXYdEb)q{n*BXSoH$pLY(1CQ6yWamSkJd}mR5G9@@9(M|T0$=pRicCX{^r2aPdMPXa(xrZTIneuKBN)$%rtjiy6)fMwYe$Jf3jPHfSH!vc$m15_si}0yE4zLbUF{)qgv8UEZ@+kZ**G&(M`eK?@B1=_0M#ZTYPxR5;Y6qr;PwXcx0}fJ3$o>6_f4BhNP=t5omAuLG7zL{Ox{{5v>*R*t<*%j;o5dMorQ0Kjzs(#IXA`_+tbtQV>RsqP>Zm>1MMkDpPZlldbk~(xL|X6h?Z;^^UVw6aSvKvnqU<vv~jaF8lHR8{c}>W!<`U<AHLeFB6>>!P+DJiR*n_s9yIG58j9DJv0!RRe=e?26Ouj4xgJ(YPrFqEik~gy1bDEEv0H+3>vrS&;1kGlgW(jo$D9KkH;T9b$y}t&4Z|(;n(Wbmqumakw1g(a0?{-m_o>peYGOzdAgh1=X<{h%f*>n9yfyk$}}ENBX$rkotarVHDc;9-E`{Ad^s|C)To-6ui;863shU_*?Y3pw^qCd^rE}I@vr#ky2VXKI9aX1-HMF;5S!1?=8L_1ea3*IEs=BVTcFO_3OncpQ*OHUOy(wu|9sz|d9LSbp>CEJ5bUTweB?L|ti^xs<?(wC;bOfncPI@mB?4Y}YwprSY*i#`aDCI4G`72YVvBH04T<yS7rwmx6K(C!j)BacBawZia76DrEA`I@q-Tora;x=GkLbF(M^4n!y&`NwOoe3B?;d0a16%a$o}3Av)Rp}k5kIQvPj+D{B9-B$KbK#(D^dJx0BeT?c^NFumt}VQLlb$Ih25O>)*pIajTg`3u)q3V@X-^4F0>J5QBL@$&g|^1yCLdfT5qsN<&GoOJC}SocU~!8(3clfA9-51ZqKaEb0iUxNRJEj?cAv$dpRU;tTNsIjiW<8w2hjkxrmydj$(FsA&Br*Th9~iPmbAT%9Q)7U@sojSE4}^w4{vKz+T`QJES+G634QP#@_0bi;K?wp)Ja7kf9wmi)J((SFnL}&=hYs|EB8)1`u(`3A;$sAgM4^it6N_+H4pfvE6pW+hyY#aJQM*D7BZvn9FF_Yjoz!HlOepD}5<gtEd=m)TUT&2p(EtG7FX0;Cjm{POIiMSnPg1{o9!F7WJ*+qw!UOBjNPA{2DK=hjsH=W25qeg1S04r`JBswsA|2Hg~?3SCYyVxUR=TdMw`W42RukDJsk`{w-kFu22GN0@~wnDrMhwf-jHW-P-*vJ#!gdjLUY${n^BoZbEB<c2*Rk&XPBUShc}}^q`h;nP0V=5-ywXU2rLpUa!{LGmR$wP)zOYXyy2_fYym{kMAu+7F^oHtyT5u6Csf-a$+WVPyHel6NrE!zq#sqF=pjFGpN9%w;Q<+x%|GPZ(_YE@a_bQF;>q1=)g+)KGqRdE1~BzzS3@lnxA#n1E#Y982$75E8ApRInL#}sI;6LwXVH;;B;_K?q4P3`5Eqwelt)Drduv}EFXOZ^;-z>bzj4{!p354uzmo(_+S+z+a24f-k%L_!`^qn9K725yOyHtkk4Mh<oEj2fE(e?+2y*82R`0gpS8B%-_@JPsMZ|SvEO~@=F8T=qTV;HU$9e2a67nbq3^g(oQC6BC5ARR?*uEkXA1MfR6oPaPu;%Nue@e&({TBThm;Nw86%HWAZ6p-d^tRKM?{%=D`=d7<@R^ji1>>q^BXW$uYx@JuIMZ1H^cZisg{Vc2?o~kL1zIa5cuO_9x%2yn{?*!OA&B{-@~v94j6i(YvJpU)27XExLzqM=`-wC0A#+~PFsAQFP3C$(!QLR<w34Xo57yOClqm;`QT@&5|38!`Ec(`Phs!8x~x?ulm_gMBy_aI{j^pbA*mZ3Gkk>a{cDYLK$i^$Yn4+fqyA*jT0b)DQGtJ{S-*XUS*94G^-b?U(>-kT%W!g$^7X@&vr+SRHY#;Eu?PD_ryCv(g^<6!vkULJ_38$OK3NJ)Qu_CV5V>c|JRcuw{KUv=?WXvrFE+1)C?r(!o`^iQqtSRoDh#aBBL2+Jq<Q%pKg|j_tGn>?Pb4M~Sq9s|6?`@2?-`QTZ&0&pGh*%cozoEeO`g5^@J|fxeB@$jCbK^0SKBMINN?A5MB7!K_<_0p{6p83BQ2(%F23H1IZ#i)L(OW8?**4`fe0Dcx!D|8*|{z2*jAw7jWD_Ad%q%5`}ViT3wt`a%@+s$6YkpCeOo(60B9YaO<?P7Uhb!^SKDXg3P75M=M5makXJ?gQ`Wy@35Q=Uj8k2FnAx`@dy6Ao+jmYxvt^L%t9Q=3q%`MN5eG)HZ^rquK&zv7QpXQySDHPWlTOx@{8Z@tbTk<wlX(NE%==NuV>@Bh>;x3&^Gec+w?y`+5~p``z9aBy^TdhdM3>HXZ}yAHU>l6l3_)61kFsp^wzPNPbfNAmwH{Y_qlc%xF8+V%N}QN)8~Ron8}r51=8ft9Drs3=9^#MlRx5k2-mdAwixM$J?kYaRRR3t(@qAKB>cc62yfnVp@Ky;o8>Eg^#qs5~nUcH8wDF8Ll}jGKo^$tjkQ~+SagDM~{7YK0|B{y6J6&$S*tEjJSOrufuTGXJR2!vXDclY9Gd;1-+iZ*&=)?V^m^J&bcjLs76yy0Pc^@EXzsk0A-RR5lk892pMj2^L13Eg=?@X?c`n7fJbQU#oX}vv|-tBJ9q50;k-H-NI6z!|Y!n@!Po%f~tGa+CLi>AT#$U0Oyk@24CJo`Eil=})<j3h}RYkp_^ZpMcOy3TjS*&en-+IrT$4EQ&;p7UV|zemhxNN}!Fq?#myoW93_Z%5m+!r&|4x4i6rcb~fNo;&yFOBn8Z3j!nMC7LzfZ<|wX;pNi3lm-TK^r2Lr+O5B8pNu;euSjXunvqGdBL<VLllAy6T?O*kb+LmdQRd=6oryZZFb#lla%JSms$jGF-05o<tD`PnTmd3mi`dlGhh>Ova&%Id{PYWtTh)uPO(fSeR{XU4>Br}(JRqQ6XP(rLgoWG&A&3$o_U?8W2AwaeVlBiP&!=?3YMa?X8~!o*HNkI#>85VEF$uRHZ#7O1hClgCj0nzYgYs)d?dYgtA8Ky7OAvxU!`m{*uO{u&L%6xt;r5hlh8h0U$OJ6a=#Pq=->h?94t=-~`mLlrZl2cn37cIh7-8L7aDajS!^DD0eWzL>-=fl5Cu(Q!3Ghy8cu(;zWL+8Aw+do0=Ky9VP05PtcU7)$*T;7_kl$MPkkFQND5U<Dx%81bF&J<Dpnl+<0_B9opbEbH{bLu`<>_NgQ8ENvu{QStuso>0NpF7VWTvqVM!+B+Y<hbA@5eh&=VtIw<{raYUlaW{c3wqZ9XaQemcskl3a6Mj!|C1NaBm&^&=q|zQOWwEZoD{spRCXl>Z(!~&u|9{*1upm;L-0_SH`{ZE4Tj+ofGjuzgh*EZSckAl#{1hv>MUayN)Wq@HnpML*oY~$NuYckXu$Cp+PMqjqX}9$0tm*zm=_I15X#L{QOH%Lng<qcXZ$V<>rNa<2|p$vu?x8*Gn+o4}w98oF3V);t+NJJD;Vk1AjQ^Lt)&7)~itsVH6*w2u<Ue#k{ib`Gf$cyGbhSwEBF}Vhd$>Kifc?9_hs}Ls}*KSmN%|KTaz8(LTWHQQ_l0UEAH3{GZ4JTv5K@nxQt!@1Hlef)@LPZ~V>Rb8{(?six7%NXQmuyDi=L{VDWXzC+7hu_-2<c^k*ie}+D)OXHY$E;}ehHO*|?A41yxWcb+iHS*EzyaNqcVEE_U+VwvgboAx^+L`$B_<5{gi}v|UdFiLI1^r$=65)=r>r)TVd9<+eksX2aCDeM~Wp+Pb^R86CN`njE{A{L!pFphnl!<Dw=Yjq>v-a1?&(LZ17wz)<{M$<z#oIqfq<Lbpi|`FT^=#v2-ca^OtkaR2c|T;VLhV_o4dQ!hfuk*KOv9Y_vX^RW*4QEao83dl>-KOqq>ny?ZnoOH(e71yY8{U^!<tg{p2zQ~7@TAFk*US#B1AXRdoM>`XNg2^oBaZ})$Fjl8);Vzc27BXW8;&dl+KU%JwJV_$-=Yzd^(60!_qc8Z?bq#&cio9zu~c3{}-~H#LA1_*6`0lYH`z^&l~+K0798*5&Lf!AeMEc^)j2U*9)$D1uNQ<<MrJ@+r1oOxbTux1LCoI4{R$0UW#|S_a}SVy;z|S?qAS4+cpll_cz)*y?&)=Ag2(xf6vIKV7K<BON{rl!@Ts@pray}anr_M-uDo_t5WsRYWUQ@kR>&d+u_)*@De8lpMoqG?pvKG?QBvWhFV>{&+==XN7uD8Dsa->Uf0(3qBh!8hij--0T1sbKJ%AM;x-HtT2A2UzT`(+_02zD&zN8la08!(<n|jhV`jK^FWbD-A0_)97?VvZRUNWuohod8&EA#S(dt2v^GJ%sU*{|1;uX`}E9BF-%C}!xZ{(+}dfroXwgRS*6Y#4<1FQp0P}T5K%l9eFxA-6HGpO*Sg6MPz>9%v=foIj7Zc4Uz&S8}Ny5m?M(pMyFOj}JBuOlw`rp3%50;UR8x*~C;PV-LQIk}hfkndcw#u}pZc{gTjto8tjub^k`+lu&f@b#;@(yA(LJhcPbnsQ6XXqc<M_e1XDa#AwAFW|G{uo#@$z5S$l+XsI4L|(FCjD>0QG&um3$1?4{w#(1kyLAy;>Y!>>uR?7+{b()h&%V7_S-9TMtDXq)ReP&b?3a4Y?<3+Q=TD*_K37##!Exo=8%b5jz5T7r{k{Y}@oqmJWf6VgjH{NtQ?x0pPFwBlqeBJ%WvAiQ8%I8>{n?)0xw;cS2`)Axf?RjHHb`HbZKH-m#}s3g^!Q2VA$Xhk<l4+`9sv=(>*4eGMu*ps*Q&7Cy8r2i{;*o!`Y6t-U-~J;Gk5R?_~G@mxYgP>xBus%Qq<B3RYSsIMHqH&8lIeg1F&j=LEuSqqwbuN>{bnGihcgG2tebt&r8bTD&p20P1rZvT-$xD-CWl7pdk*a#qwfEN}#WHvb;n_BmSWb41fC-(OaXFx2Me72|x*2<-^`vm7B$;PBc5UYps?(jSc}*Fe)q>)kU|4)83{wZ%+8XD*Lv_MYHatM6<G!YSVs5zQfNo)aUJW2&ePEewK|HGk%WbMcb0m`LV=L(G>lCHsxv3{3}wZjS}bn)!ody$*>HX*F%p9-co{>__+rnIr5YH&f5Y&q?3|^x;7BD*z#3Zn&NW)s5~U{XZrb*frd5Ook6;=bEj&78<VzzF(U-4eIC_GkqusciPbC6T2?kQk9_vLe{Y|Y2HpN;#9>v0Lz$=^f)e<wSHj;J1DKVBIh=;G4|PWExD&ncofI=s`TIo1$<IG&ik&^Txpn3z0xnx>eH7lx{JV6j2x*p&ta^>7fV_y7^%-yj7-+T*HL%8vOjoZyjfc>f4A_Vm1I|&M6Aif<-=x-dN}M%o__~2&EPH(F-%VvY8Q}${zbp#9c2wIyb$N1W56KRYWX)82R7L_8Q$3)`A3p*?kGSojej_w^Ff9NRNDg2HnCizRXs$B_ZwJD@VEZDJ2E=J(kJO^upJY-hy<GOuer|3fW^(Nqv?oTMIsFOYn)#}NI_T9K{NLR9LLIPbqQ(@pUnzb4A+>Yx8^WqkDBx-sl}f)gtG3Xz;xbSwC|AvCs5;N(vEw}3qUnyZ!j1NuG1*F=gOo9mxcOxFrx^vD%wPL4vp{^C-wQGlj@nO4M5w6GZ^LhZ?wZ`2t}`ukKvyq>_s$`jy^SC(9Q{76^v?D=XSH^MvF1#IPI>%qYrWLR29a2HFIbuVPMV3@jfv8k+>Qg`(A*PjsO8sN-vD2ZQBA&dnl0Vq;JLHj&+Tk=!2909Hd}-K$K4%Z+9K0~!B~U2ujtR`ElpSm{f>OSKHldxaZB#wl4gketdUe6r(rP}{T>HG>0YW$Zrri1!R#|&XAqip;&aXra)Ky|+4;T;z4m_1%%D|<Hcv*)M&1wl?3cdg*OSB^7<zi110O0fDuCC8*=FCv9R4&7A@k_1FBX<1(&qH<L05a6xbb|leMu=Un<4!hmxaxy&ttl*eckGdX=@dw`T`e4ZC0C&=iuu|mcJUz-vRKbH3tu?cmI-v(l86*-+Q~M#<Ab*nTtQ$|C;fG*JXKvd2Qp>b~BC2l-;>%HdQdY9ek`7HE)IC4fkY8CZUl2Mzb#YX*yU{QSX~0*zg`Tq)YS!HXWUp+M=_Gcg@;CSG<F^LT;Im=AADs9qE=f3us`$&Hvc4-5!DMMz;K{olAd^+f`~h?Mv}C9ncqG(m3o_)oQ=qnOq-dW)s{_)8X0|j!V_OQ|xczSG2>^1^b=fXE}#2Rp#VZu2bFngzUT_q|>Q78CMvb?G4m;qD~`7vN!mTT-GBoXwB^YG0m(8+Y3(mbFf)#Q}ZjZb~t&`oy&0wt8u@X;>v2)1&>EF7|3LEx1qQ=xnR>X@oM}LdiYo3qv$o%KECnh5a+{q>{+;OvDQ(VXTj-GZYw9txT5S>>m3H+b|BuRyt)0;fm@5m8ZUTs&7<tSLiui$-30Zv8yQ{8<2U)n9C$w`Xy6OtGSOafkw9JEo{VGRF`R<=E;VKPvx1V2g*1kF+#c@_Q59Yt(Mhmt_Tkq>w2NlTUgA5(ZW3}K0V~Mi_E=Sx6SxNhY|vRz82=k~>U}anE8X^TvjH}Z!LzFG=>GX3U8<`h14>J4Z-Z4AjI43`K}ZATYsU2KUv8Z*IvOb31Y0g9B6JSp!ly5_aHPF{z;NMIF177>HW(OV^bEIfz08&{+R%iq2cF)pJQ~$>-&59SK`fpEAw8qR&b*<EB>yu|-`k5h4n=Zi=mIy|gcCM4e6e$|n<f~K#x&r}x9oKvbB&YP0qoBiCK$e2fp6pXGGrfLVY72Z$4B7`j>B+K8qV?!_J6P=tdHF}cX@o*f`5hQk93s9t#SMB+%c1)`yoA!#vXka`@AndKygI5-ga7tGC4dUQ^cfyMwl>LvR!F}AsyZV;{qW_>p_ww+@RT*@|XGufYmW6iOI9Mev&(BE9pSd<@Q-bv4h*9KjK4TT*Vss#yZXQ^>@`DQ3Y-@qxQ9I0Z~uG8r#~#4eqt*6dBrpaxCl|ci@!xdJOm+4>5_`G!~m*m~B$M=+hB<C^>a%*iU78GJSJ84*FF*OUeeRnomGHt*(d~L{skBaiBJ0j^)O=_>&&@I1+$3k-(8FXQgoArtgmqVb1NKQtjNSzhb9ExjC2nqlF!#liLW0CzXCfP2RVC=UJaWj+{^TJeUrJ^To3nR9R1har~%izsd{VKZV<w=;^ykgw2k4fBCGMfLpYUqys&mz0?ocS$k5mKe-_CZT_~~^^^9`x(Ww&$cCNCM{ir>`K3<uxx;+%cYWrhCETB@$Rz0Q%WBRi9zb4{#sztqcdkw(r^fcjV)F>MJf<}s=oetFul`PA1CwzOwJHnFVjG7;xjqb*w*`Jd$zll}W}}&mLMtX4QOG+@k^Q4!mR0xy!|~+!sr*e-ysgLL(r&*B|B2s-se9mF<D*}%FwG$sA?;?bJ1;$MwKu0pJ*eL1TWPwli>Sh>RZojh0l($7`Uu~}R%iYv4WA)%v6tD<W~yT$tJ+lc4aS|*KI<tu=oD_PV2p^_y`Lz|cj10M?rtLE*Gj5*rK(DYn6&*Fo{EcYggVR;BEOH`^Za=n@JqfH<S}`CN8z4=8)Nk~ToZ7=3kf>sRO(s4lMG(%N{?<y%#TB|y|P880=D-H{4hSI5L17Qx9@az6}2DLnZEY69XAY>sx`I(S1S9BdjF#QAQLK}*&*-f_-d|_4Ycbr(w@y~tKxD3UfQD=7)34Vt=wyfh_1{jXssPp@AfW3QF+C*xNm)-Usr!}DpzFLmD=dtnp-=+B=q}fTR$>!$K7fw4J{ezyE|m#%kAfeu#!-Rom<O}<6~ww!J)f0)>IDf%5_qW%VobPXSKBJGvwOfeDG+zSW5GcPZ3FQde|-EL;v}W2Zi0TtBUj^Z#&{n+LLbuAlhq$R>8exvVZgQJ9^Z!?J5a5@6(bruOO#qG&#%NE?I|izU;rF<_E*Mq^=@krc<t@8SLbY8_3uKyl}g1M;tOH4ABy4#+!6t+)jlq&?>@n&!@w)GBBkos^S>huXXLq=^b0qhjnp!WKS;LSNqfX^upD|oTo5q4GUu9NZxp*{C4GQw(QLj1#XnH^<|`fWat=88mGtdAw+StxX#oYRn~BA*4W06zUUmJKlgW%rp;S<1AdL+$K2x9`Z`+_jbR5oT4uL<a(?C?d~%cc0=eMN<dyy1SvzpKIx9!e=&%`~b%)T&SNKI|N#9Y0^7-TtCU06d8+ED-QQ9cRirqNG>~9`?gk@VTD(mB;*3hi*QH<zsL$3TN{HXP$krq9xp0Kfa{WS9-&To^!t7*Fv@S!qht-b$(^Dm**Q5)Nj4(D>U-YR(GZ_1AgPXZ`tg{<W1hyaFz^L`FZ5wV`CBJB5@pg}O3#VYMr*PYKeIpGo$_Akvt3!DC`JnFGs`)rEpWwg77Pa*(QfG=7OUOeS}upchV-d-crmZ#rg5)|w|wGyh_JkEjQp`Rw{11vk}oDvWK?301NCHjk(jB?(O(rmM|ItYHz0dEFY@oPLP^U3W+UgjVL(M5K`RY=Gm<@B=R1DT9$-Ngg-ae}ldCj=ndPx3b6j|ojA)K5yAMaQl<+8KR8&oep45YCz5?fWJQmx|!!2Wcd;P>Z|ljN@kzQNz4)-CXE5T>&q%D4WtxqprDNaUKNiw>a76-r6;Ke)Jo#6P`;u?9~mfXR<lFL`Wr{{w#%#Y%TvnjneD9TCM>jgo<@fQID=++ylp(p37)Og8r$BG9&lo#?>mKt8AC9F4mG+*QMu4Yt~NNL1!>o&8c7Q`d;$D;9XQP&&|$)&ksx~X}V{!60W0iGLVmYSAM{i#6i}|`ca^f+X@uFzeiTYaX+7Q78}w6ux|F?y4~(Am*s5U{o#*fm)SOwcdF|QS*;-Tj!UiGwN{M3_nFB(bYt9V1J(0*?SsRa3t(~w%U9iUS0nLrMiy`!u6{J=rCXpdm#Ag(B3SPvKYJn9ij9P{&uQMc)ALX7J?EbLGqCy!Dy~=RgU1&>KX0|i@?Zt8;KZ4ixeClU(71<%b{Gcgh@Nh2Ynp#{)oGRN=H_|t&kmpW?^yZPtWkN~l8+M2RigeAba-~Q->t6I?E*~<KvdXXVo<W4GZFME3NT*pLLi^}ck#WxKL?n#%z^9Ac+c?FWV1f3e8m2>U+L@go=(FG%g&UIXyN_n-D6jsaTE;NJZp2ucHQ5e167{Sru9!pN3XvvGzHa4@~&M6Kn8SYOk-ECy}KE?6poe74WTZrcg4Nq)zxd&dl)G7V2gX+Sd(afo7s&4T%$It0Y2Yv$@q3RCx#J?OAO9dO84}gj596Q(^os4dXo9~`J<h5_2Sgg2*PDud)&eoNOOmQhHik9JA6Hc+tpStJMn7sb*(PerVebFoxEscwT|Z4bMz;u5oPuA0cHRNqnh-5a`X3d`qT3zEyJS8eh$OC9vGA{+WDF)m<NtDu*$A?cu?NH<K0APwwW*EUK?0-PWt?~B-QST*u;u?2?yp?LImpDq{d#V*LEX%<Oj#Npzn2tlx|V0iM6!{!=CW5Xx!V<sfAq4G+a^-Vt%I@R>u=x_6Huco0Dcep44}A{rX|55aRpw#vkCnUc#X_*9~)Lc2?`=NUQII_KToTtW!IEDf?$M@7?ti;_eiBgg+)MwhDq8xmx@A<lTfsvMBf;&d#J~S+HBPzlA77AS9v)7!i=hmL6LM2r7+YhlmO`zx_P#mz+xFqz08UX9C>&+G{17&Cv#j_LWq(>n;Or6lc#u$$A%!hEu?4t-w!x*>b4N`LrWle4K<O$J;$v)QOK}V#SOCDs$Oab5k@Zz$y>N4>0Ob6m^tt_>7oNtzbl^*S(1RMM8{7@opJ<C}OMKs9sPHB=M)m$wWfFR&y&XJJe~7Pg@uKmKNwR+7Bm~88O}Q;~5o;YbgPOJa%=HOAkL1(a_lzaJEnnHRD|-kg?2aN#<_l6g~>C4F9l|ui(?{GCFb4_&NUwH-%yjyWLeg(c)J<`+^U}u0I+sgZO%_b?;uq>Yar8f>;~LTuc-j_IR<rns1jfA_+xDk?G3n=6?@6ag)wpN|rI5x6Kjawfb4<g+q95HpB6OW|qN)UAn4dOvgoaNKO?Cx?kpJMw_=!ql>f^L?Oa^wV98)cv#?0+((Z4(m7TU^xmkcM(+jCFuBJBSrJGVIqj}tmmYwZ>-?}H8gIGYA-3>?)9d3onP|E7c>m$xqzPXu!|ve|C~aa9in94cDB+Ga^N}tAWLCx%sPd38QFq?OU6SaLUQ@wP$sy?l^(Our%pV`C`-Lobar*?)3bT<{y-9Dz$H}V^dJ@xLcb9GI*x1DUgw)S+^NGy1h$ztq^gO-ycc)~UKM#>}@gv&*>2|k?@5fzP=P6yPIHtc)lCw|><0;<s(+nF7FHL9PFs?jIud6dQ{Zco}`o=*Qnp=-wj6~#_%@aNwwx)7-+UW63?Q4ubKe@&Y>Z>mBenZ4xD+D(MiLcvb`@85Vwvo=ocDu)1l^Y!nF9j~gX^=^u_|aZb#iK_|QLHwuUO(5kp;K`Cc*e1cQ~NlM4#?d07t((D7JKmvT`Ud<vEGpZV<$EYP-%*1mq|XDBG7stF;|{#FB<A=(rBdh1d_N(>SHXun#eksMk2n8H8DM#f5%n#ht>&Ps9Syl71Q%#@7c69`aFbsBbua4YaO-~o@Cfq?;!Tg3byn_f1{rV&TO0GlASk0?XGrO1$fzCB-ci@)#XCiZmjRB7{~1!<Acq;&c`a$_l`d01jEWxtW5x9_P(ZE>pQ#yHE%H6T=tXf+<1Ij>3CqxVi}y#Y5)BDF+*c|M#fX=uo{q0Z&Y1Z3_*R8%xoKtr`+{5EB=f`;?5atx;L84pVq!Ihz8(vNCnSH$o5y}PvQ+y2LI&xBwk{g1i$dpKdYDLKHM;M-cVJk);w6smq_jQcWCimD~zYDWQ&PsFdx_A>Uv*%lyZeF`PT2XGvtz}otm##4z3+aJIGB3-z~7AK{;sG%bWGnIqlm8A+#Oxc<sIxF^D<th%7J@?TO-I)@<BsTfI8uRN{#*EvY}?b@o25Y<t?d|3!poq|)H!Qg49fqWi(7^V?tw^&U{+$kG}@^*KCKy)brG`Ju+w?<P{c-Ov4-cVBa#>XH3LD!9pk+3k24c&i1H?&AlvtzS1-A@^Cv=hX7s??T^h%b6|JaR98GM1FX%%yyJ;((wf;QLBFdN{<5j@BXC6mm`J7{ZgN-ws!Yr+yZ`2J#(^H%_n5z+WyB5(of0nV53n&-+qo14lA0p?Al;UpPDCdx@)`*F`mM-b}uF3&gcH%am>E{13R4rV%`+}HrNR#<>q{BG+(A_Ha~yeRM+qAKM*=8Owt`gLQ+w*n$T^z>fk2U(N~8WWCTmQ6kD91B=xS^SS~kiPHZAAKrAPaa%|4=8ME)V-^>#ept3BGBt(jM`=<Y3RkRE9xKCG!@mlHN=#|{69lp=QBWiU#_IP%8AN6`oye*P!f8WKk%amzX$=~k7`(^D5zoh)0)SRX`z?JLT;+y12V%r42{?uyomegOu%JnIo<jVE{?K_5YHZLxEF_E)l41|i;XaKD=xTkyg_;%i;d|qyj6KI_7v+%1i&|=5_F$Z1L)myXfuJzl1`32{-^k;<1@r~^ak{xQ`YJ?+eQg&K>Yrm~a-97n}cAzElq@C7lO}da*i*f_=(KQK~>1I)6s|A<b?8kcFZpN#~v-#`Kow%LVx65!vQx5ES2~Zf{8D|2UHRqBoPvU&Z_sMLD*_R+$@2fK#y*xOp(YBD9IML2YL6NAh%I$iQ2=AK%;$nsj-eY{#QV6pgnD1<ifP#)#RMtW-FL0h)qIPL)%KX+=Mu%GZV%<yxqHA#3BA7(KfP!8l)%)#zH_}Z^3+WsyJ`eNG@qKstvBEIS4ZQgkC&x45RSv`e2eLG}&$DdQ57oz9U~PK@BKr!!<<*S^d)>ILt^4GzZ+BO<g%tv|2IRSx_xT;$OR<adTkiMiJEYZ96f0PatKP*ctqZmX>if<_(#zLkcm@yo8GGQBH`Gn}O)>;sp;9u!Fx@_RpVXw<=ItWmNb2O18ymFEuteY+tRNxdu-jI(JXQOy*<|Xs)@!~ERwIG;`oBr_Hh;hO`mSpYeFu63@?`2Jt3g;Bw>QJm*>{_kf-dK;pr0KTcSC)3$u0*gw^i&me{`8N+k?qis=gWdI}ydh&5>WA4THP2J7M)r)M>HT^|cNB^=*G$!k=4J>|;IeIO3N5rahTY%loL3d*11(s~nSf)7^^(*Gv@b#UVW#<SMG4jAuUM506XN6Otx=L$k`g2eIf)l23oh5(Q=0D*gaP>6IE$s<rvi&}02125HvL@8bYedffgDM@q^#G*|~J(^&14OQOL9-R^M2Ve3&i5kJ)}e!mLSd}zAUuI+A?C8rLUGqB*MZp&?4b3k5b-PAMR21BZRZwic9Dj#c(@3h4g$rRJh4$!}xuPT!*H^1xsb&#*L<Lmnnu~{uiTGYFgv^`Y46w4hX-HK}8qnppB1>?`=Ex3ky;}HX04a+jYu24ZE9#qBXqT~17%I7@@dp17fWypM+Wk1uFj}q^c-htRgcG^a&MEjK(^U)Tg>+2GFQoJB7R*fBvXXD4BEpug!IT-9`e8_9n^B%UReR4v*;1#$3?(5)dVXT9z=Lu&Vz0>&YTJ8A#>4#@_t)3W&-50Ol<g8`66?aDwV9MW5DSh^Xm`nRqQ|YtgV{+M3p3`GyL@YET?hNV;cUVhBYlc*!xKqtq2?Z-wqj7q9RF2!TQ{#@F3ZybXZ`jh~zVGlGlecg`+5xff$e!w)xTHJF>A4NxTcN~?)|#q1-Z>qj5$p0z*n~5QuA=tz3zG`b%o_g3e4C73b9}wLe2)GyA0|%HI0zg~_rNe-4k`-qn9T{rJggRj2PQ}dBWqgdwyYO5m=%EKXbz^*d<0`7wAf9O$Zo$@f9Utpe-8xjwTtL*%n{<6*;kLmXXlGRTLby@nCZ%UJOdWQ>yX^Eq@7#i929ERMdv47RyAu>*C9nDtN`ut;$ZnkR*l0~$Ob=dCFGAY*<Dt9=S_Mmp0~po3eQyUC*tTP6F)Y>^e=T#K=9~co}%5WFnh-cd_?EBA}khzAVyQ(w4@LZjOJ<#5d&6OUv_r2{@_aDOQl$Vo+Kyjrt1fS;jjI0GbFhE({2T;b8B8c*cch;HDTuDuqDgTT_*~%m_a(R3TYzq6M+l3xU-vkUAcpV{rJmZX*gn|nwGEOtUlAFJ5?@B<<gSw>&ms|>n1g!uk*kDTT?GEz5?L$+?#D5BJn#jtA|?FZO7}6G%RR4l)sn{CMN)^^cafa`}uCO;E)JrU;`8%wQG;Grt0H#Emx-lvej#B_e9|DVIOcrm84uA!FaB4PK#DSdx~=l#GL0HQZcs0V)5qeMHK|~XZU3F1v#sX-=S8yced|dRt=wL^*$Q`<6f*(cV^{qK*@OWW2e_8a!?{{M(~CB)@Gu+L&Wo<<c5UL!w7Z#p5p7Q7Iiy~O5QMsVdb;l^H?WJNWqMbv7i;dW^MFW`!1wnwCEY)<z57;P53rdL2s!wQYok=F!ELbr0(^jbjFR5k(l<Nb^eWf=eqJb=&C!scpyt4)|}bX=?hK*d(Qq5z@Yd3e9W`VE&e~}MQvHsHR*ppF#11MbGAOYqe0TQeI0DG2#M=v7bltn7srEc-@;<>g^&6>)W){Y0)i{oT5BY}M~bv%9{%fh<Hn&NA3vdRsmY-b>C@xtuw|I-klH)%!4B)(_SZ4=0K3che%l$|XO;!b13lcKa`+<NM%iqAv}L&)Uza<M(G>0jNByDt9<2!@_!67lx=!6z)$Z{U9OF$-<qH}ZoG}Uj)@uasc%P4vCema#6f2fxCcOz~V-54QC<tfQ2Rx=^p`2}LpUHvvbOQU<XlKH_&FPhvfaIodWu+%$ug~*|{s)FffZL8AWl!RYALyQEc*p$<m?e#FQBNFL+z{P~N%hR~=IzUvfBauAJJ94e8{R*WxPNFErMNbYoF+rvxnphAczUO`(BfymXjCn_j=30#=51VWW|dv<JLq}2Z=C?++zznaYgH}dR!h0cu?wn`yfyR<udyl&BZ-p10QC2)zD`0xSs(lqpp)$)1S<fY#XM3wso?t6+sRATDDDrQp|lt672cG|2Yk96-h)wfC;f3*&H7wB?wFIyz%?u2bG3(BOA@IW0Mf<k?SAjA!yk{E66-%CA3e(`=T$A9FF0#CJDjs;gM-J0c=+5;5%!V8x-9&4!0k5v6CtnV{HQ-AJOSI5@7>?~|6a|Rcdyg-dlLlWCTXnI>yF#5p573D;Pd#3j?RyaJuLHda!%m63RT|vX87KqX!Y}4-`i>urKd}!57?*T8~^%6g^19<C&WY8?NQ%$!-%fd1f&1{H#IsQ3qP@iM%U));!Y=GyC1ISk=64;K(W>rb`bt-p0LaEwI=W15Z6MgUm{-OZ<6adV#X914ha#*&sA(xD}%)V+fR_n;|fmD#`Q9idk*meg0|@52P?@M!H-|(>7r64lVO$HjVi)?`rlKdasHpFQEzu%47d4c>+N<oQmyx!=-EhL+M!$w>)q%zyy_mia{#^bfYstdy&W%{2y^|ro}Y+SMB`iup%%KT&x9SbiyF(i)B2cSQt0#^>I<nx3t*T;b81uAgTqGNO@$+xr>p5|`90jR$QW@Rx5@D9cE7vXaWMy#<=M=Rwd<;NX>2#Dez}OHx@U3U#fC>X<f3YtJU46v(A?%;(m&(pcBQOXQ{tN*!TmN3o0E;*zwBC|F-s0m%X4Rqt%;u8jWS(|H){!9xB1Xo4GbEp^|m0q8KvQ%>oNoBG8;@^z*`*URm41urhitISgq-q?PlWWVi8qj5%1J~9qf$q$+I12CrR@gBaMa02UDOE=?-6BqnEG|rB1ykRnYBZBTMa=`Y~n!Ui*D>8b7=XftWc#%EIAZAFj$uuc~#&_p^Gg!mBR7)g!u>9E06x33LbVWc5H6?PD`&|CaB>?E)wFE!3ppXE?6I+zsI=X>?oG;~Tn1=!_+C^i8$H0Ve!heCjTPiAR(0;InBq$;1B7sy$PijV<$5IbQ+FydNy=Ga7dF$%LE-k;GX%N0{U7vAT<<w<-d{%H>S8-5t}IioLV4lldc{_*DB+->(MzdW24uF3I<^UU|vs&o7Dr<Xcp}>xYF_6b2YVNQ;|O3j?@cDIQdFa6^IPifhHS8hWRWsKOKvx>vbM55K%gc2B?d!a7{#wTp4z1~?DwY8`|qY5q0W!(XjBCCW$x>T<jeJ2@LU`heKF_4Y#wk5vH|6tjW=-+WC86Ne7h9DLg!o82k-X=J$$HlclS0Xf-g)8H_FqtGJk+s-%;cBQ_aRr4WHyEolQ3<D}&vzQSLUNM|ikY2Z(gkiS>?<GokSC296%XZYp5t|bE0D}VSWtG+K(R#J=zt+$UB~TqZ&jSbn!vXRx46pfx82py3TIbsHUbNlt_)xabm}#(`4w6pX=dA-m*)OQRp3CE^EA@V8bYcyGRt5O+8WKaW^OU0}`yp0Ev%4f_Cbh#mb#j$)b8h-<)t)&%;&60%!zHA?4_>wRE61)ky!*3pPuM=bODBAh&4uNrOX#>I2W>C-Gsr<J>Qr^onNM09o$iEfl>#%Qy8uJRM4V5ElIC(2cMhXsFs@4(e0v9#MRmvaC!7MNJ@m%%!clsf!{+D$!(*w|l@)oo*+P2VgC1YELG2HN^OA>xT`P1JojWr7LA%jCdz7f7@BPzGv*X$6LYbd_aQfBXJL%INJ%?P_Pm!t=(eGPj3d`^SbulVl7|jt>kKZ+^HbB=B(rxF_PZP-zb0OW6fh#F{9Gx~Vbar~VesdWeX&dq>sN!UI<gQOgGiQ*B4~Eb?;QYm1zP7MvzErL(!g@Hnqlah$e{i$1h~ZY}kTz?RMl)Gyov5Fxs%28XNg07S{q#<MB>q^u)$_0R;n4f#>W+X+>NU*6i)>i_aX#i>RZ6tqpXrctyZ*Zr)zPZo>aB1{`dHgxx_{2^{JFm`2Q}iFa(0c8eyi?z-^rV+ApfpNf*&^dvTx2@+Uo;kXFXas;Pd%b@}GbfXMGKN`}r|Lha5C%QLX;50i*U^*fQxraYtqnPxkxQ4Mkt^ZE|y`x?k%Zv$(zU3(I7bvkG()!?`}TKn-UmRE_%pa0V5;`FL_PIXKBsr5EuwGjs-MGk+%e>9-6nXz-?u?sgSHj}7hm;<Y6%hw9hr0HFK54#+kU5S3|*GS7?2`I`Ov08NST2K(73(%Gi=%oH`Z=b*dEhtCtxCg-Sovg>2yBr!rS=)CdYQg2C@L3c4t#xK>F9~$ij(ww$pc{MWanHFOS7hR6dxKlCZwEqz}5YMQI1vWo&quSC2!(lC`O#b9zm=~>m-g+z~!23R~H5|Qmm7h42wzI8k%UndwLBEle8|jQZD}MV`j+D21NA_nw3B)y}A#kNSLM=`cuGwev8@Y;OY^<QUy2=l_g)JuDWj?`>fo+GcXD~*}d6P@`r`_6Pt4C#ts1ViJ9^U3|&;kgncdhf0*PB+A=K5oZ{&hkWK>gCG<^Pt;oqCtbHb31Ue_!>8<>t<H({Vlm!|rr`imoVHB-ge@-Mgw~1E3biKqM2uG>WUZw|)KaFAC<{vDJuAvxuWAzxIK@<CT$pt%XBpS86wDD6WcHq%DB6`r7U~Os8&s>mg<fV0nmuA5iBmj^@mcpHt5_=q_a;=8|WIyWt}cg3ll%Zawp414l%3srgoEm>+IkH3PAouA+ukv~RxGK341MINccd0)U#W$M166v<c%C|BjV0RnxDc4ZnKb@www_b>AZ|&J28vlT*1JYi_hfrNLwa@WGP|&zy0+zj2=$a%qvlfACrvVDo+0IX&^6B9!ceJoN8lc^{`v6`t1ZVor4V>V5P$739-hDO(1*GelyRQ{Ey^`*uYX2cPFcXI2PeUv>%|;kK%$u$<PvH9OCh;}u#Z;qb~=tIIV53hNtXfxFJ*0W7z@f0IAPs`)uZpW3*Qo!zu}R3qV|S#Pt6gUr&rPmEhpO(qsM6EE)*WY7G1vd(_r#s^fdVunf)2C+s6j34IxMNnT9Q{zdM=WXD+BWAX_>A;hVP;<XpZWpg87-`jBi-uc*uv_8QW0Gz0{Ck`+pLj5yS>4V=+fJqSC|WJ6uz=RZvSY|rSNSlyQXz2aykl;da+({nlHZMnxFXy%=1l7x_W;*3e9C2f_tFWD03s}{b9m$I!rtvFN#)VVHd?#+J_JvgX*55sSSM5oJ}m{{or)K7*;CGzsBJD8)sOV%l9+n(aHsBd>b)Tcb>O8w>Til_N`JL><(0~#Ufs!2mr<A{|D(|AXnn5d`cON<E0YZH&KF}PI40qyuE4j}{WIR8?99dBxU6nY_1T@RIFHq|O_EHarj2{^Qn%Es53~{bKFatPG&In3c(<v0e_6XvL(o-+%jsSpFd+BrACtRit*aRlMRQ=>DvxFxiW}d?_|<EUHFYwU(UPfY4mP8$k)KKa>E2Y86s0ekRA%(>(p_Rq6ec}ED(N8W`SPS_%}AJkrCO;bsr{>VlRYx)`TQrJ-;BpZ?mZ*-O=_F2_G>p!d8q>Zl+xh2cd>@^pgj)sS7XZ0HynO0=&3&bKqVs>0siNSoZH3Q2TZ>%{ZzO#`i%@9!_<Cynxd0@JcC=qCK-$v$wF3hVmT+LD;Z#(!-{|0G$+Mr{0P1;&^fNX1n0Bd|AFHcx9V;toY?9<^e@yEp~K=01mmi@Z!r;`X0LM%$R|s~ERJT1Ma}Z2W5=yq2A4C|T+%GU?wijO;U@=gc01Om_(_9urCaTn8^ynC&9yj+Cw($x>FIM?OnZPgF^=V@koffhN$`)TXloji)l;QiFQXHak;1AmB<HCUw7S<Nz}blTH4LWreFY=gAvaq@c(=OhynDrZT-g+B!Hpv1>T51l)O<ayWudG08M?-GX5M!}dGpy!aQ5e5S~*NhftGu%5s9-*)_Az}Zw6gkU=HH^T6j!QBo&<x;2m1Vd%2xYsP$ukA;Eo}eopn5bqxEx%}(hLX_^9i&m_#IwT9WQH9VGf?u|Sr7cL1ivuJ*dJy{)|rLmI$&XYka#cuLcAsX8MVDjqoa%i4ghl3>r#=OzHlquCZ)uv{IpKj(YHEMX7B(URFJ+1HEHUkC6#AetPlg-JH7hg-N3_A3KZcF)iB9Gc#=C9Lro}1^sz(*H@m`uDwzf)Wbt>>AcbA<ZLsE*>P3+w_ln2k>J1<E5>0Sy!d8KO4|-d#3=;;&c^od9o!MVj*!YY(SOvejs7jOFzI;_YPitKKzkaj%atubVSu6?bHcJ2>z}#ydyW)UE6?8caW>lB5r4Jox>BVS=qhcGqZsm4`u%T2ri{<<we@wE%i0?IoX0za5P-XUoeE1Fgg2TdDPNa;tUsFLfXCiH22o)6M7>pvuPGKkKG>ht@4#><n77n)$jgU^7dxX7R`-YDT0uS?>7{^6?Hx`Mr(Qehs>BK|?0$;pjFCZihb1QsV(8rif<r+61w;?NPf;pL+M#@?*m38cG!TF*%e^147sK*jo3yp&ja*W7ph}HIj)A(97Q6*>1{DU7g!F1n_9`uPyIR<D<J>+~`*24U#{iwdn~%;z20IYxp`HUhbLVr?d)gE#R}{ZX+8!Zm9cT;z^25Zre6YFhC?$neBXd!uE;`)--v4LSf-|>dG+YdRk*yoN97yjpDOk_jQc;Co4h7`)!kd<^#NjEuWN2>3;Ovw<b}wf#pQ#>JaZXl+8+!mcRnuj;<O(#Oc)ewN+nj!108n^IGk3E<z_WKC7h~MfKUKY%QxLi9;-*Ru=kdsu-8Fx;*ovYftOB)rS2(lt&C|zvGnOZa#<7(BR;1ojxzKt2OV#O7-~e1sJ=yGh`EdYmJ~U&7i+SeTNi#x`CG<4-{26>KX!s)F`N)2ulGI)bibg5}h{KdyAAGO>~Q6`ZIH=)NgT~#+y_2kF|Gn&F?M*XF;Zu*$*{EVwSAAPKbc$cZ}8C&0g?YEe(=ptG{mf7hxoO`E*=X&eoIbcrD_$&`1{DCYeum*rl>|4|FAkqa6$!gBdig&nrEpbG)eI``7n0wcM@VSMc@DnvkMi5&Gf1$6I9s<L%A&V_ugyKy#h@sM}fBka;?NuV+S$U(n%k^e{q=I1~uM)V#l*Y_(t1Vh`8bj;Yyr@uO%5D9!s;fxwUXNmiJqd`i~+eleaNTV%!(>x!q(>H+8dop`p`H<Gcxx@1CSaKl-z-`Y?m^qghzu}xNfcbqc&d*C7+o<8X<nAUAT2Mbs6=f!&4e!o#rdV?(2caOD}amZC<nz>WTDvud`cE3S|8|H)C0GE^MEsS(t*c)K1QvH=~#KZRh2^ZMN9|<dxU+X<TRvAdBOTGGXM)!Qh?|6aPf?M?iHZ_`pvzs(G;zo$h@~rYStnG(8rxUN2B=2wIXWz!z-U5rc<?fwbTZ?*y)sP#f&m<C#jmlw!y8YTOE4u69?o8inuoX@D${*`+E$1YkmA&~^!VA0h_|AH{ItNUdt<^is_1w0P?H0Um9>$MvCSZ<2Bjn`O0D<FR)wIuZW$q_MC0E~g)3-0ArO$i&U_QZ@#a>$UE^b(h#i%{G1R8qqPV8H;Xa*G+zl@H~LL5*#OY`p43y(@)=Cqw_wy)7EKOuvuYd5n-HY>01(R85!GkWgT7X-R1bz|R`wh2&7@cSW%a*^umZa?WQ&#s$$FD>iFR9l=@#^7C10yEmuMyd>&xnFd(sg+;SyX~>Pn$E0seSe$@e9R4+A7TEri_<O3Kf2UETSi#J45|{Ue@aVsg|-XhkahwWNfTiFfW9f!278S+@MAW-|5fB=ir<R*=P6d~DRSmBcyxX~a8PKBh4<)sZXE09jyH|m_1mr;uVV?el8t#Jxn!*qizUpqu&@&zHjh^dYSqbU7Ca#%R<HHLAdsVe5`~w#P|aRoJ+DDHD0ZIRES(pxO+KMSsne$APjr3y$1n4SiyGX0g<vS~2^>{WV#<EIwe`EY?#y>lIP{a%b6D!1<GF)&*47-|f5upNnPODduJrp#^l})dgC3GD%r3|5wvj%==edibU&AW5Iht8N6GKPh)cSSN#^Jg@8ohjTcDHV}IF)y@^B3LptaZeG?rVcL8SQoL;5qr-HrtTjSakZQsS37AznSvX8a&+Q;^LQ$)d|rmoxeaPvJM&!t)!8hb@4)6azBx4UT7$(uuVJq<AJzsTCEKb<K8qYzudrc_SbrV8jN&y_xmf}doaCro>@du`U7)n6J(Uq;${_|Uwx<v-Q<A_n+hz&!`6MC5ElJt#N)eqRS&-z%d>6suDvBALzA+xJw(vq052Ms(eu_s`X|hQb8q&pAKO0f^R-VfJa2w-UOV{satU&V%3=$>W)kfctFQd){!$gI_51#HDb)%c?Hjn(nK>N$MnZ3GH3{$jk?Qlsyw|z@o$XBvlhpkkZ25jw+lxvvn{4(!&$Hvf^g4X;<KlAIQKI<EHYHiD&0%76+6V4?*hV_nz9OsdZlwg+>1>=n`}nU(6*o(l{JBE+2>rO|gcHx|+Al3<w%_pb7i`Z~-KpXO*1L{wu7+veI%w_4Uwy<|=xu$}v61{M>6R=rL<QPcOzDN4em3@PABU;NG{xULBWp*9K|bySY4(g7YUJp)6OMXG<7*wA-t>-Ti~59Yph~RMy*)-Zj{XXYZD;oe&ZmKM(d1i@pWR(7Saa}4K_KC*VZO~g4-k`4zTM=;FF<Mip~B!*lPc@a`%3xF=mG`GqXJSl#lL>)>AiEqREXQo&{;fD+V7I<IPULxfm<fAKQj~SJng)ap(b3V@j6qxvtdBnhX=?Hhi9nyw10y>kbs-`nm1do15+s1jDWWGthO1OZ8>Z5eO*je<L%2(TAi&(Hm(#?!Xq>rxX}o6tkV(VpQcs{QM+dED)F-Z;*h%WaIGg?MdRUkTlED})Gu%?SOJmHel2q#Y#6TN9--^1+Uty1?9%4q+#jzQIM8i49~_s)boga&AiQihZ+oI*4>QH)mgn91LDuz+)wSwx_j&{SR<@;1hv5+qEdRj%A<dZ?+4XMMLGF8Pp~lk^!`6Le7D}mfJPo(hZCw~Ln;YD_&i7K&&xSw`n%}6}_53vGIh8yE(Jn-em>FD8^wAU5YjSSGh-S{V)$i3^*t&LF7)`uN__@zV=X13IGxq1Q_!xUms<hIuRi4Nb5L5Dj1n6cpxNaKqSV));G`(f5Kboz7H)ZorU1oQ*znn~Qc3jw<%e2*qeH{*O9rF-n#rIOjlJTcMTonndH}1i39%dz6gqgN-*t@iSO?{yq^cV@`W~ip(XjmDCFHB~IxzxO0JF`Buuz4wI$Xc&>0+_sCSGYiL6t!O?`GL{tGUm0J*0zRHYOhAVcF4KY^RyWZY~Vc-dh>4MX0BE{c=#bBeK>EhhvWF21N+K(t?2v&Ufk=`EYbOQQcWrbE;srM;gR&A<3`jGJJJ6Ho9!!_-Fr|D1z)iA+N`Qx*Hici-~4d@+Ud3?Vg@7ZE?y0$%(hET#15i#A>c`WUk)ct!g<NO7Uax23k0XJ^J!RM;KO3{QKyBRF2a)A_JddbYmTpt>PnTtK&0kEMm#s}r83htgh0{9PYgvOhS7GWo85siZ|*Ya)VePL+tw(MnVQSUIqO<5*ll&IIoGxKth@D3jg!keYR8eIc>5RWbY=0~9a}@TFZb>DtnwhWZ*V_6UTB5Y5}bQpKeg`Pqm7Y}uTpJcLmoa(C)ns(Gp}?*PB#$sb=YBdHBi=7Q9)+(Bpb42uUB1{#Gk@7s5cEHtD@#ouj_^GAbH%h<)v4N-P-fMp9A{W9}YFcQJeGjBfSc?b6SwUPI|>vhs4K=th%U{S6#6oT+QUgHaLFaxWGE0!4}@Y)f*Q(rQ%cDd>&3&e>k!Rr(?bU&{3I@H}yLQLf*JwS$*nDt#I$h(lQ&~2Evr*CG5-ZSJ}<-KP^0-R%d_%OcA5Uv@?8<`POgPPf>A(kC*mb==|t(6}ARd(4t@8+sot`@PN6*UE7lS(_qiD6#Q;4w0G;2^SYa<>4|N3)@Az->@>fJu-rWcNRMV(2%$c?nMzc=W=XJ6*hZ5;jO+EfavM=(ri(qcd3hM$z`As_jQuF|{EYkbB~l33e7u?ZRH+wGZ>TORGp;(gh>@j^enC2<9Shu{9*#h%n2o-C$8T&<x9)B$b$ce;jnaKURj%D^xzhpDnGgSzq@okz1zw^1<7Jdd**&^B34;8>;~k=%fVvSZHhCS6Tqj?3@~YhipH^p;tBssKZCd!5h}l8SPLU0}+D`h~W4@!ZFxug-=bCSvP&vl0`s+<Sol)+tR9GGijzAjDem|_z0>raec3Ru?JhstCiW1@R!913&f;sJuZ@Mi<ZDT5RNGe{XTDQZ`H)Q>SZWCpegJ$Xex!dg=4*)A){CK>&m6y}!B^KjcH*5Clpfz|32l`j6dTE$^^D^Rs#aV#P%P5^RYL&->Uo+E68S2wXLO_egAX_O#C(H1>QrfBA_krIpjqYw>VK%HL?80C?pZ4KH9j-wjQXCwklAOL0*ZTCW+0u~UO=UU1Yln+ke=&ac^UN?kpPTU>r{Le=y5MrHS@S-SdI4Iy!|6DGX*?ny`g1W}wd3a9FXxdPLsih*{kr06tXzwS@@aB=zc3)L&plYqpp-i4O>W-Y!u!WxK6}!tpKLe-<G=+A?UvYdc-hq#%~j%OeQ;I&e#crT_7ZvlvEWF0-Ou-NwtgWqW0d32YBZNx#%m^uwO@SblvDb7T@l9lR_PMs<d44dv3}S>T%YtUo}p^6jq4o}lxKn?>@Ttgs+V(nq9%8_epjY%cmu-OB~<VB<?>~-a-piG_!#I3UiqR&X)z944SA5P&do|~{%o(P96FmDAh}h4GppB<58u#^`>GxYE9weo@O0grNHq)ofRG{dyODeQ22UmT%{fLivaX2^)MwCmO60QCkb{3)m|{(!40z0(UfFPCE|AxkO~^Fv_aUku80gw;@DXsJ2aEE@*J6J2#^zn~s|~aW<bVzMYB+1~Ltf&`VzD}z$oX-i+Rx&xm^nZmCZUs)XhT|pwNYQnF4gwaeD8CqvT%gzzVp1%yXL?}{AOv`&q-E7YwYs=MphzMzFp~YYL2{R(J;2J$YmFsyL+V_`MRiRii9(nDx3g&g;ZlX-ku{K;fMS_NP~NO_H9c=4w9F;5f^+%_!F0PjOIwaHn>*B^7Fp1!oOuIhyF1=56Dgy@%$~XwrBgaFE*{kc2M^W_)m4m)Kv$oJbZH(8-Z2O{d@OVt;u1}p8v^p)?Q{WR^gZXy+0`8MRkA!_mJ=HtrmJT90*{>_I<WiJi`BsDemS!oq=B6Wy>S^mzzU{1uJKhNbc<meQVB68cR8G6xk%&ak<8D;Vn`bTG!b!OJ3E@VK5-O_2f!izO6|=JnR*jsZOYS_Bys}>#I~bGG`s1%q#FY`Q*V$T+NSwQVAe_$R^G-V3*#uqXY#oukc;iy)e)Ac&J2x0lefRgl7cU_TZtOJi+^E_17cvu`{uk#%XV|ZKMWT6?l`jR&-A_kqsV!D(ju5nDrslIW<bR(<qqyc(S`;r)z`T_2Ou;Xw>QY>fpc`N;s77*4b7MlJ2q6I*ny^zQ&tmJQkZbW+CyKFYp$zO%EU}4mJ$vjH!$~Qav_q@z!F{tf;<YVIQe_eBLJGz)kjC>;7@h#EhSbYH>-+x7MSPU@;6H?t^od=k317ou+pp`JLfKYeauO;c{lJqfXkSwEVN@clnNmj`n8n5nGFIOmeA=sz>Kkv`2ALIlVjTx4)%&8#GW#Pn-p}JU;HcqUMd}e7ke|o{#6^eWwml2k&*6$1_g_D>yCoX^*Nc$f~fQQwFN-w4C+ozFL1S^f4>J0ri+Zx77R&Ay}_j!7o$VQ3NYr|B%tKsE_Vwyu7vBAG7!F84SH?By*s8PmlK;Y*sg_0XMtaBhLo%2Y7TT)55^b6KT?iK?Czv(fqIA_o|L>7yT-nwdVP0v0JP)_&qzDREmBIU?3!}vh#20!Av}HqOsqrxo-Yp`tML2_9J+(TQHqWFzTv5*zUjOo3I#vcPSU}=0mjgd0>w7*<(IcW?JQ$HTDN!(lEa)vUPfAc4;Q$8Q_ly0#y{H`n}HwRDT=NPrdwZ7T>X3yIlT6J}T)0EyMu3<FC@~Fdn4ryKSHMo1C5&QeL~K=8)by3yQ>2+>xa&P{em9gyl``(NDy6WK1t3dHLdMN)b{!ZB{|}par0RNqfXzf28g7{F88!k8n2i+);m+UQjxCY*>ugkl<z&EZO-=YQ=kt^3Do`6X1Nel+5Dz!V=`w0bRYI`HA}&eQRkT^!jr!CS~QM9Cr~w>1){vD+0@04_b0+phKprN8Mh(_WR}l#oi}T8vj|qU}UIw%o8m3>g?vr&s2}1@Vy=kuJ>!Uu5Q&wxXwi3R12*BU2c5W2Wuj>&?`;xXkYvdL7<v71N>awmoT?hn}_LnFzu(uP4EEDGgW-BQ>TWFpSnNWI-kK{sJGZLmhR8Zw*qZ*J??K#OY~tL2(klgt~Rf6$2NC_7v@`VGJ4ToV3YSHHoZWek}JmFeDT+!N9{H!P%%>mkZ;rb`w*_`H}$wSQK@LgU0GRfYcu>=`r8k+-+c$2gnpzeJ24)qjl0=`*+|Iq`nXJzCPK(N-e*QFRqJh*atJQ{UE^nlrmD9$;&7Ur>htyA+dUn}J9-C4=NW8?r{B5aRJLHLOgGSA&t1Ji=RFwnzhje1{_b`TOy6}ZnERt=bwY?=c01fR0M`Bq@)a!mm3L!YL!7Pn&8kD7dW`yB_s?yGCFrd7EIADuBI2!$J#||d5+mhsfuEmo3HA4<ik8yK5iM`i&GIkR(0UbT_!kuK>6H5SVWMjdwAbs=e9@BHd*Ok8lZ>;9G^#^WWd96-A)vJ(qhW~Rp6Il}5PULtB#*<o77^oJ?PHEwH!ElY<|&k-|Mvi^A`g1v>i^8K;&qYuU;kEKXC3w7ztwZXp8l+PLQz<AGch~8Lt~qwB-w4ZsCVC``mbT5>R_-*?aq@gZY{(%Fqrd;?m?6@nQKL8K4`!GY*KhoR46Rmuq;N!;J7;MXl5%twoY}dune336^=08u#95A8Xv8-K)l^sdb+fF>xIGvtiQdfDojo@kzfbbV)z_PquBchoI6<^j?M1)$w|!qiUy3|r&{mSPDGEwkSYMnhk@sj=idcyUgaCJsqi7)>gexXo9a^CVYq#-Lo_AUZZ}{i*IPYs@pjw$e9n#QRDjju*zZAT=4k{ZX!7uh3>Mr+O_<K@Z5RFc;k9b3&R8a#%BLd!dE&2rZ!QTox8^!wEWtQpe^`(Kttw^hDDthRYhq<4mEWXV5<G8Ka;<TH|BBXdNuuitHa!%BZPt4$-hH6g=g$*77(zj1i2^O>2Vxw5Y!|@>Tt7>6@Sz3A?I>OMM%_#4_Ud_KfO-;}PVRA)Dp*Rk@R+zEURQfky+)PvzA!Dyss;I-+&r*z7(eLW^@Tj{?|i?QJKpoL8X~?6_r)1;{){u~aPnLFl6aSg1Bj^uPdu|-S9W{2G&7W&r%{<8Xv4i?ly2R0Q@o<W-?VXgw@b}<O5o6{Lk)m4mq466%r1U<>z%2Kb~(?ay54+~rU2Igu>z>#TQ`^Um|P%7#QSMg%Jkxq)1&I^;Ui;@yF8t&^Fim*w08a6at_;bh&nDc+V`G1hQzhH5BcK1-Q#wTbS4X=Lk7UHmiOm-;P9Vir!7vLFSpv)64nM;b;n#Crv8nsYKMaww5jz5|GGfbUjAonwK!E?Uck9n|6Cs*p;1el!~b*H>H4yH@h$q{&y@L(!gqnY{m<CyTPqu_gLNHDC+$I#OBb(EMQeN6p*=roLs1^?FB6QY!rjf|_Z%8sesCqc`O&GIDpp;$sj!;d#*HCHkJ~Nhm?blmj>rF5c1rE!#qs}IcG?fhL02V`&;k!lL7LHZ2{5a(uxyc>50B?JvY6og{$Ox8WKIz=4gQG%`VQ#AS?|fLzTNNB-LA5HLQsb~SbNC`xlL}p{bZF?MLks>O`9fu*>g5{EfWZ8&7Qkxdg>6J0NY{C8sjVPY>xLdsMW#VGKthj26vhBlV!EhS<QMdcR76<D0%iBe@|4PX}z_r-IjVRt(J1K?7fOvVn*9?iK@Gc_3LinKSASDR`xzWiv>w;U8oiK(34Lo{b-~{I(Md^$VAlV!58<Fz6WdQnwLdu_bjTrS_A6?MB7K1ox9gyxxPtIJ3Lk&qt&k7oqU48nZ8#~<6!H292sKG?3UJpO}58pgLt3l2j}%8b<sepII2HQu1tA!%N2`pQeg$YRvEz)y$`DQr^8|Fj;qBoAlvg1#`uer9m_MrET(^bB}~#To)E)eFdGBl2OE#iE&tv<#2o)^i&ecL|CKe*?5npIa_GIX?p0b|9vuJ~25~Z$$=f@uc>J?j);0rNpMQ**UjwEmzQ0oLhn6weKQABSu1D+WaQaOSIYsPqQf9aGSa*fZUiyP~zW5a~x_EmpJ}RJzF<jMp<=v_@km%U1%J7gkYp*FV@BZ#9<^0XKjsc~@x{aN_=j*%)atR@?J^K^39jh~$?PN*rAoCWvJ=^GZS({D=V2?I>%fwhw!~3=|diS?Nlr`qvf4@BU#}f%Yz7%suC8O40O|P)DvWTqxR-cs8ecOW*2a;2G)8@}S2M#xsZ4s6J`ZKM4M(z-1j{V)C(7&~c6dcEmvi^5{bUb**slB<*>0k}Bgieev;U+9^YkM}EfiRJsSF6b~?cG|3$KqBroCUM{c_XJYw~e^k7l``;*_;lI)q!4Lc;#(f`qk-i-LH?X&*P=?K^Gk$;mL(D8h4mIRxl}gE#$60V&}B_ojKdSZ>Zeu;#`+QD7<S%2_Kc5$lmL*MO^&FCK>sG)=U#<#NlNs?I!tE>1}ktubofZ`z;8oAZroEcG(VU<Ia0JYkjMW8~Rq({fVUQLYfb%KDcjp>SItmE0*!8KL%^{p8g{BS%I73l7=jpK$o%B*mPpBx<3t0j-?Xy0WiI3;}hoj;gIZfoRNmSXFnov&x!~9G9@S11a|-utnwGX+U~dT#`^q7crRN*yyCDG^K&)x>zk!^5scPc5GNM5t7l4k;SMGbFLcyD|1Ms=g60(ULx<h};_OU!mWA3SeJn;K3^5TwlprDqD9Rv%g8mRE6a+;?6mdBA{k%K*vOAr0y3;2>fOW5`tCC_t&&!n<kd#E%rpG|IpU=WOkZ$r(c{kx<=TjhN%UOOY5MNtWVSmQ1eSjSo8@XIGPCN8&ZKw7!y=TWPP`r^c@vg_=;5s6aa_4St&|wuQ8F7B?mmR7m1@7_M57rxp6do8K|M6PHSE<(E*F;3(5jW($vs~9ESUEmr0e7=IX=AvQ&dsdNS5$4ITKasHj7H2!OsX*7CE6YMUr`2v(lU{5`q{98`J^W+;c{(BYn@*iKdY+QK~rHU4KC=#TLyNWs4(3;DC*})v8;`ueaQ_3dcFSMOQcWj=*gRMAIfP$=gQf2N#rYO8Zkcr&9|>Ts*QegbK6(73IaDbKzRsgGiV<DXw`TV(Tm*I8w+DTB&K<zqk7HJ#~Np{=fAMqiaNKOTj$+8&q$Sv{KpbmN4XHv_jawn<iHZ+6aTr%@_X^evxEc0TJ)_>clBQF!!<$nFDo(jZ?~-iBi$7XH^}vuJQO+(zTQ`dZcTYjBRE?(0fREZY;yhP2P&&fySLqjIM}t;9qglys&qBUh5KH;8L)WoUK++0SV2z{xG@v?$X2@ktyu|M2R6lJ;13Y&KK*OY$CU<NAq}{vU&kaKE|5_#Sxa?tSmzYBKIhnDbj}aLX#W0lPwjKhN>&%id^SHOHS={2cwU6pJF_JIQbPRz>)|$O=9PmCwZPwff2djN#6t(_Bs$D6`3d*?i{B`H24=ohwq0@0)%9BB4TxtgkHu^0HbQpwxBxRk8a=^gEL5|?$mou%w4rXDo>oUYottBZLvqy@8k~EpDtA0N2v8)W+4U^jG>q;@d4voFAf<!o_fBwVSM7N4*r3gxn4GY-j9|Y=K_8#Ru!q<UY*TEmA08d=Dv<g6^osdo(~%#}3>@*zf@k&fWIH%a?vuK(J}&xGxb~7tE`s*{v-&gN(LzZRHr>sms|d8Rx3npD4_>dv=f-7GP03#F-sj!ZGfE)3F!<d7%F5^-3qVw5nQ#04m!H$5<no!5)mi6q@wN{f)7Zl`9<PIa!&wuXnOtn3bKKhnLbZvp@X%9kbd9PQ>*eG!u2%8{2Yc{hJoE^0z}DURjU-Ub1j|Q=BuYQ)1)*z0(FP)wKv_uJC(04rls1;VTJNLkaIXqH%HFV(?=bJQSs4YjGA7Nrwq7Nyi^9;RDEC}VJGI~k-{ygq9sc}w4>|UQ$NJ@Tw(nYTIir8t)4=GZkgRsZBPgkQ`PNqcnNsam$@x;K;_vmgPTU^nCv#i=Xyccj&$sLci3Na7+z}nZCZSw(OFu1m^<nNC?$&?m@N#t3^`XFN90BHaeF?&>JzQU5t25Z`$BP#`zWPd6t)6PbvVZ(Ja@Oz`yM3^_ze?*D;tPdhhEE!<i>+Z(udL>gEk>3e?pn717ipTg8TY3);4Pxn@0k*TFq9-8>c#=SQlnzHY%POmR4PkVzjeyVa@+IM-p>dwMs1M}E{x70q}gA$K@_w*iB^D+_)zg>OrYuwI8e0<+>vV5E{x94i#44X>qlkG1G5+U_WK6#J`rPEhT^`g6nTSx;_TaGO|Msi*?oEscfsSZO0H-_C8@~(?>5hsuDi+XY~#%OVY~O<<z2i!kgwgX5~|zHcy3&8vx(@YkGr!bpEGhU$Ov_;kz3hXM`zU@s&n$E{gDRsv-D5J-mKnkd+eygz(6|m5h@C7a)LU!4OP--=ZbGh8>UK{lJ>vJlnFNcH@vz(z=eF_9-E7^NbExNGMXrdRRv6kkMgJ;GQ!T!0<o8Oxl)s7HH4lWn^Iku7pB(l+g5fTe%v?+g;uWUqCgkqSa`rS`#IUw+=-!}Fhf1F?8-ALPMdJ90)l)bH?2upcN#|)xA(QNiJm3+m$L;4pr5^`o+gR9IAru_a{ayD{c}N#2($U{OHd``9?)y?9WRC}KX&clE0yUXcl+Ay3ZmKg2FSAN2EnuyJloH@UTo&y?T`YbVM0R?2J-NIFld`+E=k9ye!9m;<g8I|1m`RGHF(T;1X>+oGPd@tY%K1xk|yud)XAn$Q><m1{@-TDtaOPbK@u0d@rO<B=dn%ks;>vD&mP^pD%>GC9Ir5-XV;vw&P9|JO@ki$$53g0CmcP60`(QKt!y9{-F`9NPfz3UNLa7E(ZsD=c^toA3|1{yV~~v|_t{wN-~UW<(W)##jKY?KMXh4q2J)u*x^Vd&pn&6uxH22+mm>Gi@JJpMWj4f)`+1pPKg+$64B+$4f{aEeeGG0Z-}B0)*~Y)Yj_TM+=cW#C{MP;9*EqY<)+z}bI$5nCUcA2KYYVN1T}w|;65#&~<>O(ePo?h~MrEZ;=&p43Q)<O8oS%uI4Q|K9igx6`!|{FsO%DhKuwox+Jg%=lwk5?~XY}3DJs!}=S&3g~#;JSWv=gi=+|{a(wJxb}gsxgJm_XO+g86lnU0<GCr)aj`WAx{amz2|f-nV~QLD180Z$2BSx8-&&bS@!Nw0kTg1^6AcJ@nmHYX-dhE<;8c85JrbZ?n#6b(s%wsN;593wg`J)9xG8EM_@RDx5=W_<4JIsG3*y%~yJ_Ma#bS8)|YK-|X+L?P-{@bRO>&YWRbbX`Y6|yzXgT;B6UTLOT13kx>)wc2rhC14+eUmkRsTJv-kI#bw(YunkW?Uhkd?YntWEFi7g0%}L=(w_Ku|(#ma5ugq8mwaI{~dY81O|AL*n01pQ*%d)ycwf-W|KH*RPE34c-GvONtD%Nbwz(=R4POF10O1XjIGxgeg`M6}@<mPL+*CqTE>zpV*=s-Pl4PHa<*ZT<)xh2kp{Y*TgRi(R!Ro0TM`v7&yih0~Yu)k)ii<%@n2(`VS`Sty1We4h^`wM9%U)IZMkR<uL?|I=6p6=$L5f|5aoBh{4KRoX9Y33cI?t0LqCUMiqsdqYo*sW@K+W0l}{rUZO0;r&MiE6;U&B9yy`2H=Xr+Mw*7U#ZWxPJoyHg*K|sKBtAKo6xMPQcD1Y#wLGHo8+zdPb8~6CHND@4@0xKV6nAomiVB?BAwB_zX^q^W#^QUgV6pt!13Ojm)vA9#sm!txCVOb>rH=r+UYB76Gl&vJdYQ-7DWf)g`Vl^#h&I=6Bi%^0vAFkDTvt6Su-~0X@B$6U=Sz8hfWn_Cr#5(ND?0r6bIM#dHgh?mvccc#WyQ2ZA=Et6r@;>c4L%zr|>(0d4gWe8JAmfx}de(&CC<MX8PBi1XO(;`U$6Yx<i`-Mj9_u|&~rkb$SXR1Jsw1G*kf?p^Mj8WCJ=o=;X<slo=);cM05BV`Uek&4`GM!(GEjb+DDgs(2{e)e5K>%XDy5;hbb89&IlSd3V9XawEaDXE{V_4usijRVhwF{XgXhlFBo4?q~db-M6nl$7+M4e0Am|9A<vhoDxyT;4@{{6Kk!G^E}Ap07bJ_)916`VrqA!yw6L{8*!Txp6=W_4w+C$AbDS+=`(gqgoubW?z2=b5@w`1{#}<w)=a@y9niWhR41hc7FIVCbd<(AH4n~A*88TdOa<+&x;6Y_|shv1#n5eD5B5|i&gEafJTNdW*zM!53jq$<3=nOZm~P;zqPL;HR$AKBulBM)OF|6>4nNBz;a*x9Ba&B(;I}O>sMZ3nYoKW5oV|ye(Th286GjV)^Xc|+ql3q$vH{m8hB1;4#xLV1(x7WoV0>z2ppycwO@z_PfS*`Wc@<kL&NY}uyngeBdBZRhCM0y4Z3)fyD9eORnS#LPkA?j!3H@5r}LqC`TE$6ucwBOUI0|O#4@q11(}Z=NrBKiG;6fZ|9+c%{XYLp@oLfa%PwiHl{8)okGB-<o&9*vb<9Pskpt2Ol$S~G<ByJqm=UXT{&m#6$y;H=X>$E4qF?!C>d|g@s1jSr=%crW11#krcae9s_1^hTjpZu3(y*H-v{=X%1K3xRM5`L;V^vVu)KZ&v0+Txde^E>zeV5SF*&ow2>oi}%lboANqiyrI)}QxlFr3dqe*Kc873p|3)b^gI7tMT^Kv>4w{)~dwrBIVi>tVkqW_4N1vWR-MT{e`1IsTiV_HC0N`PovRt+YzFlA)EAKUweeMQGaLNv@-W=%LJY8-QA49@`;F^>6Z|mTG=5rfOBL!F<r}YS^AJO3gbSik&^;W*W8J4RAF!8)ozHUe&8n><DQydOY{Cvy&1TRUSmv_-I`kWmMTuPL(W}O}2f2d~j=Wceyt1K`!Dv?~MkNg_J7lnHvMt*GNOVO~ZXFSF?p<H-j%7Im=t&L(tvJ#=T~HfJolu)ptu^3|gB^YRt-~2iDpv?EbFBhq}`TK&z>SSqg5I`=hGG$7p>pW}6W;+KzdPYFIbGoy}UmI<80^Bx*iY>c*t{LilY*Cce1FKe^<NR1S9<%%nm#u@OS7G$w0aoAE|^^U1Cw9kv3w9m~rDedrw)$*+gt?geLs+i3W)kIr3NifosylI<EEp&0tN<lgj{C1HpjHvm%aQtIz1EXVD`Cb2i8OLW_>#qoA#v@5IlwFYM6&GkA57k79u+M(r8k)5VfGqYIhh{m<8eZtvFCjU~fcAMX^No!pc{JK;2GVu0|ioqq`0A5&Io>(Z};Xs^nh&7(!t4*N7mjET}zvjWB<yp4_LA?U|9Ovh%(}ZewKBWZtiFo#wFuI1&GTOc1s|B&N;bQZdL?cyy9;-mpuD7ZD<9iBl(2wQ_Z{1qoUaeO7mFw7g0j-V`aOond&FZV`=rLR;%!Mnt$79^A`|sIWISs|auTJRGgU+s^5mWsbS?Pi4qFjB*%<oL^-dJg2<!4OMW~R0gD}uwVwiT0Q2<q%v6Z+t5R#`HslZfV5nYZBzs60O{=KPEvH!D0j<AL#OwjSCWmk(&qGv0Ow=DPekAf8t3(Lql*NC7}A$Je7n#j;Z96Wv4EnJ)0|3`&(Exx45qRFe`#+fKx*t!6f0KjLL~?J^XbrdE#~h==CjD~x($wHnIXwpG)Gs!u4pw)Fvr_Wj*%qsBA<9~Y!?vHhHko$|a|b6xE&?o3u1whFd!ddIce{4~7p*49-!HhrpZD_O69GPaLT4^kDtk^86Tcm254U;V}kz9{udg#brvmpN@tyNP&w5dQmBvoB41+&KSM_)n3?rjTA{{aK!zbsHkKL1-K$bRVEHhi!hJT#rC%FgzV!f?hE8<QYsZXQEnrmD3QJU|8{DMRLk~g%3UD7iBCT-0gC5ory<FHPT^tls6S_!;M@)GcOfnM%9yI|FVbR*K^m<@JV<t#I4;2-G1`6x8$tabsA-uNQ)Q&Iv7HK4)dh3pu0b!w^+80@D>u^?Vk7D@A!ds`J)Q--fhk*Ulk&<(~|+1hdXr{a7?&0yH?NE*;anED<&N3@@&|Ou_{w4VYRCVjS1(j`6Qs{hE2hWqFULOrhM=#Z8o0z6H#Y_F&%hOM$pzNqz5V3olV@nOGBb<-p1v3U`fn<F&bbMc#sS>ZhQ(Kvmyni`^rfqVbHjThvsJ1L`zbvtAAGlpu-9C1tjl&^qCPm-?XZuTWV_cC&M}mwDw~_F%RtMcdu-<$6?!I!wQV-K<S3;qkiL!j7C6z>LpY<;(l!xp7n`~_T=Eprq_zV4u8D?bGk_UI~f~9N)q9nTkbZz@Cqmilh4fJ0@N;(<)0mDoFo%>zwZAIi^;lf&gPZp_X;UgL3T8#aVvl5!)oH2XtT@aXExpYb`2_8S^`wJ*QRP64+PBj2|8^lop3zb5K?iede4;uuqPhZA0gFJPd8+CQ;vggfycyj3xS>k1kLettwjE|{jBguwyL(v^>Vx!ww&F>TodC2U9#~LtYLZD3ZPhzF7K!O2l<crPUuerpPf3s)L8T9UT-!(KGhX-D5{U4a7~kI?S5Opl^W0k*)n|w8<ufMtJTmI=rS6NtIJ<<8>pcL1u*~PmaW^-9ic8Kf%$8I_dl6!<fojOrx4K%YiqqzotVz|IlCIy;<H>y&5vn5d6Q9s@%G*Enf9*v(N-o_J4u?rRNvmg^)m&S6>C8B%##JDK^b=ox2Vv)^QXL4`c}hjQZ06w;|p2XM*LZ8??7tApLEn3kBIIPv31FV8jES|)FRB(#9!KR^9pZ`N^%ga`gIO`+-iWhAAd#kVRAmGPMWXVd`vtZKaf&gZRZEE(7DgnLZa&>XOau5aU@UG1=Uk1;V+K>(H8!V#;k?L#oE;B4(L2LA!=2>5Bbp70U|7dUbQZFyY*(NrOdJ%u8v5vie1m2XtrPMTO91g#h|lVTPqH4wV-*^(edW`weQ{F=x?lhZ*ZFrm}C;G>%#}oN`0l@ki5P`C(9x7r!$qdwvieK)tKJ5*tp&}U$4q#x{IE{Y}@_^H31mWP5xZ`^7Fl%T)VZ~U{;_DXO_VEG-Q*Vd^zdo>CH9{W+k0I0`xs?F^dm}O?zI5>FP`EnRBR5OvClY+-GvNhs01eHnOr-1vV>kuqVtgx-K-xS#PG?WvhA0V@u@f7D^_rKNOlZXoW3)OcU?ZpUO=RSdwYaUe)pFmB}^r5^Y!Ba@@dlkJ)lvrv?pCkP4o$GXmeQ3JM8Wb!qQd1s8%PaMkWCvZR^Obv|Fqg&3=wzK(|J>K%>tM*LN34P!HYHeOM`oz?4A1TF*a!);tKsSCC2_54_;OEWF{aq;-X;A~VqA<rWjQT}%riAro@-R4SO*3LI&07H#&R4d<6ZPmW_s+r%OnUw=Ls(w+B=9{F^R%%1LDV(+Y*K>NMl&`+a>Un_nFmTcDj^op?e(r28ueow*b&jxN!sAI45PQ*K8w_!}z~!B|UqKrhao70y1EkxecN=hr2?w<E<z*0CkCl{r9x<ry$fc%OtLk8+?)~f{Yb*ZndxwSk)OXL@NDdlNfX|BFCToL6B;ULp7D1+w+47>U3$sppvG{C4fja$6>xuLjg}fY7bOcF5aaaHDh06v%yxj41b6Nhb&i&lbISK&{k-&oGph()vABzU@^Bfq+W8_=qk{&F!JHD4~0BrBBn}gQt-h93WS#Wt@mn(hxePe^~o^Hdor!UqSHqZ5VHVDL^GwY^NyCRU|46(yc{fcL(Hd3f(PNnSXTb`2fQxKc8%t6q>1RvaGG4~yGuQkauCs1n&Eou-6tsy{@!KvJT;woC5Mw$ZjMk#ZB6}Z#n`zr!9r*+GG{XjJC>J1-Ibq$)8@ItRLCkc|?sovM!Z9mj<4C*(kb$`pz`F=5!z-Sy7y~b#4wH_P6p|kT2*Qzbe^UVrcd`iS=DctE6RdJ5(8sTXBBn)&3e-pZOAxCFpZW_DXU)!DDDV)}f@w3u}9@@DHj3)kPjxV&W%yH=L)Jv&E+Qt~Ufo>@Ri{JB1l=s61O!8At#3$`3W1QH$GRiY}Gsn*2*7`VG7@Y&Vy=&??hI30t=7-_$gB<Hcg<Q4Hwfx;bQ}JVqlN56H>Uv__TIx`o<rpi@u8C0C?)MLGP+&XAsz}9R@y2iAKnlaEICQ?@UKhP=lk3q2hQ{*#{darseE6p!Gq3x&qr)`x9kQ#|>wzV*#^x3xjw1G)YImI7;dI&5_K4tcx7lT`>2(DXRb=o-Z6j*8VZ+fi|Hkrt(6cn_vq(nO+Q{^y*P@EHoJ|U{-b-gk=3febY<{FaFj;?4bC${@LYqB^!*Hn<l-9!)<KG`@_SB0@RO}7OCgIekiFqG9hV=WYq^$o&LyWELMGO1}W&Wn0kMr8p52v2yID_$o>h-MmDB5z8hS-t|V$nPrghlLgB>RBI;&nJWOk}3O(KGMsmx&1Boike|d(;kx{p)K1-})k&Zgnu(H@@%X;j_wn<gI`3+82Sjw!TFt|Jzh*XEUYhKfzq(&DHrH73J#pCQ1GGoTJ>O-3yyR2M0!tc=_ybIkfAN*)a59V6?6P`K!~QjNpm4MV&s_Sl?XQVk@RN`?J&V74?TRlMV_cT6H*tYCwIz*oMQY-~Y&aZ&qRO{G}^SKdjZVG~CJI3i@^P5vnq6D8(CpNM!JOt$R0$4mxWhg2cFBZO3HpQ@{VkiazsaPE!7tL{?d?)pedt9atUG-?SQC?>;&mc(aV`t;BKnV1M=bs)G>9fXDaT*1$@@dpCa%(<(h~m}0$NB^duUUWE<yTj#XA2X1B+mD2%6s&4)_C+$)F+aF*vV|y1!hFW!9)hVlsJZr0tGd0$Ddp?aUEn$Jot~Q4{D;XdFtTd#1nF1hq#7N{bev!xeV|)|mdR*%2=AllqWRX_hFQIMEkW{)_V`_>j7ppCx<U#U3>a+|8tOJ_;laoBh?#W}b#!kNr8iRHL_7d(pg?n!=6g%3?-$o#$7$F$3b&l>|gdj3Y(4t2glXQE3Cx)N?_LWY2XnWYH7Py-5%)sQ=#J%i^v_{gO!lty!CM8<l_H=}+&hl_&I_{)bDbwL%Y!Jb23wErud9BzYPSx(U=BL){@AW^g4qwT6)9}>i%)Z@Ss|TFr-S`Q4?kECl$4m9_kXP-}Y*psZi16FuDfFUOtZ*!5knWB|Cei!eczf@oTG#4Sck@DbGjUfgIUeBlDL)XWnv;9^=1W&;T<9DxkDU_Dq0^*(ZCrwph%m*|#(M%v(NiK@;W2huKEu(du|7yx2hb<EG{4tmC;v+QrjGi9M}P5MJy7sj9QHQOU6J#wbQjM|zIT>K^bTeFt6*J+mw}M}ZE{_U+<x>-d+<0v+^>PY_!Ylp*T^Qt6qkKOtZq6(wvMXL#wS#;beh)BkKcoZ9k$Xcq~oY#((9TQ1q!oTVx#AWt0K@obA{b--|o0$0ds{4+Z62g7ToRo*aGCi2d>{M{UWUAkFLvB&qHrOlyvK)D$ILEp1-q6sla(Lc-3F88?jtQS_K@J#A99G(^3QJ9~s!11<f*v7q-^uG>?D$)GTwR?2(e#+qY7CxAaC?AO~+(VWd2A#Sz&#F#9V6%#cOz(7CH<;Ql}LF*^hD|GSP^a}K7{0?qw&_Z!u1h2{`suR;yW7_ZC&f_>}QVF4a0_vw*q6B4yPZ60^R{hKRL9GO}rjFk)2-5vO#GUT1@@<1jncr{rr7c1gcw@X#ec59PZ^3%;3?2Lh0V+<zUPvgKN_VmCfjnhZ{onFZFa;5?7a_Fsd$3oWKJm=t3;$GL<iyuzHhm`sjK7hP=)s1%zb=!Tfsday%K8v&Qw&(Er{&`Ifd}K?dO70KJ^W$f9icRE?HuciF8a5*8yMbm8;Y9{Mm%nNw<WNP|M!6O0<K)=~{oA`2WHUN0=1)cYsHi2tRqMubv(;hn{MPs84H>%0^8ON9QT+LUS7R5Oo#lQ%kp1}&TzAPit(_+CrYE9i!FAyI%(JXR<?ZPy@QNQDbwyBelc4wl0Vb6>NIWcXi?}@{n7Hn&Q(owyoo82axX!O3ez{*p#JGBVjBa7~0oToOb1#JJsr#95ullk)i30=tr0K3)OVjPSo=tnGIY{4E|056;vkYs8WC}>mE&Ep)C$IxJoSmAH&z&)7wvns9a>UM7_vTCZT7UX?K9efc+}`rKv;`ZY?Ol84@=Z~PlDI7ItIqGdeJY<|d!~cxb6x-N=iu2;#ybp<lX`R>zs~E=Xp<S_H@Md3k3s!#o^3>NH{B1Xr`0w(tp1g8GLL|Oq1m_1j<|hNQ8MqsH340{`FEuPTGeM`@evojlZLvJTBcJ^3czZW<nKD=_WnX|+EDI|hKC1zb-QoL9d}D;mv)8!DPiseb!d*+25OHRM`qXoe{@Qkb$`4M`_yO`PCH!v0(sxk?|y#&r$5d+|0!X**8jbPd5*@cNy=3|TKwni(D3@-XNUGo|B3Bh+uo>b0<hkYaD?UUQPi*hFvwQyt10;o_v+mi$1Q&)(z);Hz^NWb8*iW_BfT;1<X`l8;nhaeCLX^A5=M-N-?XRUva-Rp_GSTqgHh{1JqRJZb-xN4YIUfyb-CP{Sfi`|%I8()zWC&S4k|sTbv4LW#7?sRb9Sg){daa~d)4z@?Z5_mns(o_@~HR^<?T!QcS!?G7dfePNX6jt*YhirMf<%t%x*CL8cJww&%P)`(2A>8kn#9>_jdV6aiEf`*0_1k<HC#HSi;F{DD1=uKUgh`6a+Jkn$@j!^FFD%lAmI)2`n!Hkj{p+7JD4Mz_Aw`7>e61=B;uly|a@w_uAPyB%2greC)>U@179Qc?_1j6uC~L;q*K84^y$PGwyuGd$W+j=40q^BB*MOj{NWf0Mizo2;IBVX^&8O*H!7Y$}5-o+UsT!d3}x0j`1?7H9o1<zcav`DAwUGjitnWXjd=kh0MJbIf*nhpapIE$7AnJrgUe2OGq7hra`>psJ5G@TdU~I-E?)C-ki%2RmkG7c#JBo1%wdermUIWE(P+~#lInI_<niuRep086qBiA;>4&a<5pg6eTXf27HYHO&#cd;jZSw7lAXT#@NUcc1kZ4{4^;D73~}~*IIo^6CE8(z?&m&*PC|>W$ojqBtP3te-I;Zh<Ma&+xezJ(M7}{^gH{{j=Io<BX!(_~=4`U(cUXO-2l{>Ka0^X#$M?Yn{*^7|^^ym8ezMC`(BD<}1IqWeEmA0J%XUZ3LhL>I?$?z`#;-QHP0<blZ`m_ZuCwb%Yk3eT>8;)&jaTUP!naQgkiOKFU8`(#82zdj{S5SK#LD@$>uJdBOJ{BbcekD7Q}Vo-5shj6Qy=Viy~;$I0_hmq*Qr*iTm-|xPsM_6k?Kt2nf;=7pHGD?u2B_`g$i}3SNh0TbYHMqyjOnBVdm*l%TMgEDfwGK7wO5#U)?v`L05@Y-DKV=vR@v-)tDOs3wya&Eic35#qf7)zT||N+{G;8qK4`Fr%rnaeoxY9TIm6k_iUMg7kLKW<1UMR1Rt7>ZoYh-4(z{VvAX4qSJt)LS;}>J$^RCwTS*J;{#blB$z`vjW1~>Vm7`&<E{ZFmGf230xS1HSt#lkz3hU1YUe`p)5337iBnx5BG97&R9-3ZfM%aGmFP+`@i{k#Cg|1-2jLHnSF|CeUC1)+q(d0ItHhCD6axR%0xJa#e4%-}+u#)QFYoZ#}duUaZ_iOHV_0xXRV65?F)q5r*DTjr<eF5Gz3*g}K+GVwMDAnY~C$`VU`ou<*((CV@?qSWm>Gp%hGy43@5sOg!<2sZa?XNo6vn_aCg#gvuZpoDFcX!mJwdk*tsc{hM=j%cdWhA<Cm8^+(KJ+54CDYI|$jzp6?NqMDo_@%kE1-Q^V)7<qrb3)0B)C}T88;xr%G(?s0%4E%jqSMghj2X&w&Ng-a->%+j~y)lpjlVDtksoQFTp{bP2jdFw`?Tyl--_`uQYtxGnaILOBZ(S?5?Ysjm<D3DW-p~bVxDK?xl8ar|Onw59Q<iAlXGXH`Zsn60f63OdcFptn_JZtq!+A<E5?J>jgtU`;>m<i){58bLl$${!+v$J_qXqbg<vJL8ROCdce_i)GqHE_H$az&hn%7TRfYeg<L@!y^Ds(;Tt_k40L9!`~ppm%)D;W6NL)IgWCJ=Cg!aAcGD)v_Q3yXVh*nQHUD3mnBzX@an|LX32*O@_!ixFP{8ayBZ7}#U!IwTKpo4QTk88Qw`++vTKf7!i7DxcCUrJE1U0F>$5sdZ2$)wqzJSe*5dHyUc**mcJvbtmn0$qywYH$tq&M=w7QJ_IQEu*c-`UJMgNfmI+Kk|8?Iv0k%@Q7&w;MUYders47)2d(2j#o(!^fDSI=y^1xbpel3}riZHKf7dJ&L<6rS-JY8fC(j+x@vC<P2#J`ra3Z&!;#4$Xg*)ph>)zH$u~!&4lgqZ&RZ{kCC$9nN_plq4}Pptm7)VLWqZ5<$ahA3}{MKUFy-Gg4R~=lap?<`8zfRYxj6=yZ!!0_bWdHUDbiHI>U0x>3d!7QOBGodOUtUI}-TzceN&(s8F@IU#2H|C*`BzxRvsRpiY04*Mq2D>}#$|C%wOmBB*_B%o?0fPO#cTIcEVejOXC=(HnNJK}WC`!&exd-}CqIQxZFFzZ@dS_D}umZLmUh_Sm^P>uKNrG%=rHVENMfj-ahqQc~g9xwr7e|K7xufVy7@MEPV^S8QA<`H0^w+KZ~o47De0##an1Uy#t~7V?#dk3d)r=fji?o>hD?`F)I<dSKX@II~x>Cq4C`dia3gbP3(M+w%~g>eDX9C+E6|?i*~avzrW>O}9bzmgvzFoB?v?n$49xS7xhX5lhFmq*Vv=W_xe7tcR&4<z~5VC-!P&*9QtI8{8IIZEM7xrh4h7?UkPA34GkZ6it{nSy1Ybe2Y5beB&tII7#$}SM}zAr&*hi`sXLU*=wC8fjCaZ*%Q&K`8~C#o-%s>O<|sngbH>ZzTV?Wc{!>DCulpYYX&16?T=3KEWsGD>*A#ReBP3G#vYE~OSxDrZ{=1(*P!u4foZZZLG0v-Vv?`l>#Z`mYz`Ok^1PkQxihzL^K%`p+Ub-A7F+fellvE5@|_FaCFz)IPN(Z)sb&kUuJ(^!-=r5jvEY)>S>EO$4MJ<YWGWsakA&;~S-t}Ep{*-JR7QK@s2%h!>f}rpe3iJo>Ar;i=_}u78cla=sgnOH_4ft{-ox=v%X)?bIdG=6zRin6B8B)_>=ZSjxXJi+v3;ei#j9t3pA2k+8PoXRb8(%6Q5=(+1*bw+`})zHE(W7Tb=Ven+S*5VRLhYtqJNpbS5sd<!jIwfg6q4@I%RZpA}gEc*bl9s5`XV$A2t=kp-8#I3!N_5ANC&CIl0!C9Pt}tvVG?_vH$Yw^JegPTmyc7O=;Hh5<}h$la*3g(DxB=T|D$6nH&QrdeAzUiRBwb?}|(<uJdYu+fUFbMexkg*aVA;x6P~Dg@*#)!4WsS*StX`V)b+S+Xcuh=vBJ<vA&g058-&5wf5;-kMK<6_7tMhH@P=}<wi-vyS(koE03RxHHc;BH8F*8mr+<})kqZj@f($YFYPFlFL0s3kiVpbZY!IN)Y++Z+T74K7bV()7)Q~ui*Mg=-GqmYN%uFb?ZSTJkpxEhn#!k`m&n>vmc{{kzoM>V1<xXcq<gF2Umdc7?X*>FU&NbKV_HS<J{04g)ll>1y^3s?%GBuB#tkw}>nxW?+*SB&(^Xr%f^;-vCDPWccNwbEtucOHEX)kVMIH<;&jW8OdfB*vg57oh^0!sj%zqyxa`pL7SpEBEc3Ro9cBN_jCVXX1=A(D^d$dJ&ipe|Tx#~?gGqewfHoZ*l`J@8}%rKVIvo2Ia?LlE`c$2WBFc~|nUR*bwS4zoOHF~>;v`HNF@Azh#6=m5P-m^!J*YWDE2o(F+f3clG|5@_%+LD*qZY<u?>?HkQ9ZbAec(ew0=?gmGu~pfg*Sk)C@%=u~3G_5&Z~-pQzI40N>pb#S%M}c%O1CG(m+Emi9go+1)ve3D)Rg)u*Rv2{WQEd8+UyZ_@-MiG*X1`&#EaA!+pNKF{$e&RdVkSlW;fCNGZ5!dzsVfnvEF#jp_jOBd)d0<@s<IdAT4qlhc4cTi)eY&@|#oeK=<0O6HJPqc38G{@JXz91cXhmClw(F#o?7L_A+|`yXc>}dqIozYyQC~>pqdhjpIuk-0UGk*NV}2@O6oNiRlpvkgxL?Shn+MKSUEJ=p?G%Oo~H~xvs?dL#T?>EsA^jq=7uf7IK$9t>BQ3D;nsxZyHWO!L8t%?)WoTEPQfJU9hu=(^1Xu4N3%>^ZP#>3%j?dc5qG=ll{C&z#J`8z78AopI`|@FFAj9m)v~fIlQ!OW0LKQ{oOTR9Tg&O_2t@QA%%YHVFi34B0hvx-GKRYK8~;iuLt|>AJzrCd)-#lNP%qUdiRl@i5<EZLXSS(+QOXBNVC6p>D3w6)qZ2s6|(G6bm^>;i?t8WS8}WJ3iiF@hCVRRR6Nz0q}mu-H(;#-Mk*ta*7$6SC2^a)-ZV9ec-26V%IHG<CcB~tP>a74b7p(K@HNms469J4RW7Ru#~g+bwlekgpw9bQN=GeWdMr^s_+b5oP10rA?`NM-?PS{*UfknX2-;hx%04`8$r<#V)8JR6NtcP#*|Hx34g=>26wMIfDse&ht-H2;MyKPwwYfyyL<_<_?VVoBn;TH>rVe?_KPTHQ-o+N#ULq!cZm`dzD$aM-Vlz8Ugq3LvLPOos;Y_Oa=IRcRqF3YeU^e$w(;UCP(e&NHVzTKC$liwhft}ro<>Ra|+`FqVa_mb_O57vM)N0A)x1aRpe5jxsZg16$&s3Uk%$wO$&KJNtdR+rs*Sp%U-RuLaqtV3vRH!lz)!ZW*yXo>8=>?Pa>q(MedZnG*w}8Fbx4#gxD5cm4;`!9YjM}S%Kb~`Cc{~FPG}Eh^3}bX+xH0hZk9>{&c@0Mx=EsHwV)m3R{}c^-F*Uy1R6WhSjjS))tpW?bN??p?9yFud@hZC7-8EkJb%D<v4L_U41`6+&-xM*=lUBu|m^@%lJCf}fhpkl$xJjzRHKhq*!J9%d7@YQ6NsbO!>D{G={%koPnhp2RX4{KaOZ()ba8uVCsl^};2``mi^s4_(+eX&~`agO6f&*;Algj((5Bo?9Le4j!(boY?ACujeoFbhW0=w34pqpeKNm{ao<c}3vtkHV?WuAF=gq~~BF&-{AKw6tN?+@z3AUTJmlTMh378asQ@_FPCROk<(D_)=3Rz5Z4j-WoLNl@(9Dv@^M!*G0LDY9@uKW^X!)!wyCZWb^-vr9ikOdjnQ*z9IVkq%qR(R3JldvEww=5h~pY*WI#`r@Z29eX7P@c0y1&F>7fp}cBr1ajZbxp1}2=cbey$aa0Rb8%_^f@4*MZ_T8eU-H_E9B57b(aYPJa}SQkTBUVfYn`Xor+?jhhbk+>52l|UsARR+GMnONw<f<^&X-f@V+INXx_q4CL0woMf?bLf02cYkx%Ji+FbyU~8Q4O#OMxR$tnQZb7j9|r@GDo^hds~6Ni0?+dX$mra;ib=f}h3G>vQF-DZPJEzw-n9n*5Ea-`y^#A{L*nie|;0dO5SJc3(nC?=K#qUJRj?q4jgW<R;1AP0%m;e7&cCBDN|TZ53V6H+XYb>*%gADqC^E5iQwHlaVW)0I|v(8%ktB1o05K?X&&nH;Qj1vbC)ivhwXu1<q%PZ5oCzQ@_6C73Jt~8ezX%edda2{;(Pgk{&RTE7+hl)sd>Q2M^c*Jp7PJzvj|PKQ!rCtZNJyHH5H3I+uESN!Mo9QvEpT-~HkHm_V4@82IFoTy3Z5IfsZRJ}%CVjNeC)Ryj1P0({t!Px8gcU@XAF8~*i#?-baK;KMb-{uI^dQpTktp-~0jo~k99W_MfEXLwIIo=?(IDFzC2D`)(-GdzQf46N=qXDU9g7tkJ)`4-s)jWObG|Ew6ky}MH1!41uNZM<tSg?r7}Zon(gIeB$Q++AnyMvH!&+vR~1E14n5`H0@47<^DJ%)<2c*{t8-l3PkZ`fg4}k^s)&hEE2l&dkYy!=)i$&Tq4YT7Keo2YCK23zUsIPDOi&*Er19_WMc#3CKqjh<n_hy3X{Vy(X!qdu14&8JxG2we0METI{QiC$#E5n--t58*wcJU7wz`3wQe6w0pq6WhWpec0so{bzYt(Et+nv;aH3gBn#B~U1p2XS_H-B`MGtLakA@1sWYhR?zFZC+Rf@5HvXoL4mx%Hw<TJ~iY%02Z{8j~ep7NGg6=w9pXYW>py&MK#`XWgSBJ&QOTVndNod^L{NIuRT+k4@m{k@_8g4U%y4v8!JJv_Yx$Wa;oueBjfNC%*oJHl$pPXJX{!V>$kiTZHgRwy-qn?)oFI#wyFBx8|$%L4aQ88_?Vw2J@Qg1SaY9F<>mKY)&kG=WKPd!Xx_dU2-?V^jnKofd5w|B({`&Pi{%ywd^W6YettW|t%u;OwKt=WZ@!k6```AnwBAm)dELjGankzQr{Xw6x7#qU(q^n3)~M%~o3ntV7}z9kZ?+}roQw;cb$cV#1G?~5eWkKE7wwCUcx96hhyeB!HL)Ub!s*M+d^M_QCBM|Y&Hr*ZC3*dw;6%4dJYivSRz1=&%U-MThCldp)#(T7+*vN7IuxF9_r@7Ei=0Cy>YR>8;e^R7^B{_xGTPZ0Oa@7GuK$K=C6_&_b->0G-_=>BXA;Xr#<AF2m?PJ~!v>6L6x*N<^Sk=t>}QcJJ-*{)Wn@~Q*GB(79j&SBi1b)k`^g!4O}t}7Hb-wo8GNkBvGJPbQV>r`Uf-3#HrWRs1eVwtraJU6xVaI?Xpp7?q~_h#h=SG%3qx|h^;Si{uTRBN)#<XWBtsP)JqFPrugV7}hembrW0R+hQ;1dY>MU7wI&w=gF?b8NrfgS106V0g2V(?en@a#|r7Z)bYDuP~UhR%>BmZ1oRYdS5q(JJ=C@y{t#%s6%bFzI9Ujf{B2F$CI1!&6ulgLRE}@4aCjdk9>DnzL`YWq`N`uSH?&BmzI<WZ+*w8I|3Ks08H{IX5YW@qpJvk75KB$k(-IuaPGCE&USSA4Smu`kuFH*6-=)kFRcg;p!{+j)x_GB77>%KpDwnsoR1a*Wj8*gSZzFNuNU#guAXQ3O|ajVwQMXm45e9SW6cfRxWA}CgEw(EpTF$ufSEEL?Pp7!4+Dy*9gJ+BZm-fJiVyxRgI&^7vbbR(#SMP%<O#1g3a#t7n?`Ix<;@$5xP9~=r}|pt5NyUh!pI{pFV3YRD2X-;#e2*LB~@9Eni<Fp#P;XLx5Fn)BmC2UalPf=M(7Xb*Zs6O0MMuZl>Q_gU+gCQGQ)u9DM<pl1Iy>5wcH}@Qa`w8^~~4x&4v~`Yqp0!65!95=(u@?L}1UcfpK{ftzq}vqlL4LHXkFXAEf9U^G0`kst&sY6cFJcp=;eP*BG@7^_^`d)s1;KtQoJhpVR(AL5+xaP5O_ZT(;2DSw*>B01fVZL3O{Gy;MZ{d{t(dV09jElVgH(=^cVC<CWMS5ieV0TkqIkxy}K7Mum8iC-(V)*wW6LS;cF&CS+N`&S9!1{SVj_B=7m@;xsZJK#I5hj9T}6=EWl;WaGi`1h^YBQS#&Zx_WSLdxav0FEDg&Vy(qzKBR1jJtByIuTXGZt!g!*)_B$Y-a|(hf^dg8-L!rpyoU_Xo!)iV!8UJ@D8Hj|lqw{E!%*0KdVTfH<;%(jFW=WlZY1|T)W!$TjJa%v$6M&rGl~S$&7?Dr{AE}(HxV#0_TIGbDAiZp!fWhKo3FK^g?iqn&v?HmT3r8jk>{T}%VlGj_X^Kdp@Vf?Y<9i0UfO^*yDUR|@MsISz=!Jnw9bo*<>JtO)~dLCBY>`fTjWl2R!+ES_K$NOwsA@!Ua)$d<i;>%>nUzbc1kpnf+6Eg67J7|1Buh`i-l717iXW-f?w|kW2<tI-t)_=db`?FT8$rj2T-gVqQu8TF?tiEd{h3GusQ5zr3(QnfFs_j&5oQ|Al(*!!Q;Yw-@xs}XW}guKg7<$^Gk8ll{*3ByK#S9qR%#HRpzfj_0mW`_K|xYca`qV{+)83ZwtnJGJFDVeN@R-)I8hmkB{<U+451?&S(<`zL!As`(a&paEdX6OnF+ZoWtXg{V&qaY-d@h+tSx!1cMk7It7EMD2)gzV1syoqJV-Tf}()Uwb%RoRh8N&mF!d<dV;#vY@U%d8SA3=eUo4L%Ld)rNR`(ySy>mHlxDKg+?P8e@9ohtdNMAaUzU?%`SW|7`DDQ@E_HC56y}N=M&(#)GHI={g0^)RMhMMI)Yr0EmSgiW9Ti~&?JC0MzThTJ5z;L7M}D5kqi_4{zBLgSVubHsuAhbt@T!nOZ~hA_HTMvI(AjfHVT&Eej0~nWc4UpO+|YS*aB4dl77=egPzS@x#lKz#&(|YT=iG-fM(JSHQLjICiSFv>Y&E0AOK7Ie=iAiJHumoZw@wGC%&{i>J(=HnIr%&e?7X?vN}IH@zcj4&!wuPZPPz*b5{)W8=(4EyJgPh;d&df8CQl7PLrDGhZ2aIpb7PNS{Lp;j?g=54Lwz^b;}C}2*%&X9L;#+bDtc_|pZ)P?PU5t-YE{W|<+I?I{cEgesp}>VzH}D%+P3-*%&7pi^@>`<iy`3o`>;dZpC5Ol9y2X5(pCNeOZ4e}0NH!{%vdBTy)*mHcT?mgH&=g(Kzgn*VWs2}^}-UAs=2L6)SBJ8#aA6FQ*l3i&QSb_rNk?$hEup`)}(vxKeha7iT!)A&+%=wzp)3k^m>e)^;u~4ZGG7cnCDJ9qCEB8zbeqUf-3RvHPz+$BX6{|q>-02yY#=?UQfS@@g-i8_g~a!*WI5+H||T6KD`XPxA(hr<3iO3{Q=D0FMlF*&^E|FmZkP1U_3*9vhFN`^g4O1w+pN-H%#)gMO(qregB-U+>k1<32(O7-Ns2Lu>MzJpAnRI**ukwx6DTKR*Nyl@7H8(23;ESpWS0Bh(GYr88^}nbfX9So)xdP#Ws$*0Dd3r@jpIw7vq~=?al_Rd1YN`agbK5i4~bS*LL1pzFmLeKA76{GqhjZ>Xk~3CQCb>tv`n)?IXw2s;~Dsu7kY1ZRKFy8KPSILR1Y(C*y-z(68>MZ%^%vY%Jc{kh%%`M>GI9_$Q@JJhSzD1>f%v+OVT5@oSc;iENI-u&fuApd!Ayh<rn9VeNF(qestQg=S^E0=sw=%x3jnUhbz9-4nT&dkTAkppM-e;4Y0bxEt@PzIKQvb+mSbrpM>t8nnlEYdvlSlpj4GdG3A<dHG(!JLV}jUkVEQa%*V^B@yYD?(*Ug#;B|B7N72=9ti!`G3=!-Ry#jKf-#uH1K;?S+b=X|3rOYReh=yK*#7|Ga{MXMb<*bnaYXm=*KQlss{nHUT4$|Q>3WnM+|(=dz4khYK$qM5vD?`=!jpT=FX9|S!WKfKBDj-6dHw#Pce=Ns$Z>G1ZOjC}C5LLnRTjvWGKSauzCFTl16U3To;McP5g{AfU>E+7PUI4OE>f@tWP!G&K2oIts4Bu^XaE!k>$%yJcS++~fV%Mjwc~_pMIdyet1tc#?8+!uY2I=OH2+e-S}T>g0XiGGE5O}ixq5x(XM5g9MEWq;v}!kG8!@A!)G-7*&49*OiB{+>W7)l{rOth%HXK{)7+|yN)%bR|$sqo9JzZa*@q9*<&l}q&&BkN&K8>;%m=-nTQD4(l&}*QdJS^1K#YjfK$ZV*=<%Wk+l}5YIbyMFBV6}7KOdD~>xb~RJxSx9l9K|l*yY=#a3b`DY(Fo6b;G23A?9rk=5tFm%K_!PDC4WpD$a0)RpdFiLCf1Pl&FsD0g(Rte)H=CVQ><6RKjO!!U8+O8ZL_RQ*pD8&GsCaK*ajLLy3<2qJ)gWw4qaM&w0z2O$2`kAD`!&8@0-tgm;S~DA}n3R2VWBWzAs~zKnW!}Tc`2;u&xQgkV)L&vE_Cm|NGP8=^+@NzvBT)%#Q0GcrH%by%e=~I2tW*=kVgw=Ujy8{%x%k#c8|&n`K9*YfhD(yX)P)x%|c5j@1}Z*&+K|Yj;zl=+q*I)7C}UoXY;B&n~L!!uk`6M#J5VmLY5pbMZc^Bq}yM1zT)DRBfc<%=b;F%ce3P9j;?pbyt(Nu)^fDPTNzx>~rnrmDiN0W*=de{Sy}N_5m#DYy@7S(1YRJc&86E`s;!_XpW!X?qd%<*;Hq%k3MQot#t2p^$(!cT<U4PbO4>T5o2i(-3@e+(PPyJb+(lzy?pJZe-tM)foe72;XfTkWk725m|$6LZhlk3?VV(1q&2Wwb3w^q6nqvYuIB5hgMkw*%;NbiH5BboVGu+*eY0|~==Q|@;mtxCj)1#JgoA(@lrS*3UPntPs95WZV=LGy!M*8Y>aJVh1m}^%?$tP7mdJrQ`5Af4w!7bEiG2@cV<iq>+nu`Yk~!oO2q%%j{NPmm-i{Q1bC2B&rlXnXgHfSd1C>v<>sAZ4SMY3U@8@!aJ<_#5?!_B&aS*Me-kfxQv2vFOUkXTOV;a+#YIdH!bD{AkmXc|^4Y-?fwmm`+)+@352Bi=!VExf1nirenn5e!-6|SDQX0r)mZ1^YmxO6u3xLIL@h>VS|W5T6`j=q>(L#`JYgO28<+F9iqUe2>t<?<w)vKB@+Uwr~#u5dbY<dvgn)F*3ndRUCG)zix^x5d7fN+3uukvHZgRFhlP5M4R<v4=r`HC#A_yu-BJ=xqMC=0saV`u}52^c2C)Tdk(~<>CToCkvO=j3dgH_j;WjpWGG<nceRl0x5vXZ=bIDTejAQF?1km*8{e5+E{q&MJJ+3Hy?Aj%gCpw@mfEa_u(ofZ*_-!6e~R5-f3q>&Ur|^m6$zNZ(8<y^ar6_|4c667^xynW9!CGee)~oE!z--s)G7GXZNX6P+2sc6L=)lEW>v729mnpM=X{4M6oJ!Qo(8R;I=FEch9Y?wlSVdOq$nn%csW&_&taa1g?CmpPq9$%Y8By8o+c=j6eM7>ur}-dQz<FWA0OA_x9QbZ*+8-RNWW<Z`N<$xiiXmNJXB)jGMLcF@fHWv)aD8zhA%KsA7H>YY;USR~sks>QSHzG&Mb%8J5Bz@=074@Fg#>`w0uC4TWUSR_!mkj5~o`tJ%EHkyE}8T$b?8Zl8&N6QfgQ!JXJWZhD^g7)Wjd13C!gC#hX`b&+$}*P@R!tgoNQ+9m0ZE?=)g`{n&f2WN*hvp0|=vy)fo--YK1fd$2(@;;2l+!(ba0{#$Qb(KM|y{}(G;k>^FW*9w{{_J3Wbg6FszVD7gI*WP=99~}pgC_yG<Ii?w;;{FVgRwDl9l&<9YQU|5gEpHUZ_X=s_>9G~#yCCTby{GqWd9i3N96CJ1MH(U$X_<?4`p!Mi#Bcj4JTv-7fHcz<3%O^;8}2khUC=0HQjf<Sk=$x%W<XpiyL(I<x*u9J^%22Upa<#Q$~-D=`z}L*-)%UrFs3sP_gV}@Zd|v{qf_O92jsa%=F-R7@W8k_;{A4o$CV?2Cl@XYIrwJobRj6@V&?uOrH)g4^d{lC<Q%LD;BWQpniF`Fy4l#h6;Zhy1-RXB1`$&VtKFf0F4;q)!UI@;8Pq!Px+cJ70~rWNgWqMj+kxye9a3EAEwvQRyd>O7HxCXurjCH-KvBYT_HXktlH+Y$~VU_9T9(&jyxCA=2h(-=i|)0bqT!iZl~3!TU3kQNZy`9s4z-b09R5)#CwOk^GgndQr&JJqt3QPKflnuy&5g#bQr+<OVZLPlezT*Za;>)8|l=1ou;!PQ~`eAiSeinA0f~7LaUKsucP8fMLYjG&);?HGr#NNz4+eg>^E#bha@uhP6m{GPRY<l{Uxuj%b~99yk-xsh2|ib0L#{2GBSm^BTX6ccJu0k!jIoxh19f23XLb)q%N&8Pg)hJc4N4#GK~!DcCO%yR6J66vUzHYuJ!$5jhhogb2_J%xVXgIt+0If-<vC*bDYoK-Tl872cbkbd{^ZUaL(UB`_iER+(92|PQETXEL>litupFk<Sy$jr}st@FIKrjP#^&A?H&ub@N@?NEr+UEt5Fl~Y#J8!0L_Qim>>2{W?R&M`B<#91T%WSoEbZw@9c+k{*^hg{lh|Sck54?9ze9O&j(+5kKX;Vug|`_0s%*w)tWJfL!F)1tes3Q?EQ9U+}hk`=o7EuRhxF@N?Zrp*|=zZvbwOm0n{smI|EmWUqlIB#2l;ZanZSb@?rh+T&zayZ%t0A{%wIy*k)fT-m*3MQ2FMs)bN)ZBd|qBD<9nkd5got@jQo>NY0T3xl<1u8#nLha2!@2BRGvG%e-0@>EtAAe|TjQjks>UL4b7#&!T*O)JAOZb6ZbGOC{C>_gHrD(Vk>s5VQ(Vv8qVkujCB=dwf2UA%?V}R<|v5r4a8_AKSYb&fCM?*VwDT^Paf#)<0{CNzVQ?*@v$Smt8S&qQ3vC-}~z^UgEW5-n@0=mcLDx3cBsf)86?!R8D0VWd>%v_YcX*yTRoXT`HM1J;sk(*J(uM*m<2-+rr2Qus&?lt=@RUf!fnTi$$niWtIlTDr59!fqe8-XqhHg>9hcsr~bLUJZ~<xh3@4IC--TuT0ODOix;$Y9RR6UG8K>SgV8&v{cx(@o{ihit-EFY!B;nR-`2hL=^>&8o_m_yd|c@BaNnMm+wOGiJ*&(54UPV*cjwDnDQwp%Nw+<}darzhEtaUx{vYqo2^g$9|I@pZ?-t2bT_GzhE4J3cN7n2Acz3Rjf_pt(Fynq4QB%|W6Rx)x)FY1cYP^`l_Uj~<y~D0L@~2@5A2QMJ)Yf<ZUOO#GX8(tH=Rt3>(PP-EoC(*TdRf4IM8v9|wnP8?@TZlMT!}xlK{N-)1m)v|Yu7vDoihgE!eU)$a3D}sAeZxjyy6sy2)oSTp{gkhfpT>g>yg=|sjb}qizpKvtjjKe_Q52EE_O?Wd*i%CmyX~Ag>KC5n_{=Wv{0bPVFQKkINhifEx2@hor%{+dQ)&;?eDntD58RB^<TU@t@(=6qPd~}FYnIt9{-Pb=Qv-3*8RlYXW?y=Zf9rgJDpyy>rV3s-Fh{}!Kc)q=tR!4i&@}Mkk{*ay+%RFPRJ_XsJ2K5l-rWK8aJ)4x*B8DDp&zXR3u;T94k!nb2YoyDhLa4F%~WO9yP6c^$6X=)oZVb?XC5r{!Gc(47u84FPrP`u)U9J>1PsB1CZz*X3LU-E5lbPM?d6RFXBEhb;;Ucx+D|~y?peHH0who8xH@1`5^a)Zwjm|<T^9?kc5zYhGDPdU&XLb-!HY^o&x*ftX9%c*Lyj|t2e31*lVfPOz6T>8qzHfm+aM<jSHGJzqJpsxAE6<@xj)3a=i=qODbUFqf}iZeAw~jam13bg}fO^WH)nf2N{uRjdT#_hww^n@A263rXxLR{WYO#boTH0`Tl5-DM0yqZQp8UB-I}7ucbT$PoxjE$w$qgfRo}ClTm$l-Z7iWJQw`iJ8V=PW4)cF%|k^}1#LN1)z1o3jo@4Hrbc_wU$Zab5WnBx4T-SW>~0z3K0+yiWKQ>I>yFIf#Rx)fc%JRdy{BUs_w$}2Pb=K(Nu7)nzFw2I))+#gs8@xe`z}fb?HU?Yz43bCm>O+gg*@xr0J5krp2TIWnY;GydA>|3i@YlJ9>5uGkm$_i>;B-8_|)kqj}pFCHoM$JwRw`?#*&l!r$)WM9~2zVkNlUr7DHhZKf2Q$2EQ$=AJ~(1&uV>fz}v`e%Q;hcsl68;8{)c&=!R$OxhRMliYnl5NtoBeW1(}^`DDgg()6@V4W~i?XH+E}`uYr5OAu`--RSR#vlo-9<9er*JNC_{I-bGIunn~Lk__KP99~GDytXHeu~e$h$MkmnE%!rgs;&XUJg)A((N$llA1W3wK2Xl+$#<0FV)Um<R-iAAMo4$a%{k_sSyJb&9-CSMQjw{}weu#7&;3oif`c6y*?r>ow%5vR@mvdq+^#j9uj*_@yUWKG6!LVd&ymlg-xMt9Pe|9}-~pcNP5apB-@W_WzCN*`yubRa_;OVO;>D)6sa{Vb6*^X{rt96l@@YBg=<0VEORaaL?6&?qRCyAbnqqq^74<ZB1RmdGw6w_T?^-A|y523hZ>5h{hJ*8#8HcbdQTX?>k{09EwJy5DD*T)sz`@S#&zL`<ot#}lm#$i+{#WulOp4$9`2gvk%uDht&@Ti#)-Wv^G0)jRzI^o7<KrcI=!DsO_^;I-5|JuFzNsiZJ=Vy&zw#M``PGEk6nEq%FlKJfs}WCD*AhDG+3q|Srs3aLD##!{;PQU15o9e@-LH;0>)Pls;EC#A=cUqlU${9Vtxn0W=J3<7G1ZN_!-coF#^`$kmxPBRLh_|Zk6baEwBJ=yW!ML6EmNGY!}_uutIe|3tjG(FJ6J|B$;27U<~D9Iui%Gi3o>+a>N4@)SK4U#kHzR&=dO5PRvSNudP_nB6Fe$fr1g9ritg$<IYO=R(Ix@hXRWKPR`9-s^eS1{Fx#8p?~!8cCJ*n9_S>U%bBL&eI6RH>x^ce#ZF&1E%OM2#gXF6`V4_hzoriM?w_qjMK{C?VXHMIiMW}u;x%SK$TAdRu!-d-!HnE6-LbVl%w)5RUxDcTr{v<vlC^+nyt|UIe_cCCU8#YU;OlM`D=f81YX+pSk2J~jXzPMuFQH`1)iZ)#=@j`L#RpP86%tU_6mrHB9s(HcU9PZ$k+LgO)czoHJ8yq?0T(Yi*<i@r`@>9838{RK`@pH!|=5X}RnruYxMnj)25oy&iHl1mFtiEo?S#{!YFC!T#ZtK$x_i4Re-gkpPReo#UsMn=&9i^Tb?AP1@&-Lz9#w#Nkzn9i{IkVAm%W*2T1$b0wXDH8uJ<EnTk*T*@UH1a3*-I}&tJ>?X_?Ix%cu`uLyRNYPNLI_kdhG|G%fPSWy2+D2M-KK}wFOQPPnR2@4)m4M?&^Wlb!zXe-6riyKF17+IltT&9KoZe3!P|JO@XkoXRt)L`+lPh(D3R!Yji8sPSlG37J6anNe8v`cI3>3;2m7=rmUdz22r?PIUHD*%Q@6sfMY<%hJG#im8@w6earXPr2T+*Ub62;9@u6&AUVFRkzk7F;~0MMWv|YVFgTS$xQGXev!{>Y0=TFJQ}?<a)sJ`M!mG!sr?r~{gWPMYqzMcM=yR<yU3l@N8|Oox^X)fiz<+hDs4|3nua7adGg+Pu2Y@?!^ROk}XK%M$(elkHo#p&TexT>+zV$q`H@mD~j9>2<L77hcbO6j9xvTG3wCIDs$SH>WBGMO+%haJql{ucpeSC5QXZ?vcUthtI*+p5ytgg7{gc&gl4eqq`5B8oc(f)k1^S6ATCV_3E9^{jwNe1<(KRt7Aho@%iX)eC%tIT@n+A|dOZR&4kYFj#-11YPZqvdb`beZXWW;18RSc_f#!l8z$R`f=iA6nbbUAhL*kZ)ENYy9G7;H*TJ8;|54q%(`CMKd>9v~MMI`tTtt>w7jBz4>;3?RV(-Xs7dG)!xSyCR)R@vQpQz?h1y$)t}6MPFr5ZMX${iZTs8G<M`Km7p7C6Lf`qQHpHw3`-%moVtGgojk=gfNqRfHgzJI!qJMIFUTG_&++8Awjqb}r=JbNGMhh*E#_o!OPY^iTcOsq5HP&-C{#(7LoBrPNj@QrhM4y_x1`N7%;+;AtKp`gXqmrS>@ay3voo@vyFD)vbF)9K)&DR=?a!oh8YOydD0)h0x`L@^YyzA`CR5Dd?26t(1rSy?s4*YhTUAUF@%ukZGxv9Sbaof=BHt`uU7x>VZX7A~raun^ZLyi2Qem_~sqq+QcW+Xo1$=aw?)-!K5UJ7dtnghgVN#u2m+jQ8zXQ)m>;BjY-@aV5VPVp7_m86;_7rV)g%@OkZ9Cyb&iLhmGSUTro@z0Fmh1uz~zxx0i*||rR$4*~MR(*#(Tr3!oSWbSpPAOat?QpfCL+MRdKZo|Y^{f3=z>|8h*SqE<tXckh)Mh5D^;Vx7fr7M7Yy7#N{r&hr`u<i&9?btT*AS8D4xdP?f9a2%&F;Q!;=-s~Z~i;0aR;vtlky9tN(k^^9oQ0|4#|7?Zqzd4wlulVET(_wcmnJP_wQqcI@VLLI{3;6B)_jA&owmnq(8;;whvS2maqF4nz<7fJDtFczM<>3bLupIq14+x_K5ojT0)B+%rj?u?xE`SHw>_z+a?0w9KD+rb}Jn-L7|@rbw~gqNPz6OU&QWtcu#K{)*FG(NYV0LP^hz9VIAJogQU?R^z8DXGPW_e^X+S`yx863D^#JcTjwi2@`d~sZ|gT;NU<|eTAuiGG6sGvZ$~o5!eD{UcDGh1fMB@k6J^NuyfdE*N`8E9&DlAeDUbeZ;&;z|*cDr1P##y!KV4zPn$-95>0zcPVd6Kl$=;c@yRCq>$r#vk!H$!;#bd-rt!tR>igmZM4L5X6DNV8Znw~vboZVeCoMx1!R;EagpOuGaXET&u*0{<}7xY1F?W#9vq-gLu&pbF!lfPC8+}Iq;*$26{itUI@%Bk9<`ov2cO3S>i{JdgxRK|r}Cp)H~hiR27!6}Kh!BOpW^O~q<)j+AWh(nbUzuN_O=Iq_{HZ(?$JLu$R+&H=xuvB$`h^y}KGi)jcF&UmtndC1fT2xR&YjxZSV~ZU(kOw~6btbcO#6Q|OBmI^8-f*z%QTO8~{l(oY&pmFICe+zUek%Tw^e=_TpX#f-ngx_RF&N$w6pXI+ti=uH91Aww{tj=OK{ndfkIPLzFDg`{<1eD4T)2O=@(X8r4AtA462E9@Q?8}m&+=A@($-2{un|q0+&9NZT;s(%)0MvNd~<RCSv>7E3tmTq+Hrbl&gm&5yz<&(z7RK)y&Y2ZJM~<OiLC7woa=w;Gg=7^!&n?TvAU}LX#I3gjMk{7ZB;W84)s<WSnJbMrks!Qrg|*dsu$U}=+fgV2(>#!!Fm-gQ!1PsJ>o8x*78Rl_BrtG4leCzE?e}RJj!eHKH;Dx1TJx>1<p%yOa?N%OXX94P|-s;yj}QW+mEsBY?YrCEPb>)64HdDUY=s&N4J%Jyj83fLF->%s+`df8zmeVmf6}_rTFPJg4gv+q~2Fwbz!dAv0dkHMD_NHma5sx-QBE92I*2x`t-Uawu<nS{teP3ZE$n%r@>)%HMD~?H{G!6v)%jPyYtCfk#<h6H;!6n{d0Pshd$!>=pu~PWOtp`1s<C5Cp-o>=2U!dpvrha3C+}KOrdu_iR;SjzASsq4!QkK0{ruRAH2Dkjz7r?PilWDJ=Ti*LZ68vXa2&UXSeCisP%^Za^2@|UG3DaglCL3Z##ShKb|;c5R;_~U?sjOJb~Ba;oUp;c_#vRLXM}jOmYc6zJhRD&M>52DL(6XS?~YV0IYxb_e$TbMOZ>tR210Ow_W73ZcZ48Ts-eBf3%xKBXHsS#Pwux3i^y*5KG-IGfSegFGJ02qOL>ZN({{4MThH1EyA&j+BzBKA~sy#Okh3OwpK|C+9`)Y_qsv<0z((tpnh0gY_mJ6G#3MO{GA<ObH9640;`_BmCgI;;gwWF6a*W97>1BgGz%uwI(u;^keUO4S;Mvq_``m!bP`|Z0+7wTCFygoS#^%So7rPZL-Q3Hk=`^ns1p_P>FyKUEm3cX<*lT;-gQF5``wa3ud^+>Y>Zb}M-2XGZP@%}KmqI$-O5%nzD2~c#Y2L_)r=L>V2|lfImF)}EPGlwY;u7n-`T{2C%)Dau5W*LsB*+IQA7K$`P<?Ebn1U;*C_Nh0C~iQoxYxoQ+2;IIlZk9_9%gNm&)P+?IIaTDc%*Mk*q}c(QXU3D`hC|_~A}4y{<|1T$1<bC9czt=eb<$RTDMNbhFY5_}j=kRQv%mk*vvm29meMAeTQYX9{Dh1#A0qIX5nz%Y&^ib*XH7>ky^6YVR&omFZ+uT)oZj{B(w!qz1IBC2-rIw$<T{Q?0Kzwtr-cK2z$W_GjQCI4x@Tw-npI@45j=m+V##S-Im9ZJIJNM_IrxS-o*+@RhVrBvb_fNG!Mc0wKTDxAS;o@CSH4J+v$Fqan)er4DLee(>4~M2^)^0&F^4;>HJ2a>pz88@2Y&$7?oSta|lG#)V|I_yd@b^!{D+>(z4Z^)+5x5}jr6eWIU~pkC&KblWkg7c7Z45zT?x>ZhI?tvC@ku|33iX5e$3s@vBbSBe@_va#Li!-L{4z(I@XLgV`lWWpEkhZ3{At@7GBg`<E(LS_QoRl~qqZRA)hOAN0R^RQoqD3{9uit(TPEN%3c!g=>yJ1=zxUAvoI!?N~q^Dp;&Won}R=f(6d)7v~LfMU|p)N~P<K~s{}_N;Q<VePt8%+UnaE3IhrRD`M9-Dl0f9Cv8S0)N9%GIGFywKnG2MUMHMHhZs?t7)UH$K#R2%=q4&3;BDpS1SF+R>fNT$+4D=tL3~a^dF7>6C*cQXxATK*7bg`hTj87PQ0rW(3?=dNkjXUo4pV#*_k&s#bNML?60b%{fSFm2K3A)FbdV)<Nf?Nav?ulc2Vi-vx}^-`SWtJyhH0LjK_tuZ|2+#A3WnLJik>}`@4WHXD9t~9qcxnfk&xe4B1lqAdQ~V{o4OIyUYH?&%S^09dIIfPGxu<cNedX)iBD#eY`9!@1{HY(XWt`Q|-L?_|}Au@D8uzwb#7;@fQF}S7T;Zhnuz_MOpETiR>3QZ0z-!3p06XL@#$jAGf=)A*tsfEZ4JT+SIxpZVOXnmC6sos!0p*`zwkNyWYU;l18_g`SHw7mOj%lJH|Y__HL@HFZF8h1BkyU5Gg7J@mg=WldpD_3196P5U}3G-SJf}ui3u~7SN~mq?3h;eYWCzhx)x}i*_~_5XLY2L-yw3!IU^ido7^6h}cvO^S2_dg)dQ)Dip!`FDBTA)?S<doO2m?SOy(o1ngt&(v96$wisXa+%A*G??L)q0AaGY6WLEasDN{7{WMjO52Q-X-fm~}=uBg9qxF+03~u%(>mzp;hgE7^&i2szFZ^8g)VH^g9lz(*#`anr>^dP+YqI$<)WmSWH4gm5|M;RVSVJ*Zv!JNxpBBFcdjKwPHl%u${O2BC76$}eLz~}<TAW@jeukdiwNbZKA+Q8g+4t{&<9hf=#6M;M1&5ueQ9lXq7~-KQyjTB~LDA<Y<|?^mf(Ig1`tKkQQf?aSt@x`Gt$ccakegN`Rv;Umzh~{*QbsC(vAq{<8JrF&(ZzG>d1?N{Ge17p=CT77cH}N^Yv*%Y^Pwy{GT5M}(<R6sDPTc7+ZnC!*KAg6jfL1T-mmnqTU~?H;u&^sq;v8{iuiRo3yR3fMp|@wb<GGGsHvH{=g)W`gZ(=5yo=>#d$=y0a1G9SBymop>t5G!GU<sdP;B{d@~BVi3ibU9Uxwm(sKd$r-g2#^`U0DY6|lqE0?Ho0wo*fD#u?m-2n_O_Ey%Pf27xa-pr?(Vc*4HTxdU9}^W}Z9`jhsvc`9y+bZh}0J-$eNqW{vSl#6wa(D(c`u(`DO6oWzXSwl)!{VLgCgfA~8M<0m|nBFCp3FDsqwCay8C2)0j=3giM&FC)k@9pgtOtXI5iPqX;yRDL|(-iAy4t5QY0Mw)D2&ux&!!x0RWY`b~2ku^Q&$4K527}%uZTZv^7$jeOPM?*Rs1K|d^Jef4-_>VLSg1x9?Jd&avfVv^%b(;l`DP#U4MBe{gL&<N7~P)Z39?YoC;E!kB{~O~o0et%!^~q~b!il;T1)ucRVrg`4QY^vG5jqy4cMu^&{$E1+-qK8BnY?v`a1^PBye7NpbpQwq^Ep_`@Y^48ZW?Oq3#E%_Va8S|6xq}LRQ}yzF++j*Y=~VSyoqD5iG6l@+H*KRj?fQz~Pd$)xJ8?6pvfa=F@@LTvqK9mrLCn@uzxr<$-!0R9HR`WNb@E!~fa_?rm|K02Y_YNlY+;;p)};<~EYN&rGL3QKu=VM)H#mU#w^L@B>!w-8q9gb}NgSncsHu#ggLy6K=SE=+8xql4d=OSsdPQ(RP!|rZpsI)!I;^*Pp1JfxUg$coOx0V=s&+V<W2@s$tECWLt~pV_tMZCZ4va)T@u^Ue+!>y%{#EhpnJ76V>Nji!h$M7KyV~Q(p-@f8#J(xggVFuL`Nz2%ROw>sb-N>Nl&D)Be$?`s)hP4-WK74&Y(J+-zYul})(aiFRYO*8H@7#)kkk^z%jJ)*!32$MZ85gW0-&xUsz?={@E83V7iA@y(6X#_jMhUgzTFg2JWiSH|)m+QB&8jg}Xu(fBYu^N!*>WrEz|eocLyo|A2A>E2VS5!FzW%K1$Bc3!n;e*C=mwz-?})pgy&8CzLhHXm`W?`piWt=!(Ya5OVLYn{K^2OX;LvV;^=#x-`#05fW{EjQ#MHO%0B+1G_IrY2)6h*n?dxJKKSGrXPNaJ9nJOO7IzuTy_)@cYNbjG5Lj;oK%J&IV#RXOd?Bu|Qes_q?TzOZj5I)i_XndRFxmx3A!jo%Wg3AIEAvyLajnbip;!t1ggdl%Uu)1zI?f&#=4v5RL8{!`9Bm9CvFSn3UNH#+EyI++UhyubO)Vd#(v)e4(*HVtPzFZ#;i3Y&QmYr2#qAVrN^395yOJ(XFj`z_fq&YNvI6TZvgyf6sQYRm;Thw53X;Q<b*gy{KR7;_^4q?5q%bVf#Z#3*S4?AlUHbXCF$Xml^j<qqS8MSeVvF#wLKQd`tu0X#6v=AplfHPtiGk#B(ySICrdf>+$T330PIY+)Zy7EHh~_de@Vxb-BYs%7K}=1$yt>N4N0#wRe3Vx3l(zk%kh{Rk5tBPlm>7P+cvC1A}vWy|VaI*jjp1Tc_D+^s?0p?M3z1fE;cDH)2<lzd+9`lYd6_2e(bVSK6o<O73ZmU7!`LH~NReCO11d^kgsy=eKXHO#k*4H;;m=9t8}dTA;i>2$q!$V&=pEn};bo>(8lyHPqr(I~}0sO)rJ%EIlFyecn-jQVpets=0=?*A7k3zmL?0wdMTUG2#i=C!2qg`a^9XJ2n6M@{3?r*_4pdG11U!&13zcS{iooOtUX$75#No_Kofr4nTZ4Kd004snEbhE>9|*s9zsiUbYl#?+XwsTBTHWgT74lw@h2)E3Kv>z%6IE*x23V`~ighCV!7zY^WX9*r6xye=>hMqeHa871q%IUaHR*tN{BgPF7Ct<lBvi^q6UCTxEUM=)ZIUoe67~37o$QG1|<J%Hl^OC#xLxEJ28<Up<;{On);eSGZN5w+8n(ZkX3}Fz(pSZ_{6=ME9p^d<Gu@qu%`(_<F-%o%LSqz4H$?m-Wje@AQn;K$tG;D7?0~T)Pj&=4mo}xa`cYHe7#myh7WfvA#3{?6rF<E=Z2Q(PM31l?OXa-Pi89SwmFs^Jb<zWQ?K8d<LFoCBbU%Ha`^Fox}DC@PnzVPsF=)<zz4$m~{Lm1%9(%b|qi4#fpxO_^K|MO1)g6^WA6w40?Gp95g5FAEA;K^84oxk%fZ*^&^BMna&2l(_lzrOF}muOVc1l^=(ZrgR<Ui3a&F)b}!duf1VJ=)vE7KNpcq-mW`9!(STZJfoWeOFc=rP^Y=0Jj)w-E%ZRE#)8)Wjy&eW!B`We8J6lbmhR-8REI8%qwg3MAaCfZ8osoz%v;GUJLXbG@(U&H_O((tcn_Qk>t)#KsK7cNQ+#5|}Z$}?iBje3~bDk2z=UY<Win1|ZY%R{1{=a%~UO)e4EB7wSK>DC;4iR|VRXmyo37hzXLqd#EY04cV@{34WHMDIQMZlpPW%iWVRi|WewX4znS{C!e(H9-n>h;N-!E+L^*)9J*^xuiO{UXyUTSuyZa&b{2PioL83G>0zP(RavEwY&m8;#dNGT;sSvUYOph>B4V?(S1LF=fs;H(NL5U%xd|Xl10)WbNpPzcaCh(Odi2=`0|^T)o{owcBMaY5C^B?niqq9?WgY{_ShrYVGsGZ&|Bs>%*nd?-6J%cP_?zrgLn07L?~2UW`OhAt0|Kzgt(IY;5kk$JvsYkXjYcmj;f=u9!=C{XL>T65c;3k3<v4KwsZum2XGevjgEfer~z(;k)~w-r?MN7Ue+N_ZB!pjLk+$zTZ|~wFZAA(o3j{FmJpRu+TDo%>>31xvmLu)}9gZV}}cqt&{e+E(KQZ*TVtv-9MaJ_{}qp;xtWGXy>ksoUuBvSbgsBTD0n@i?aU!;Oz>xp^a~b4g21hJeJ(_TbJ8lWNc&cur$`(wEN+&=jm9$YrW1qMvwRk&V&$X;34wvc4@jXUw}g>M988LBLuQ%z9v15*)7I0VQQnR!<DErueW-QeZ!l4CdQ?Hr;MftJ)onb4*6PZ+mB8mP$Atd4gVl<pZKADL+i|xXoV?OXhaO;>IJS!&TrN^o03<54ET1;$gN=WoUrMwS%=vJbn@#J{<TC-RdlDWLayCRpw5L!-;>HahOnOEMg*#itgH&J==146SE8>j=u;h|+s*ME3gfpHzQjgb^dBO>r6V3jaY42#r%*TqCnZ3xfC`k}{<ZtRxpB^){V`2$Pos)(7wn#n{w>~lI$xe%%G{;<tM9V$CI1<ot&e>tNb<qEXopi|_`lsakNkh!I2+>X+c;bo!M9)s?E$JP=<r<c)J~h<$rQN8_p2*%PcHv+CHfq{3i^M!5}gRL%X&m1Z?vKO!lV3G0F56IANlg*kJ~FArT{G@jP6EU$^kO8zSXU0RsMuu_2)$o2%a3b?(y*EO}m7@*FC3PXI*l6-<ZI4FPww~4qoZ6>fosNozIfC&-uPC4diZZvmQY(t$2+lu4x>-m{<2b4<FQ%-6R?8G<3dbvkygjo~;T)jr50<!tI=Wnb^Vfj+3f>wEVhyRJbcoLA%W4V!*HTA9-1a47HViYkhTfYO_T><1&N0FHMn1G>wW!gLJWE7%}7aLFl$UhZ=W>tzq1mGZ=QR5!MA2<ttv8okp$MTBb5D)wF$Jc*w0Bd5!J7Q<K<=bPA6Q+T}KmZticLi8q#uV-;C2=X29dKI9hgdiT+=-cfdZ^94AsHgZ0)BVcaU+VAI>-Oo4NK}2t&5OJ-KJr#Zfe8VO51-iN};K}J~f1CUaonn8N-TgWL_7Ylg_b&o&1#}t>zNZhG+K1GY#@>i^I%4b2hxF>9_ErrG!1m1r)@UO#Rqwl{WvcBNxr6!4>NVsIBh4i0=+zqbR(q375A3NZcC{r)0o~&G9X*8em3^h^V-yW88R5Rok=rrh%Yk9Pm25rpcTXef4o|ycd_P=`Pughe+=$zA(?X3G?#jDQh08+9t!3v;Jl>PD^ycP&DpW;lJ{V1|N=a=Y?6VNt{b|qR4E|ODYUy%?@4wv&ZlZ9zT(-)xtfTH<Z?`7L+na1Ky?c%Bhfxfk@Ynh+SgY`ld)(~FPxj+Ov<f|7U)DD1mh|tguu3~UzWN~HAgnyUXV`wQTg&d^@}6?&@{ig&Djdlz3oG5Pe|du<_-d10voZ9-=48x-vQ_28zIggPi2P4)sS|}sC%G?asQR4UZz$K;K<CDwFFvgev_T)x$OffN4XOhE+r(zxA6XnogQ%I4`gEUjqpkYpin6%kEDYH7^B~TCLhDLPn?tm{=lUyW--0o*6A^TXM?0vp_gng|&W@{It>(PqN9=9xYa?<W)4Tf^2QtVpW!f8g39Fv>O)6birZwlptz)gS@+rIth7ok%Cv~pP{aBA~42tII68Is-I6hY?z{Awe*+((2H;C^+zUmTngHv+aZm}2&ABZ<KOdb40gSBdRh##B5ymRjahv+PEohT(YwWdDrURj9MUI6|b=qZz}@lOZal;BzeRmvzdztNud&6+_j*S!0KpF?w!)4gxSV~5g12$<e}(#rNHuNx3iDqWE<X$2F%Qhk|8x7?bax0`igOYAgMPzP%amVC4}(wWmGJ09Gg=XGyb<3MYxH`#BqoIgi+FuoV~1OMFAp$f>=Z*L@mwL|u|FVCk5xWlLQdgYJHr=Iq~ycb;T$Ew4qo&CkHV!t$yZmxXs8d?*0_RipL_=&M$>w*)TPS*mc$k~z*Ry!sLt0c!yG7o`l;t`u=nz?ul@7?^*>wED_iPQd#=05-GhyGf%-OX`01HV);xXup3yTVDgVDaw&=fEornKV{9u4-^7RR>Zh)3Esb698}poKCJdH&7>t(+oU8iglj0!U|dL?|IYdSBKl>Xu`hP*2dxyre&h~i5yDJg&7&5a?;nk@z{h%BkrXLhPV9=8yVTTGgErwoB*-5zDvEgI&M8|6mNB)8w4eV(ZTD@dSkF4!9^FsDA$JOtqJ!RWnV^qv`{C(Th*NiP5F>`ho4)h&zVhFPv(F5JT_*uT#UvGW;Jfik2w}xrj6gn&VxzoFGy=l13&Y(gFWxYQhsXP{2qOJi!qjC=N<sx;oq2d-m3}#cM?KCHDRzt%o0^vLv#MBzC_};^zH>+lOXCatLcN?VX8hbW5Os1J%SPF^8zQ2R4Bb1L94aRSg~g=QS@A^j?6@%%SWR=0>b?UeL+J&1PZho{?15+UW<s+*D&^_&YHV^K-r#?Aj`)1TflPs^8!t=(qemGow>;X8?RJ!6lVGTYdRoIEb~_i-mcTicyTe&StYB`wb>@r0Eiwf4?0*Fe=>9;uoqmeI7fAklVk9jiS6waKWi(g%z)u_>~e|sT>d+f!5)ZSu9}ToN41QGKAuF3M06@~+Df!nbu>V(&0eL&9X|t{8uhn6n&Dbqm_91YO6&vG%2Yp^z;cr+7;_rzA1pttB_})>StIq)<tM3_h~<HOF@>Ft=*g{PQ10;JE~mdixS_x5Koxo@$^G@6Z><A>%Z4#+SY<-pelT=C{iM1&cqqVHy5tJqo`LN~Qa#YMWT4zMhg5a0f4@3T!3>v&QTjMs_O?v4lR`-7C46la>^_(^E5Y)|yk;u^+uolK;xagDKP?{C9_T!izNf~n)qhhc-EIgJ7~$^wUZdD-RMW!I@6%46Y_D@xp`ebmW@0V4Uv~gnO*}-n4D)V;<moTamJN%XD4j`m6b624kF%k6zh%4ul#T($Un)t>!%--9t9P-Tt$mDl7uK>Z@SnrZuTv``@eO(<%soXfiY$rQTH`zNbX4BoEj){#a!%3sb4JG1S0Ftmqu<NNm2L#K`f_J=JIy`<JF7L4&h>n!@o@|*%Chr)KXm9_&X%Wk9$Fl6X_0sS#Is-OcE1H8>(kWqJg<D37o%EnyLGn3yNJUD)3CNYx|?W)q_JoR@6qRO$?@SjpKMDp!N$vw`pNmjVw2*Sn9%nCmNcVP6|e*r4`>F><T+3tiTnrExn~79YAxZl>OH>%u8_;ez}lPLCVAvrz2zcRjB>^aWn>0*PP1KTH`AJ_vO9yzmI|bq?;+5ol|2ks+7meuhlAudnstd!%RxXzecDHWbYBqM#d`xb8J$gS(Xp?)7Ub)S>uYN`O9#ol^QFWi)z)Sws<P<M|JagYMgT*m=4azXyl}U`=5)%7*KHzDQDs8<`!xviXlHVJooV}&1ygC`4UVRIcsAMJ#H(t4aRB(v?_<vE5l9DK^)}VrPsq;6K{B1H6S+!bY)??H6Los2iI$E12ot>wPwkn-ACq+TVtc1RF9dtBO_tvQYlV}7>O{wBUA^+H1XI?tE^s_9Pr{gJ?d)d%8b`?VjF;pep@)AVKDv~W_VJ8&A9g;p%bo|fEz&-U^Yj!%`L-IYj9Y`1wVp48TVeS8ADo?O&a%L|MaM!EA`lZ1X$*)6*dwi|@B^_yP!LqW2Ai>;_dCfMz`YYNK&tj$D?X_jWaRoU2a0yD-YVhIC5vzG724y#jvF?ZU2JfReb3EYtH1wk!WK4fo@ZK_<SE4U?AbUG9z&&?c~@&feO6G)E|AtROPKNY5QFe4Kxd&xN^p4=?VN1eOU$EfvN_2IzzW*%+d4Rzq;MAoIw89v>b>8PZAf^ERybzqx`5m2zd+rir06j?gRA@oJQo_X4p&??UW~6F1UJxL%}^r$T(Dns)Ss@?`f@%Kq3D$4K6S30Mq2p;hVxD3Tw6zKzi*7u2ySEg^R|T1mL_of>hZmsj7Bxx-%p)L5c8MN9K7O#XBFr?&He=HTa<J#6miwi1%Bk7W}U?FMfPBmRy7%oDPUu+JC|+3w~iJI*q;b)(mk*O7h`65>O8)J>+QsikHQrUgW>#XY?kk;^n*8o6L2PfetaFlzrg0>I=Ut8G4uEAb&H^+6E%*<d+IJqJ%943iX&pTUn>phHaQ&;1+l1~aZ2em&Qw~b6Ll&=;|w84`+?(6xJ7jm!e8nm0G7x2Kulkx^TNHsdY}V2$8T>j(did+Y1BJ4$6&pcxr?KzJKvQwB67^8N6h8f2I8((Z>?*OU3jm>$Jo&N#AZ>C5*EHTzk9m+8V~CO-fhj@-|6OF^<1Lj&f&?+iebN|%F*&IWgJQ%c1xcvT((|-cwAi&IgAzl#cm)5vBr~Dl>bG~E*J?wk`Ah)-Q@NZ&b;#e=umxA9AtvYHjOgbJW<}{Gk-L<6LeOLfMiyYT5|d>w(P56J_6pSx_g*93DJ3P)9BPu!g2BdwO^%#w=bcHnqA$i#C0{mq~!~=fL+cQDGNQI?LazpRAyGQKbau*m|o#|Qp)@m7vazuc24Z<qce-Kd2Tc%{$S4k-p`t~h5Kt2TTO~97o<7c1NeE`IwPfZ=NrxR*joQObu-2md%8XD>u12!SAP%T!gvzK?aG{AbXteQ(>e5)#T+}Mc)qM2)R8(tp%r}_5y<;`MYfNeUV`u$hLc(Fsr<Dm#^{N-w3#yZUs%ypb_c#32Y#bMlS4j67_!Tm&pp4|T1wgo!NOb*lx;&qr@RdAwHOt!BC9n<n3pVA^Di1cL;7qlZ$p~~$HEP?3GiJ_*yHxro$A$1zN_W55!2rLg~B}!ma;MD#>TImf>;HV2Zsc=eHFWk^L31F=p}@IAKh2<c^t5F<_ME99=zkzmVjGhxg0u8SmGc-=QowcE$rbm26BT3_iUQM5NB4lxT#c`?HoIdgDXTg9^>`<s$N9xN3cq{we6&3$V9ER7Wn6M>&o{p+UwthO4VZVJ378ttJH-&jvj0~x7sQ{AAwSP<b5NrrQOx;aze!2grA}{CI;`ecSF(1if;4Y#$3Ow{^E2}nM`&A2E8xLh37w;`u(_W1a!jg)|yN~OM3A24!7~S`0Y;cG$jt+u67H?!ObSC!<}O|L<W0LCk2z|Qp%rd?aKKy?ijpZ4UDowXnyb#krbnbT%H`H*Ei|scH0K0gCAaa%{y+-z7?R!I0z+Iw+pND*Pq|fqj6iW(i6Y`v<KQg$CU_8qY2014Jhk9rFTqz>KLCkWMoXU&(%PMHyOPPC-r&t%zJBA9MTpHQ3Kpc+-u(`j=3&SGScH;uT8Hr0TiMlPdA44YhR?v^<H052PZ#1ZZH121trB)&U{VGdPJRYU_opJc+s0azue?<TXs!k3b&rBa~{c`2^7S$*731?2yp`D7ga8ZXARTT);f7e;^r{;v%lj&A&X}L{9410wZ<0CMV;rZAzKX=7RMboKkE-Z`B`#?cGaKNn{MwLX9GW1(KM(UZ0FY0xFK}(r@cA5r0(dH@O$aul)fq5YBh-G38jpDJhfYgr1LYwkFaFqyy66pT1#7;9{Gs+wkDMyfdyK3Fw)`|xEI_-Pd}<Y#QO{$mZZI#RUa~K)tK$KYJQtq1GVeh(II-Sob?oI{k8l!-wyx+S|O{6GHL?DezY|q1rZxpS%iIwtQt+)ov*GE=&+xEf0PdBlXNBz?Yi;<d+5Hy*^wp6=aF|gy_#Y53iR^!2FqWQZMYpSpWUt2l-qm#4&zn9{!1$X<jo`l@`rw$T_4pai<*Rh096SO{dH5CmlKrV{|L>to(mSi&N{GHs~~nsM$c?kl=#`K5)hT&j+hJy+2g64SG{n8$F{!1LiuroG=&$!YDY>l2Jy#DO>D}aD`n-ZUD;?6T0*Y~H|q$$Q8DK3#;4^}*k^~qXwre&{Mj>tPd_GxS>@uMsW(-rp4IrKP%opQ?W%dy51F?(TW5P`XLSAe*YC`F{FUuocHt$$N%b5fl}!09rc`We*%xX(_iWkT88Fh6>)bK!Qnfn&z%X*ZqqRIm|59bD;#=I^X%%rdT`zaKSW8uIN116`Wv)!f_Gc@T_|-1&We@1TbFjYO-6Fy7^{qil?jpQ$a^mN#KM8bh@_;Mp263j&BS+(1JDmE`7FjietvO@oF1`Tj-0i`099LwM$<5?`*dz6BwvA*g=-iOi3X<>GpzU31`S^QRE&ib!<2D0;(b(~;Lv;t#Cu}`iaZj&?W6=%I;pDXXQPur*4dm7mU06M_+F;Y4J)hKacYrX_HEZ1|^V5Bs{A(KltFItqdZp2SeBtO-)E>*jA}qrrZ=FpUQ2DBHKjqqC7|J6`aqWeYeLPSBaW1o>t*;(F?_W^)))u2@u*M%xG?SZ+pP<8PWWU?og5v<P4j?jYFY8e1m~;%4d0tH%?*wF~e;41*{nf88mKkvQ8E+L^&0OcO@)7&Tl-Kh3R+Lkv(^03cxWJ_N-R-Q{aUAxUUdQGGrs1!nP@b5o()h4Cdikv(r7Bm_ckR*#ZUCp!Ew52~b2oe`1QoVGh;#d0+1)X4Rj#@Z10^1v{GPQO9F^8LyVZwlguCiv=5~!I#od}2Mm&Dj;oEA;9lx{jP0RFa$y}BF!20|BkvF*Plj}xnHg}z!J#J%jg!JlFLpQ3kJA4UF>(yF+vdPN*?iO5~Ar76>?YNKKMLV#@`;GppYRxBj{s1aKz^FEOy?FEeQhs{2q$aRvbv}pTT@MYy7<s-X3)X?hje5oF9v;Mg%kB#iQeAWT??9gwd(_QfiObxv=_XU_eCk`50U{7zk{Iu=dd(YABR>q1oVqt?d{D$~vd(A^x^u+FqH$*i$2M}YuBRpOXqtDTwP5#RX@B58L(a%XGHZBL<MN?F2=V<!>kshDQaE%?-LO21T{+}PYivWNY*NS0rgr=i_Rk>iob!VCJAnr9$AZOn-Xumljd{I#yA(%maTuH6$h?rV-MYg<TiH2sP_*6$<Ix;&2pjw~R)j-k&Zixy;{7bFy1dzgMYZ)Y^!m%9fXbr3nH9RIQGijN;tw!pD2j@u8)miUbHg8#*UM2v{vshnq;S7Fcqn4dIzi(_-H~WBkEc@!`5O36STWRjjm?P^yIo)CC^(L$bv<A^llvohEia`62>pqx>fP(<M<NPp?Eq&7^-w+BX95}aIVH;6UH=Nl!ZX7@T;(flXl@mpyU^sZ_y{+dVo&@1MLE;rcb05~4HUYY@o43TmrJdadljR178=XeT8kE9B%83uiw%3RTlEo1crkLHuGqKD_p}$c*TqxLGM0Ip-IjQ%er)ySv^+PnFl?ykrJr&ucP44`$*Vdd=duCi=^|m3Mf*HXrJW!O0oJQo8`Q-PFU$!CNPH}vxQd{8vo_OuPk@HWV_oRW0_h^>{pHZ1hhTbHoa|Qf-ET0h9enSqjmd(Hl)|{b|8Q`M!<WjabNcvlyR|%sef@z@LXP6aSd{=WE5iy@x%ca$>bzgMB++BFHUkfG0ZC7&H{Gnk;{Gu@U&wHmw~sH`brX3uaC$q5r_biWli0z!vuaaua~twgQce2!1F?39DA6Z0ndgK3Ihq%bQ{beVfZqI6vx8&DNvE&!lqyvmeY2FKL@0&H9K$!)tUes2xN~f37ape9Rg1fRsheSZL%#zpjQcM{0y5!>h|fmEyx*BOdpxduwaMq#uXTqFy92!6Q0uP>!R?F0*6p(WUG`*CyDr3byT@MS8+{n1FRUM4{Y?6V_qI*F?!DF=t=A^i%jXg{RSIs$30AK-wU6VdK)<{D9q3K@7JK0nT`o^2vBC5KZ7()8P{zf?Wus4B7HD;V=(cB?%jWDkZ8oos2$H(f>!!}p_O!2p*FePfp(0)r{cpLdn~`w_m$Mx|g<kVydGt)$7=P|UqZv%EEU`YcWu9cYP-PJFX81dLs=m?B9b<R+q~sR(LCI$f@dDFhD!C@A*lF4_-B8_E>)r0LIXRi#XA)PTffsKm#~)Q5Vr>c_*86hp65mk{)V!g!O^?$oY2LrYbuu)pun$`Fb&&ji>`0p@$Yd^^>>>H^#?^I26J`&RwRXW|-n~4n*PoF{-6Vr`_ePV&!#Gxk!4RB}DE~1%a06TaNxVk(!9Te^i)l!c;1_;ul3DsV9=2?Q*Jd-RhMx@i+q#<V@5tai;t<Y>Xjc~#u$a`s>iYQl$faFh@x<@B*ZO6vcE+F20$e+ka_Zv_wqMpq8s(rJufQAU<T@~4Eurm@@ul-zhG5-s$K<Q7qdievT6i<B?bPb1J8M0#l_3p=yvpT^%C4s*gD)Zk;~5R6X`>11%g(1hU)+XssCS26PG4F>sJeiynRf^syExVOMy?~(Tb>MVUcT;rsxkKqX5h8}tXrH8J$s2<kKrBKH7?uwt3Tk_O?Os)2Oa3!AslNpoA`k3ME$!5_c=yBOqq=!MYHN3fYM{%{Chv?{_T=NGfx{+duMi@+Rf*W)T1C@_F_slFYSNqApPVb2b;|j`VI=@<#3`-_gxbt)VY2J=lkYc6T>-7D|soka=zFM?{W6s-0SDF&|2W)rVTQOX}L|}CjQiCI{x`<Q&r8|e;{<0*{Cytgs37bHK9Yev*Ar_pihPxW-X3(Dejd#Na|fR>qWnLb3z><0I{4xa*Qu9i#-n7Z}w3apt5`+(E)jdyEpv@tD^bR<Kxwiw5P3t<7afMGW>vt$CMq%J)Ya=<6aNP+9JsfjvXvZ=WM%5{`Mb!Q`WxlQ!4VP=HTKGlP_z7$H}wAwOjo9Q>!fqX|RIj%R@T%E4vePWHdR^Qx{EjByqH%Q1O~gfVlek^$5pr=S}vL)i$0&lj|`%d=(a2?zunapkIyI4R5$>^)_UGezKPSj8Kl>^}R-NBL%d_7_uh&4lyu}yN1*`l0Rt=8X`}c*G3JeUwwO7Zec#SL<e@hUA{7V*^O@IeSK`>p&fW8fBCsnm$APcw)?{Sz8S6ns?IZ$OyDwnA({PYSgiN~nXT$(>PPEi)iTla-Zh$S1F4BqC5c`XiEd{7cH<QZ@0$Z+VulRg6U-*$mR=6^cQ!#lK}8HIBhd5-E($|b(&n}-Zf$vds$HL)n~6Yh@zWhrm*^+(qSr_@zrAzqx(#XJx~RWCcRla&19$#04u?abVfa^^#tGt8POTvZWY-{HWZ8IdnB5lwXPRRWIaYvfQQaD_*9qIodO+?6X2-5AjRR0?LY`}QpI_Wwi5-mJb$_3MLmE9<HvHwJ>ZM+3r1c}vI5Ja7EuYI#0-lPbe#a_rsB`7F(Fk;fO34bN>+ZpCNJXk`-!3u<B&L|&IKQnQN(8>aGIC%YZbwwhb9LbAINP`p&&AHS#{%ySe$(o0@qQoGea9GW9O&-%r*k*5hlkpvy&aX#v4a~jx>`K_K^DvIminraSq^QNc<r};bQ$68;dCNZ-)#Ro6~)ud=|4de2KQ@s%BkCcA-K!>(gyx|d$TU#&#fvB>OC(WcUJ@4oG#{NKCTp=cYdg|E*p9C{gVdQY~UZo5p4~N8LFPOM`7`&d)o1Y2*++{R>^x1hu)<A`7c=lzdR7HKR}UtrP3;AwMDF{p?VhmYt}CEi4V%X?lFM_`N}#p*Z?Z?aK@CYR+IHRozb{kUylz{@l)Ml`9+u)Bi)^MOn18~yR#vi0L$*&CEVtv0QyU%bM^GM#*#C>HwVTXRZJ9T<FrMaWM6Y;52)YnW+qQ}-9@er)_!3t@$-8Z>sBp_2<lx*$`PujuhkxsZm;UVqwxoBz}N%7`Im#*yoW$Xsb`sBRw%z24y$62s#w0aH@pYcAI;BXbzr}EImndNy~LQ(JGFL!d2J)rR{I%gi}9{bH`XQepm;%A+RZ(UWt01|-S3t)_M~y2$*HJSlOt@-2jrA`!glxgJvPASQd|2M&l3_1&1uZKRGH0S{^8j@(Gz{K^W@cAKO+pc;_j&yc;)l+l}<K(*u4%YTpn<fI7*L{=k!>s6$*IJO+dBjj%ruYu#id+GS!TTC|EX{&GXZv3e2?5%^W?ykjfCf)mL8k`#yX_@^(0m_dqDzv&U@FO0Rpv>AB5~XQM5fR`A(uvUiwPGgSLGVSBJ7x{8|fFHFiUJZo+~`deq!nzK>M^b>DZ#VB&3=1J(%bPqg)t6@cM-K|AS)=&0wc&`i6Ny~5p-Ia}34dw)3HC}*MX)%WDWAwG3MuFLW+JETxRDX8_>$P9ODK3!KB{MG`iCIZ1(8jQTzFVsNo>;)L^*sF>4s934qyU9lb;<ms)UFxhh6>4I%ka@2FAi64#BQFxLN@$$ZK1d)WM@_FCEM%m^|+lT&>^9EKM_N>nfNggc91qe0l|X1eh&7}m)?s5@E#;@Sy(QIeu!Rq-H;BLui^FtY7IGIo$k$Q<K8V>U+P--(X-@SJJ<CcsbkOMXgg|kj}MdZ?SxpAcP>PRYE7`50yg@6C|6rA@?}A4%RbPpY!eAxzZ)|9h1cmb$YBtE+4}V`=7O41tl_L-sZvgrOI=P0DPLDE#75PrDScV|^>1yK!k7)f$-=XCcd_+LSo=OJ8>Zt8#!B69JJe>m7*5YXz0zYTmLC-R?XrsmFbiv-c&}Z0q%oh}&zI6ZpOBqe<2vUS_8tuYN1TzA%OiEZ`*O}p;zfJ1bMwW5=N(ekcCY2~-8Gk05L6$B2dgegt1@{XluFL*-o31PcqFsj8UvGFC|CD-<#a;HaQfrsmlbl7gSyq?U*cP_M0byf$+GN@S{oilsLS^pUaVTsVVaeqsgDko&-%#MnIIwsJ%~d=dHwL(_;2sK(5z02o+hUG%b%I8jY(C|+tnD)D5xf|{hbU*oy$jNCe5)H>E@6~e&daEu{{R0JEPPCxfWt9*du*?!jW$-xIY3I_THb5p7guR|J&H8*%wts`v1?3TFsMp+`MP0277YUAwGqpK+?B;8Sb(G2^)F`YvDr|!~D*`sE6Pa8xQuVS>HWg5L~&`h_Uz{%hHa$Z=SzfcXAN=@#i3<wf;c})Ol>5b}YLaQAg)J+}D}g@iKw#U}qKPx4q_ltjuTc=qZQF(X;i|N}Tv8t8zcNtoB`2k-Hx_7>s7`@w%n?-_~}&ZcsP7+KE#?4!6A-|Du6mQl|i5y+$y``+T%(Bu)1tv0@l{)SGtAdQ*QcUoB@p06ezjpn@Zm&-B##Fur+fwliVT?y40+Knh*BaMFV{*U4h4{_$Z9ba#_`*^|1jALu3)#<+huv!c-*>WM?6J3{wjR82TuzkON#PXo#o15JO{DE|cF@vhV<*)?h8JRPY{UT>q?!#l49g13Ibxccfi`f@Dlw@CxfD*N7d*z<~waR#)c?br8DyIO_>Az%8T3(lqmG1_Qe(|&1M6hy-zxH)p_IywmQ`n0(MbhKL@zzRTTA&=C~GjRRv?fWZEd*yeYrIaV_y}UVF4EgJBl>6iAUi!ncn(?_X?xfo<eOIr55BmraD-x+`0Me;9+JoL(g+E?*-dg{W`S?)=1wT{5#j<OxtW%Obnq7FJiKkC~4(gx&q0txqig33J|9r?JoFDaPhG$?$Y`pw^B<Q$TuMW$ibD6i_Ti+MAQFA@J?7KVr;f=66wg@liIJsxsX;oaONdzxuQ01*|NAE3)RzHt*-kwFl^_*4)fO&p>!{6XlX$9!t7viD%?cUgR4gp=Q3EJTOZ+LVv5q=_p1{eICy7Q^n9vs$-!034gKsMGXH$41IzGar>b4}*o1Ew6PL5X<LW*c3SfSpj}Vbl^aEUDJVwbEY>>&GclxnIC3+PtLW{(rA>_uHb2os1}J`agbMT$hz98I7vl{kS45=KmWW4U7K{k9zxLIocKDowwiL$XR26qltEXDyMQeYIK6<=%RYu-T_qR4$od6>g{-ksjyJL>&3Zc2Q=0#A=E(ah9&ITeb8Jr9O9#YNrCCzXHTRWEP+uJEU4|v9G<rQ{alFA;%d+B)%TRw18v;(y4wu9?2h}J8NL>vyh`*eu3c>6e^<GS<-e=kH*S3t!-sBAy+)5M7XWm3o0s%Y`?(z{d#x$)O^@M0n}+r2)*PgJ0@SSN1QDKVHFr9CcDM3;CEkn`blDXn!yalhRO{_Pcssrxh8>q3N~txRKY_P6E~<!r8qfdaDl&+9!gVrnoEohvvJCgLK?7vQ#q`k*v$LdZnq!57$vgW(r&125&+${(iW1Z4NfmTA-S(w+Nc~v-1zrXNeIDMuw1wzJi|h-hyfL!NO0TMPCV4VTs<7SRcWOZQqS)V$S3qa@j_f<KY{$6Y{;l4T+X2q*Tc}B+&uG$syElZVr15Rl2yf^Sp)-!e&^OgS4C}(r#pdqHpL#S24?kNRCr`&ex%PC$ni%@6l3V~v&re4EGahx+>6Bdffz&nlZcCr+;_5z_->L`*%V|Qj-96i!i@ikN_xTu*H&i=q9PJ^$9;0))L-K>HSEdF1`2`_>d|#FC`fi{VnFU4=LUb3@QUme}#e=f~+*07A;u>MChUQd^%Ixb-_bM0ZzA0~#*;6mQLj$hzN~+~MALD_2jX_!^%|93F;a989TV<dCjefX3Fa;Ml>aewQ8|}Mvh^qo7$a)0<zQvjnrVf2rcj4PH#yfNJ)65DL#G&IW1-pH(O@pK2jY7-Az;q_QurJlMRV_xXT8_KZ5C&#g&0xngc&THYjPyF?^l<1f@KK`VcQp>_P2UXK7-CW)@7E#Ucp7DO7aLEa_^Sr|Ac3=q^EiPJFd8E7m*(MLh{bM^O(eBP{%X62_%81rA=~5_2Dwh%$BhA@?B`dXk5z1Uq}~q=&YULDGXcIoN39XaJo>?d`?TyA-dVLQo!VnegS4~mqC0;OwMT)CyBNB<VG`0f`p???S=28#tn)K5PuShRE9dYe@ulHjS5Ukohi%XQ6UiYFFjbXw7E@xY(#)YfqreR5EWrb-BhKf5lKRRDnbY_+oHV2izP<g*vbyI6)2<9&d+3ehg;;v%BYd2~@I<P0`f`7?-9c)@gYI9qNgYqaWW_`No;Wzmf0u~-p#3<{?j`D|$IW@Kn4xt}Dg86>&%efdFMZnM$EbT4T#>32(C=Gi4)@_9>ei`nso`U&5x#3uZHTTVq|+{fpCXcDHYMG&hRIhh9-p^Q)H*-i4ZaFu+JrU~RB^H#$<_JTa)wv&&bCwrTs*t!*A$lcbJi_ihsMFm(Ni#mKbT%whA_dLu6S+Q#3Nf_g28n*GjwXBQ<^W%KfUuGkl&Z@S@NrWx@ewW+%b?zy{3M8l1*b1FDCr6N{QzCGapfIXY($_+1MTsJsX3hkFgtFkB>#pCxc@-thFxJu32NHpWR80Oo7{e@wLeoKf?KPpj$5O4FHl^kJn8&Np2<o@oCW-DA0RT#2Gs3LQ{ew25}Qc&0N^A*P-l=^(dSkkIx&5KEvDe=FZhktrurud%t-ZIx8nLfQdBca^C?pjGa=o?gxM~=*8>zN0%mt=RQ>F1-!|QoFR%AkEl5RR(^{5Z(8f@>;SrNX?G)D2yr!<eGLWx-S1^cwp)IyGAAfKSx%EnR?7oO4zMQoIYv@qQb%@<>bu16Y>Uz347AAw>YmNU1UXBr(DRu$_FJihlny(~c{F*>w8g2}ZX)=c2>bR}H!UTsN8KQeoe5LX``5uo=z>^AO$`wL^qW;e8IDFZzcT%Ej8X9-js<aFN`UvhUuqb7>Bv8EB<-x7YxcVVwE#EGtlUZo@~CXu&vGok-5fb4o5&Z}l!CyO>KHY;ig3w3+uzug9c^p)xP9s0X`;TIdg)?XM~0?(cs~3IQZDfB^?2T|J+2zdD`W;y%<lBo_d^0`8NEw`54_&ID&y;qCT=d%)(dE)Os!ZUnSJi1m0fYZL7QXMYpu3<w{x8oWAM<KFV4XQMPJdSZBTh<W|#n|gdq_0TfjUBtC+WY{;)3!7Q2bj49`~3r7FMniO;dh*u2yZBWGVKH)$l=uUeoifn@gF?HQJ7=-<YnZVF&|3V<Kb=%$Y1Sn=oF+h}x$G7x>mv!ng!?hF2Bcxc^v`q>0xM0BaePHyU-?z*b`V*6?bP36_TZM=3|ZOkUut%fZD2q*5p^tNrcv}gE>D-)`wUPKdq_BxY<ah1l#BU8r$?~~|U?k0*G>`-Yq-2%M-AP-4byX0?dLyfxER|fyVYh_qp91qO-f$e3X<fi0lkWc!@@ak0IdBc1yS{=TckMHLf`EYG{$3i*Viu7{M8|3-eu889B^H{2^jUZ;am+1gAXX+L1UmM?=Srl@7f$Zooy71NNYR!Ve`bHVxp1I$F)vot%07$d(&pG(iCe18yudjPG5I&0W)^QAES9(6xZm()IHM*9VzR!?pZ5q)!`+b`qaCQ+hR0`@4ryYRFUC*B_>WkKA;k3&0CUDueEK}by@H7=_?pG<y*E8_PO0`GOFd+zg8#C_HY=;%!*kV87aAFx9W~%JwQhOZOt12v@4Y6djzR{6CtSVPp7?teVTU|acG^^zI;}IqcH-$}TwcEXeYYUrqGrp5DeheVO%192kuK6<e`$|-~H?ytM#@{FZ0kci~<BE;eOoGoV0eD}<lw9@X#1NHjno)y5#aFGl*B|X?N0oYS$zcO{&hCvjMd8xlUc2ye<=&|7`$315*%kkz(CT=dGzxX3#IUWCKE{0OoCJp?jH@zyBl6E=hjNx%hr_bEJvXeJt2lRi-X=*lnq8YYo;HkGHUQcPosTpA35_)LI?7EdAFOKm`~bSM(Q1BFhb-8A4B{vkjde9cf?xqmh%(mOP}ux7C(j-}QD)OgA1&FM;?ynL7;iFZbI$RZd`0P0k;;r7rJYrs4GvLHkV@LmdYk_AMOY-vKd(wTi>~Ic+KG;c)!Xo&VsX>%(|+#}xNlO~c9dTmKNsZ;7+fg@UU;c7TKMgWuRfb|ezEOh$%~$=lMhs~g63m?p4fFeSo>60FRRT}NSlLZhD~7VI6u$PX)&>2VuX|an3W8~UbI#Va&Gql_A#n#;w?UXohNty`vjeM?L`=$6`LP8A-Z;FJMD@@=dOOCjtHHWZ@`~aXGek!uxs{AYCthvY5FVHuk`>gZ)!c>xuu^bb$ms02zSIE=hh}VdDbm%%(1fq6>_IKD7W$^SMaqs4yOb1z|r%^{59_Z-c*ar&r9OhCnUl?qO7dxdVhAVv>Rn`W;0T-n<H{@b$p_8Spi)W(Z8C;_P$Sl+;_<B4iVl&XSV2E>b-7>i#7j7wffcPLaNM)^}LoHTy?<GwJu|M-zDWOEK6{Xi9fHL=A}UQd&HQ;I5unE-NrYAE(BPB7{3<oQxv(1&Ij;Lg!W$T7E@|{U)B*nUtd4x#?y!ogWh&84@NXifxSm`$mX@C-mW!0j&|~9QIJcQgjp+C#G%)pjS^|%M1b>P(aLK-eas*lI{sktvf=v&Jt0mfL-MsnvzL}via6KidVrm8`t7RKuri8Z$E|wGz`JdSGFBJyLr09ZXRW{d8d7D*(095m6_aUy-0rY{O~*X8kK+p)r&?<|^-crkb$Ka0Pd_*@G+@UK6uY{>-k<r__`F!6JW_u_Ls>>f=#7H+=~j?8HrGX`z?<a|d|`9uXucweW?NwmXYdzqXMIrhE+Nr<ezZkHw-7sI`c(Jiz|%g)#GI}gSvnrhKcyU9@6crU`}w1&Zj0Q$+5XCRjT#g4dQ&N=^(<6;=oz(Fd^Z0w3Z+}C^j8Os)AC!X4KQ-2bdJy2@xVt)y}F-o$2T99x9;)L(DfWz6THX_39F_*Qx?RtYaM^xv*|2rT`{se@*m{mosjZ-8_xzcDBpovztuR5Z<c>M4PcI%4C~?*QM6vW)jFEyxZS4Dz5KcQ=<sz7MRNZ>J(Uj)Lf7WRSPwb}^Pq0yj=m*pBpaNdr+Lh|&Xu3LYIp0jfIH4Vcf30fV|TZ_(M079lD`(Q?FpmSy(Pb%qv!ebbQ6l7(=%{q0H0O&Ha5X{OXYuwC&?<gYnyP31p=|k?iQnS{V4arn$ka>QCRq$JAJtBdP;NkI@kKOHHulk&NB}92PZ+u`)!hc<^w!O#D?sbQXX&GH@sCffz?##s1WZq<*hACD_{xl#uug43a@kL*Pd-=O*ovAbWy9_lh?t?w2!@-1wq3)m&B@Ck{HCbW|gIC&t)yWR#yo>zVwuyTWy-}QyJHx_B*`tyY1&xY6|S<#`*KqI|`nUD%JSi^XuF)XGt7<D@}hOS<v62&LP>HZ{Zcl1FtF^bTk1%Y82Fngq2ViX5|})B9+#-oFMym9Nl4c^^rN$bwhMLh1ciYA8YUF+NQG<oF#dkT0c}5TUNC0G6w`izms~+-5v#xAQVWl?7z1Bi!hQ)Ka9)uv-R{cS&P{1phe3LP8QRBJ*}+W6J5E&!5#);--0HMMWrV*aXOpi-;Jlt%Dw%*fX{b^L-Oo`(0A`W*~wcNYj3|F{j$OUy36F_4zsQyi|hQowzL|*q!01<t{s%t>4mh|n)lbqM*B%Ek8q>y=!%KGeiZEhrJnC(2>j@u$gV5$r{vu4x6acOgUq<ry5gx;!|!^3E1r8Do6%%rr<qV0-Z0J^5L>E*9##g&ZL;#qF-jjCfmAv@Hl(*?iQAC&m#(~7yw<z+`;CIq8|1nJH?9%dshA=2%$-w48E16je*H>!R19xJtRGcxhd||pqXvdEYGA8cIC>9}!?Hf!j0KzI*J^JQ&R9rjNWI3&qI(;8!+5?<z(nIzpKG|_?5Frv+zLU`w<-_K*nPUmb?Wt^=>2W}%-bY88sKYTxJPH-R)XHkX!e`u&omI?X5}<S-9hb_y*lgBKB4m(Yy@+@^2a(|!a4U7`KZ6;L&~i^o>{N##AnNFJ<H*4&uwq&`z628Pm}vM6Y7pkBjohi1c7*H<7Tpu7n|r+DQ0gB-<T<BsEgjwUre#p>nJUIse7n}V$hzZzJi{-GxwGanneY6QwFiK6o=H_P`tdF@~HG>&%1?UZWP+yM5I4=O+0I6R(W}k=Su~!=!Mr<w$Od4YR7@Ji-6Y@%TIn#h}1xJ2T^a8xNhM+m8=s|ZE;?ifVnLDda$Flt31Su&8wr#jUv6~rpNVaD!W$IV{8e0*d5{@VevIz=UY~MRB4bH+M!n0sLEmXr?g~8XuB{5DQ9pQ;Q+hw>6=__a+h!m->p&pw<FUzc6&8G53yp-k%Z6Saq_rhpwOHM@9`yR#*KvW=7GC@o3;2dkzga*>M_}k)@p0Hf;pmoU=FA4{donI>hx^+4@e7Vm&R%6_k%$c9MXnR&7NSRs6iMgGLKGnUA&&#VoHe;)292M;PTkSU-k{ZYH;Uy01pBm!9fLW&AD%<wtiRF%wiuLMw`ffj7s$rCk)E0jRkuAOzMYpUZ=8lWiXI~r^7l7dWt$Qx0-Of=Jj!SBpnp}YDT#$(9HPhb#yGwjb8_Co-W5&>lNF?-MQKFT;?S67u~c)#Py&1++sMZJTDAZkl!8NhBnP5GdRy@AW?4cD^IP#(`_Lxf7xU<Z4ssO7s#!QK@TG%YDS4FrmeKtME#m~(B!MIyE5wij<{`lRvRK(j_Z|Qp<xC0BkrIABg{U}pW)Gi>9zC70*X@ab*DB(##dV0+K1#hfN<#6AG)wE!%{dR@<r4#=zBAq<g?4{^s_jgYwNl4mb46w%I5AAK&KPDY^LMK4Mzs&x&{}X^=`!N4Zq=QAAgi=fBm9%+F+{{*wtoSeW}*;R@>%OdviIaRk7O0kI%H6Rp{W@#1zJIy4)K%cxwZ1dHF|bES8HNbNO4_TcR$_jyXtdHdW;)%8@nQ9)F%^hQs+~^yDY6^t7i$@t18&vRYfft?~KjyYEpOVJ-u?KYMp7CBV(C$@MY7emM2I8LG5d$aJrz#!Kcf^^A`BR0?+YJ*0oZ_8=Ng6&rHiWqfm$y5g;UVo&}WW8Oe->sYOi`@fPV`hB)lfsPejde+ZBv;J+zqpQN=;_sc2wHRWNk9$Y(o>oJ(0^Ko(aW86qjo9hU^Ln;yOvxrHhbrAW*6C)vv7yj(_HQ6L51mx$-~1wR_n}}cz#j#HEoTk$ZT4}3*o@kkcz5yyWa1AMhR>Q*S%2OaYU7Mk$d~UGkh;m6%cpV8nOl7Zb$1qOg;TlxF1w85{vIhz=&fVIPK{*FJfo2!T%^f5n{}*_Pn)MZ$d5(|gg?yRZ~#Q$HoO!#@jS6F`I5DuooUs!6TRKfaDJeQkv-WxHJM;`BH6r9YzdFiZ0H8#x?`MUh=1rxDFp4Bxvzw)#<Pnww7Y9O;3^u9CcElJAVoEWYrYKxLi@QYfJ0Mr9XEz9cD2Whxq8~}hQ(&GW?^47;bIuCwE5`E-9UKN#&5?~#T;d_*<B_3<W4r!t<f<WZ})No2S&D|&Zkj~`NrnN{UJ@lj?G4=;~@D_S<b?FS+B1LvUQNIQhXlmsN4Fb^|?apz8ie+s&BFp(1R8?s&+{p8oi*RhcB8hvD>n=Zcp5xXR3#H-KG{;&W;%L8ZK;H7@|(Mo+bP^7ULwTHeuHMq|1+X>`E2ldLYU(c?LqNe<A@Iulkp5vp*3c_5;mt8SzK6jqkR^|J23G(ZOmu$GFMMWYRg&3^yu#xH0-E$X?&HQI95{!N`6^u-eT1(c+Mm@avE%w!`Jp?m6EGZNJA#VAn#G8HOGzlf$##=Y)lX=NHBrP)n1SqGn&|y&gcle_U6(zS?}%e$D6y2Iq9bE0)qWM$*-^#~bBTbkC3Tb~rSF_gLsHI?bDI+k1HQAp>=^XmY1`@-Bd5C0WZVKZTchW1dAS|Bk9rMZ@~d!BV(K11R2#Dq=?JpI~zXSuuNe%Aw%2e!bw;nb+}TKESp=Y;$2c#8k{+gxiPqaL(>J<Wyu3xdQ<Y`ulXc()6$>+2@k9j6`U46>c#<yz1cHVAU~0Am>zAb$0{*+4$;{OS5Xv`k*gT3n6Pgwr*Onlx<6(==c*tfmpB8W~S<$p|-&H8FVJ{6=0eQ1+sH}6*!5i_{059HSmS1ysge|b8enp-kC8@KL~e!kxo~Z-+fGsxIuqlzOBlgRK9+Gx<Anhr$kuyarx9bf1fr)4t$kr3tRH^aTtHADtJ-p9J;E8%$}!xJ)imVy87?<H+q_lxU$!)uFKY+!ZoQk4Mg^<=2Fi~N;60i;%0y4RYJG+$d3y^{cc7hMRR8OqJ6(!1k*V$$scpQU^9orCd;h4td<v5HX&@5^_M$8{=(s_&KxxE<qci6nVMHBJh%17>6{HlV`F%Z8-u%w_Sycnk#|AJo4hzqoo^)KaNLB_DjVH~!kp)&`j<c2+0EE|2rQgeEx-Zhh}L7<8Fs91jNfQ*MMVpntjvYL{OGHFAcj@Ypr7B{Q|DRmRCimsvLn@p#wFGj{O(fPyD{aW;bybznQJravi%2k%BF{K-8=+Hk7fy^HG6ceS*zmVWs8FjZ8Y-5u+eD9w=qR#s@UW3^saq<BW09~`^fYn!8UXn$SqjEzuCrIZoHu0Xtu0a-RdwE17kM+`PUKc7~mfDFa)}a+4##do8}gE8}6>sFs;7XEZsX)?Y42koex=NG5S-IirNxW*hY_WI?kjl4{lC`AiqPLL$u>lw-#5QK2LF%DQu>wnr-l5Fm^F(7W8>bU`Z?FhBfnwY&m;39Vqw3p2`lv9(z94eDjR<L+qkH-_*ky7p^^n`@>-jTo2aoSFaF&n204q?5@egMDJI$b%^imeMP+3^FDsl?S9bK<`P3vp?y_448PctjTGHQvQ>b3>HfLfEeVH!Q7nHvR^9bi^G6zrVWH|Z(>^Ipp2C2^rCv>&I*+I0uD?tKC|L#9X|q<j-}yB=uapOMK8;$?vN_Cb`Nd=zR#Zy!Y7REQ?{^Jy*Ei}WJd3!c#(Eoi1D_gj4FZwZ$<Zp&`73pa&%m%&ngnkvt3|Gy(plrlZnB@JAFfG)hewQpf2Yf`TPS$V`#`f4Aoizoym%@+(mxFrVraKRJl~Xyzzv})=<R<U(Vobc*IoYL-Q(t^LEfNy=eimtH%M=K^A-j+h<}Ckpl6$C+tun_aLGaYRee56_l+fPN1JQ_+U4&T*O<_Y=q1E~W9gYE`6OFE5lb5v7-WwZ5}`dUQLO#K6I0IV$K~JY(6>sbHHrS{yO^k_9n>9=8-r)58f;=JLxO!vaD-#pSHM}Cw5PKu?>BOJ{)V?8oTZ0ZZl>v%%gW_!ri=HX8ex?$8cWN`foS%Jk)7a`+Tzi+sRBy$EfC$RKg;M5{nPhghJ%^v3pRCuGkCtnr&7&;KOm$DgHGVyzWzflHu|F8YGxfB9nKzp^C6L|QbA6e+fo;60;Rzd_WaC7TYZT<zg$FK!@&Te2EK-_^(G$xCwjOne|#<Ex9|F*qkom55`bM`3qG698vKxHm@b$0Sx1ulnQA{?Z&@z@@-#g-1&KDL6<8Y&q%5trANu=PNR_1{RFBN#M(^>Vi)`>xGau8egx2-+{SAz$U-@>V`xzg5t5;LoJp-3pZgclSJN?VDqR0})W;5Xom^M-k`{6DLc!VGENB`>QZR^{XUIj>|4J~}}jPNHe>kuuFMs0Yhisk2>a>BneRZfHWnhZ%M3wZukRNL0^IuLPVxf?b-4gOQz33XAyDi7b><yPQi^!VQ2?KL^-nTtQU&f2T&$;tfcc-#zM;j%i!fc(Jsjs}6ongaps#LTU=?6o$Z3Dv!A9?np$?z2@){^jOK=D<p#x1zkA(l^{XD;(v7L12<7<93^)rMJBHp>=~Rv*=mfo`yrR(}*su^h-<!htpB+v(+h;XV17@TVJF~%qA)}T~y#C`V_t`+KU*FD?Y@JxX79N+{)WAvi|}W6}|(zDf?(oMsfgXz_TAiSk?mD9z0T`2biDjzm6y-&eU9K=cCTGks4@J;O&*L>E297wpakpIPW~K8=ux}L5;7wdEiffJlWZD^R?F9_rhSgY&PhIeRALoZ8_A&ZE!>nlJZa{&XYd3SYtRDPDK31E+t;s2)xnST~8n^4!11mOsI^EsU8;+ys;eOmD$EI>K~~ZKW@{B??%Tik$;?|W$~6cd!^U%t@LQbUyl5{d+$1{WOwX!&-1(${StUdjOowku(FJGz+7=kDLzMjU+g(3Hn&HwwKMqUw2;c68awA#dmKiU^P8D{2Ro{_MSZ#SM9aSwVZ8TVHE+D&+sy5o#Q$OMNxM?nmHqCo!0~{B2T%~*q9Wp~DB`>n8)yUt8fg%5`t2{3sgv$>gKqbE=e~E(y6bN2PSQ!GQmMI`{kP7-!~L&FD!KgJ>prdL^u~GZDYf7~k58<hC!Ut4{N=;$%3WAH2=~sjyDJ+3ec?IZD=nOLrlv!wx$%gRPh9P<=emW@>79B%ywjPfUbGh%-;YyosiWmYG5UBmI?;Kmce;~v6UBq0z+Ce1xod7^-bbI43&*+8WU)6rmw4EUhVCx!&(MLd9b_Y=^UaBX89VDX&dcY8!@%2V^LXKD=Dk0<wf=a!U)I`B566#p>Al01kJfQv4zcN<5B-PQPGGBizWmEvKMEgcTcz5^>)U*8St}c_=Yit;{9^V@Ki=)0KA+|HPY=g?X0-M&x)(kESeY(erJ5@j$B*&m=vsYxdTM-sf8wKawzaTXsw8^Z*N@`)$L_7)FpwaB|C)G6tXD@@%4@BS_QUSh)5_c8NwZe$Ci4A_uD|niyq&0BoiD^gPm!z4Msl=Y?Oas?nWF!B^D$kxKgt_BnTwZI|9-!+P(C~zpI*~rqce!o)<Z9d)4ob+jrB&R9x(HFw>z8LPsU6_e=e?s=8N&k;-!AEa(1~9tJZQ0cg_8&$o%A4`C{X^Uw;WbPQUz`yv_C#YkRlPJ0JcheLb~#Hd&2l_8a^4^8M&SwvfqH#@APtTMvhijp)qw@<jGNncDP!eso6{Ryuq4PgNuZZ0#H!bV7IQxs}#c`Rw97pS{1YycM>#TAh|zm?#}QR|;l&z2Vn$o1IMbqwtX1OUGtfiG{V9&GhH?<miMsr$?F-wb#IE;b86|-`vV?KGlo*^Jw$*ApN|Ozw(E6ds*$YboIWqwVjQv?1r9d&AGSzsZuvv-7GdPW}c59=HtIci>;Hj!`1rYYCUj%{I-5&Ce9bnLwj#V_4rQD<l~piTJa<Da-Lc!7iUJtkDr!DGwpom?V)lRZXeVO$1@xKxoR@Ae^K(EMtZMW>3yMA{@D8Un0S7=Jb65NemaTVn&->Kjp<(N+1PuWo1V{<*R<uG*ui0{c#)}0UTEd$=W*ruAiG#hZ|5Hy;j;qT-9~-4oG1tGo2P+Xy7Ac@zZx%1zV7FXQ(Lvw*4@^xx2@f^&-&cLBU<g*X#Oo6($BP=-o)N~`eUWEU5k#coP4flT9dDv<8RSj|MXQU{b5dQk51I@H}8^2wq3uNyt=Hd#PgMCdit(#R9k3Y9&IddbOZfJvsZnc+N?Fk51$hI^_kapzIcJC+Dv#Nvb^>(RjnnS79c+HHn(;7^71_QlGr#%y+rhEd*i5iFdd!PK2EP731NADs&JH<sZ?encj>Fe$;xDJIdY%bt%MKn=XcBg_v4*dJEzZ$9ycl*_i8^@>cHwt`nWG^A{~d4@*{=9v)(pqMzc^rum4J;o-d&T!)S)iB9S@tf8m95T`wOQW~Wuxnw@j4qqqMW87Um)cG4Ln@@BI+)G;zr*vK5D4mS#^{iE#^dJ>+E#zqPUsoY)$(d|?&bpYRg8}Xt4#?H4hsr=ZI&mWBieG$ZIhl9R(_`fmpsofb1qTUJgN8Px`zoUU5>6cFBa#`GQPUskej@4T8UPrAXv(hjNtZP1dkjiJFYdp-mM#Ey)@a(+UHO{)8<a4R>^-L~z3Ed-6tA9kE^8(Dd*3`8&o(Lb{AhVx6B+Iq>&f&g%iFh3JbI$69xO3<v_L+-1=Ia2`!Os33o~W?UNHi)f&1ItJ+3Y@^XINZgOg&Hj!OcuAzq7w{mC51HhVAg>U!oHh{zOB3j1*3`vquP_vKy!AynrR^sl5z=IR-_L;SKxl(azOX3U|hB@VbM1ryb4LbEk*t?ZU|s!613eptQ}b3wU@JPV(7Y2KT_=39^@f|LI&Nbzn_F7$WH76SB=3oriR4lZ;3anvZ#Mkj>_|cQT$sa=52*2btUnJ!E8tY%y$L5ghUY0q&Xo{nYw?27$cOZPj(Zd|tr=e&_gfXCnnPqma#QWZ<m(rL!3HF+d7v5M(&dq2M@4ij4|V1PqR#FP~>J8wm8NJk0JbyyScPKxhU3e9)J<%+Y@861B-30n#J@k9IcmRww*T=s8x`%ud1V=q&`KPy~IatQR}x7#@vGEi$;-2N1nX=HMt_$Q%OLHqb}%?XGS;-^>yE3KVfRY&}0nT@=<+`Sdoth|15?2u_*8#txeHVH)Ng7vAq2QbL=9J-}o&EWh47&7bBnh2zuIVSXoniJHt?4bYgmJpN`_e3(6g&m_pRgPqd@*i}2G$@@pQ1}>A06rFkY=%}!r&Fx&_1;_a-G^@<n&PIkTCkitrZ#FXN)FocT;^N{6S`%8B%NCD}l=X_QaIbg#&sx2!2YruvKj=dbwQjwWJT#hmU<vPBt>UYhwWit8nkAhzF4syO`X2sk>z!`9>BBE)VN7wb0B(k!qMHp(1pUxDoi-b34E>Fkk3Y!1SJLWrt%%MG>E!$nd9L`1^<Mmj+z$O$*h`(K3IO@S$?4Hi7H{I4NbrIV`V+KM4-Ujft)~^*Mz^CE(6sfs-zb`N&SuqUnP}{K&FuL7ehhE)7LC<0dQCLu+eg4>wBa8Nx3O-sZj>JNasl^54X$r+1N09E4}NvM6uI#wlfG+_h~79LH5M;LqGn<KsZgo4(dp}J?dBL8NbKsXHQB&XuVGBY{WQx%jWeelCt()mo_JU1PR!PtXm8PHI*4oX^P!wbFct}y=B>fl(?lsw9<pslL@axXXh&*)e?67nqh|pS4xa}2A#Y{ZL>sB!Yo$Kg&pz5$aYGUHLL;C#ikpdP8oF@tR)xt%nYz{iju8IE1Muy`L!eV?tw=f1Yv_sg7|#$lMVx01cu5#m)dQmt^y1&*J!&Q%#-}yQzUaJfIZ39m>|2Oh)2L|e2FF%mdHA`6i@b}`YN279uu|OWI^qlY$i&g->w43V9|wFZh*J6Vx~U^v=e<itx7jiA*<&ngIB4;6@H?G7r@v0NGnpg$`zUn+WEuXlMysH?wT&KNps)fuK{#@?x-KunS~}|wZTJ8%n5rQKZqw+9q!GG0D5LlvIyjwLv#UE^BKia@0_snQK&frC#_$}iXIit?(CT9W)kL5_nlpi>+qdD}HEMa|n*n)3F^u)G$6B+DCV-HXLmFcatk^jv&oA-;Y~K}d*+1g!2j8}kA|@b3j3Y%vK#F-cq=5fm>Xo{oIZ>j4e)NTt3kL%+5UgWh$P>yIBNC#bVbuNXML;4j8m4WgKvqiKcAK1{H~vYBzAagDWh9)Q2f;FEHDd&C$sv6TjTqe(8pYEIM<eH5A$4$+^-y&dZwOUyA-a=vC!2+@PM~bc>y+OY11Yqy1#c1-Nbt0Uo;ty52d%Ax1HyJFJ7QZC%Jxct4=$(THjWMULj>4bBYj+BaNmeXO!1PvAvjzzT78Pce)10mJ^2IeA+(q+>v{`*&^DCK8By*JzGCz?nlmQ6qz9og^bH}|aK=x@ZUO?{Rnh4N8R8oEG}>jvWr|HPStpMXjg*EaUD;)DBMjQm+K-4Yl}7a%iMF>5#n}qZ11yQlk6_+f<|JAOpY^s#kq{w_kc%_Sh69k0_V}_VgMRVyrgI>dHovej)3Pky+w97wpVsQN`(^<HjshU)vlhk?K!qc?h5ph)oSj~ViRw)<PhmY4qyj;<wT`Ld$84qU`^iWKd<VZl)SZ9dq80entp3u`n`QqsLmRx;1=#1d*1XsKeBL2pya4?OnZq^xK3oXHAtP>oy~M`8k&Yf2lw(j8$pl)d1X1s1?Ue~<Rjpn@pMxj9zyJ0{)u+pkn}y)QWWNx2@l)}MTYI@}z=qqcw()?ckLO6XQKAa4yMPkPUB)e1?b@@}(V>xW639bQMxu@B_oT}bF*ZxShW6q|+#wt(EcuZW6p$E1EDMsiXa|5>BS=U;2h2QLhAf%YI(ozO2ZYP2)_g`}(P-Cn`N&zL2rZl#+%n7>W&r?Ia7O@6kdr~w0j(p9=$vob8l1Lqo(h6D-Da)BSPK|eK}3TB1*Ic<&_g^bW}2ln(rGxkX?hj_N3q}|pe&pj+yS?ZP!S3C196)AOHiCV?2F#)qWL07`o&cVH$0}Cs3*u3FgS-O{($sK*a;h}>R`y(C}{e8z-v6%&4@eW&zhoR0#FJPWnTc#M4TJYpXh)YAW-zFe95|9V?2<aL<9*8&BqVD0%TfEwgc$n@W3Aip0-bkI^6?ci1G6m@%eTCCNKy>445#<H?KqIcG!g7ppdnlvJY`(B3i(tg})J4uz<3W15K_p>#~j^Wnw_m`AUZv-$eD{#y{FmrK$J>kP=!Dp;rDhd<A`094B%kwZIZ4k@f)M*DvyGv(xUD5dGG%R7?aNF4+hIkN9jWn>_)s2Q>%16h6!_5m{Koss}bWEJrQ7;#VqBQ0_deD5w;AJMF%+Wm|vh>LwxVfK4I{qln1RtK7Z;bA&~aC0hNZq_+_A=ld-^({3B>pzjRO5dR)_mI502F+795My=_m2+yaB4=2Mi26|xN5Dz1wo2d^tTbLZ;zE52Zz3X79p3Oq;+%^orGdh<@I1Fl+Zo|)S$60*r5W)v!EYKPtUqe@GDFAK$NpgIH_zMmMBE6mP<Db2T1po(RO+*BVZ-x#bGAg_&XDPbHdaYD25v!>I!r?22?iwN~rld0%+9qQu-L1ZGzWAZyCn+eME_|p!L7&NN(7psNmi7u0!Z0Iwuxl{Qj}e<zYQX)uG@{Z(;l7d%9z6--`<Qa)*XY5`(#UUUr<CXqlc?16dKpihD5cQ6=}||&2x)MRMhHG87=BJ7@JOiBSRgRlZMF2aU#ZICm&FjuEZo*body3D+q(8BBASIXT=sC~se-O;xr!bgnKF_NI&K|_%}><42?$~uZG`dkZ6Lt#iiy~Dqp26lHM)70g!M%!Pu9Uo><1G<!_(7?L;<5$5bx5i-B+>RT6qXf!i>1RjL5>J1#hhgJYb~*muiT714iQSXAl{mwpvB3RuQQckT__iq4!0?V`xEN5divHYmv^;8@dX(H3BkR!N;PBwyxFI8?`PIqC0vkh$*im#+qCiCF<)&gyThcE2>BbI5R%gsfr4VaDi3u1_BMnf}u5Qb_ne%2PChfPu?gs4<hpfGoqRcd{4r)O+G<c8?aB036F_I2>>BO4Qy2fgzD(VAh5z#83|<gEtno9E61d7-e&25^+0cKCW3kUZA1iJ2xY`-;0=&`7l?rabwjLnz;QWY-mBK+-JQgP%yI@y5fnbGt*B>-*wzrWFM<yf5qA|oWgu&7CB1-7Z%4OqjG2&>h9TS*9$QvhqLy)(jR0a4=KyJNM}iBMeLd(4MMNxQquVs1HiNwUk~1~fKyQ1A1YIO$sNMjsn0mdgd`YaDOKA7(FG@z6jOV-+(6*xI)g>^_V+5*NdYK3!Mrw=sj4u-Sa16$HEEs`V=cPvW8SBrrvW{plOpVNh&B8v6y=N5(Ij=_3>KGZSTgyO{wO{Z6$af&&xkma)VX0_*bsK9@5p@HtthL;63ctmwqFM!eeuP~b0GDrjZ*8XIj9_L1VYozO@dJ7s(J|B8&w7EGehP$aI?pCMh060Qnri3>vd)SUW4p@AvhG;=CV7~kc-%zYlk`b)=)8xa2h%YIy78WCMwj;%$cD$4SufkCh0b2zD+tzd4>Dyz{@Q_xH@@klPjqkLJT=8ce33|f>B+gVQ9sA~YdV7aC+tVnF`_k1D)*>Ha-T4t3GTdP(*r1)cuyH!5`358QWv&>aw#ES#QF*%)QbJI$hTulMTZu`+Y0@dzlxvez5F&F#vPP&=EOd0!#6>8jEWXo_(?j*N5ifC&;ozGqfvaN1IyAY88JkZadfJR7Sf=c<d{h8srRw97(O#2@l|}UWaf%~3$+{c8U3Zx)~s*XzZ!DkeTDBTHAd2~%RVpHxFLb;c64TlPoJWxlO`zI+D_OoccOE`f^}*DxX=LfYbwBT;L+h$MD9^qVXaZCYi-cG5RHro?;sx_;QO26J4g#IZ35~qc+(LANsz4gOuUA1C85KN<5YnSImr;XG)7|InwY9D=BCwHxWehG=w_)<R%*>hj3-o%AAVUuWz$E!pR8+*Vp#+B&hG*cB-w_l0rdz7Gr?^x2_Cb~7Ea_S>Lzd`WFbf*bS|-r@W0JiE5~GL$H)!$^de7?u*n3eP()u;oeTAVFd@7F^usea#=#R|7&m&u)sAq6)JHx@R?L}UYaUMeG6)MicVCT>@|LK#+kwQY{Z{X@wZE?WYTr-$*S@-;4gm6zhS^C+Z@!mU5_fL9+*%Mb2=}kIJT^fX`wJK{JpUpu$bps0^?0JO%k)J~+2W!l0MNG<PY-V?Gb1tDgl`w#QXWcdM3}l8D}DjHYq&cxi`r4A12fl+=m=5D1<q-TJ_BEs#?a-A*r!_a9$(q+t_Wm2qmI1>EX6!mttp&&gsCsNsMVHzb3Qf-DAh)}QjI^2+%UVW8g-l&R$@V9z4>!)pmCUPEVW@!)FyJ!C|JjUq$p%V+awSKEej#hA9hO)>t-WQ^s5$09&SpAMP&m@5}_rKuz7G_B%lE`B-PXN;iAo@`T*=`r9>p^{6G&`kUQg(`GJ7Sop5TE&iAx>Lg>t&Q0qyWJOTkRxW>|)IOZX#PP|ErO0m+`TUwt0#`PFeZMi+V$Dt#q9#0b7)F7wc@I|a5CXh1q4vp~e)0GFg;9Bm=x=*5xQy4!LLaWUf3(_(kezs+_s5ac&Q-~}cO~GruT$yxzV;K##-d89SA!XDug!+Eh6i}uiJd6P06Y3CP+^Ce@kne5XAgw_GPn>Rmu<_nOm<pjH0hjR+bI-fWl056_-ULC#8fZDGlF*d$D_aKeSggNb%z+J>)mkm^CL|D-Tm<P361oUK-Ch2Wtf@DE$@$LV;_g4TST)ODCB>@+e)li}VI811#U{;q?Gd+A7&u{W9+7+UIdS-|sqA1B@yRQU`H`^KLchdJ2uGBBz`-QIA+(Mg`b#=&%JT$`9H;5~7pmA2jSOdxdt4HAuAQTEYOs|$WwN7HgN`ClEq6wlVyb`#02Y{{%xz;m2jCIeST2^?FXr$pT`BrE=sAR4z;o*h4BbHK-$4|h2o>R-Aqa|unMiTIRK5+%)>$DSg}^uHv+duqIP|p7Eoya;Ph(rJ=!n6`O09t`6Yn<e%eVW!P8FOnXnzu15F!^%UDwJ!9JUaIR!ub5(xcC)RD2x+9tZ<fv^IKxBrDPfEv{Fsm8ya-Va<f2)z#{Ce9OXD;n2<ceA{H6!e|i1hT20L{RQ>%okbSVC+c;W!0d>pK9IvA2vKAqy9%(<1z(1Oz-hZX40tO5VSjCd{fP3kqqXmK=Dp?UGA#4&R1euE>ulf*wGJvjAEBQ&y6FD1gFwJx7Rep3&&B~qx@4sal~N#f-zsHn|KNi*O+Uh3u)pXA2Zqt6abKfx-%(q~)<(e0zf50IuWHXV^rfPXkdmG!phF0~v*;h6#yl1PuNjdF(ZQ%e6wCtX@<rHRC#SMnyN5FsA`Vlu6;}JVJ77G8R&Y$v6-N#QYl5gX0UJId^d~r6QECqH*ZAl4+zn`csM0~NGtKxl>@yuLfOSP|XsjdhivwA-o_MNR7c=VvcK{n)RIm+VCs?a%yG{9q8gMM~Z)<>EGeOJaG7d34yAMO&vr(|DM(2-6JlwJTE-bhp6gB}m*6hu#$VleoHLnR*A%9W0aejx5AcF;j*v(-jFH8C&LX*(ePDE{8_%xY3A}Aine7FSdfTSz4eZ3S3dj#2lvtmJGj(*&2pA0PdEPvfipSi?&yMPJ~;mn132b|=o5Az^*W)=g#pH1&&Hpck6@G(MbHQFpDOSt#(^I<*TI`m{RPc){AA}R}XIAM!|k~16*F1iu;MmYWOP0S0kz(4yMFoWN*PM<OiPznbV8av!MN9-e$&?`KR8*4(#Vz3yYSo~8ImOUMj0ML5WWLPRhGb<X-Eo#m~b9ApYFa@;ov<%i8K_8{c$bv2%R;e(8nLX=vjgfr<kir8E!YApfE^@|wS*ZyI1wj1^0-p#p=xVPSaBfC8b;l?Z1hi|KR^$>N`&nn&ATewS2PVO;sSW`_IXFt>OiGuOJ>tnFFv^-b)=<$&TIp3>*-PzS_i;B7VNus(h|IS;V^RbZZtO5?h*M(0JxP9&tpOp!2YOo`yRn7iL_r4tV`C?e8J{x|ku3^B$MLdQwDBTxB$kl?RT^O2w}Tb~9vOHT^qs@J53&4Z;7lrzutq~NYgMPajkq}zVM7Shk#lq;azWd0z?y-lJg{;mqVIKIx!p&M-YMEx%)4oTC1}7JFh8|&9Z5QM2&FN6JxKj#8?M&*<!ge$&sVOQXsw9$HjHPz0p4T?cPQToc-k}Cg7iahjX+tk1<db@ZiB;lScp?{xfI#{1}wZ>wg=*2Z05;J#CSCa`%LjE>U;s36TZtyevFk3NwgnX6ba5!He*eM2(QEZn7XMlfI`_+ojt>SXh*t_WYU^hM7==JpWwU_R%eVg=t}AsWeptoHZ=rwn~F7{3bx`W=XwldvJIx{v&_1Agf+PlAA7{75H@M;nH2g79{n7hq!3Zb!=92+i;8lL#MdLGsqlA5Ca0t$Rk1(oOl2zUIhz;@w_z>l<3X0ZiiB#^4Co(QKi%2iC<qZo1WN{mUV0zMz{pUsKXOtTim70>#L-OyW<cP?xF+Bt$ko6(2u@LG9tXa$7C)iJtLw3ioh^2`vTK6w=Zzds2_Y!9&`?w6)m&oor7<E)5>M#Pp+OocB*zzWF{;!%`C>_Wl9{g*YW+HxshLx!vGSsJR9bm9n#(62RbxACyzV_smkJ99(>uF|-Otj-)_gcKnL2#0=FGzK$MeeM#MOJHTm7{WSw6TLy((SCQ(OAt=tb>zzPs0d-}pE$H5={81mgLd%iD`DqlMGQnYXRGPNlapW6mUZTSq7Fg~^(>^*D7LzU}Q@HuF*c-DEbGzR(KWs|&_qe(!AI=_oK4$Rt<&FOiGTV<a9w3#_dbc0bBTll@*N{aSvCyg%>dkKfk|58Ll2GnJLd^ya}{#NT_2r31S&d#&C6_RDx~qPO+Dn$Mr^?}XOUf$68YZ04YQ`4Bcw*SiO?*72==o3BsDCaSB?*_Gmc@p0xNQqm&+wacT+@rA9?@t0=ncs^Aa-MwAObk|Qh)zHQ5foX<!!}I;j$MM_dx*1z**FV$GC+DliXnuD&b=7^l2tU0fPcGi48uL5R^43l>kukN6(A0Ud^SQqJsy)A-ZzMy>tJBNa;>o4iD8KgSCeMFWb{3W|!aM%NOfi1+T%3u&zU$${@kS<-e>*yxSUov<ugxF4A4MZ`%}nA~A^v*$nu>fL#kBoeVs3x=G_nzyDz=szd-H3Hqm5*GVPbD_HPt()Pt~j2Te0cY$NS9v$H}ja>07^^F0b^`Z=usz_IAHqh*ghM`P%Avx|Pl}5*Js?o9{DvzH~OVvfQ4yoD99LRn8+T{`Kcb>m?Km-5cAr^u^m|Z0oY!T&&hA#kcHA`d6+Jn(yBGPgn2EbZ?>>iq4fEipi^a<H)FNJoVe@=GB7Pt{Kzir|AQKawCv@%|A3RGvUMK(Pn$^ZfEbdluF!hZ)6_y(bcE@?W>0Q`D(@rug7NU^*DaIGxf2v7_a6g%M+>d_~l~qB60c@IBc9&?kbz<&g9$a<Lc(;>Wg`B|M+fPynUYS@82e;R${Mb{?gQL-aM`yhORoX_5FiVBC}d=O<a{u=jy)>Z_AU%=dWky$9mX*w6${he4g8!SdP3!TfI4ReY%@ZXxsC<cZ(~d)935k+4oX^u^Em3+FsncI@#X&71GaFc9XGWcw%`ouyr^+H)BlSUpzHVv{tS9wE0?ExV^tv*qW|(KIa!c^uU7dFKu+^ceMRzVt3D9ulx$^$9J2Rh3)wGOg(U#4&+PieD|_^xb>v3^$uIpujSra!>I0#>y?$|=VWJk`{ANen?LfuZW)!kosX6C(fZc$!}Fy-78s8f=Mw!~?kzKsnO^IRZXHw>H}X&O#~+bYvUPiwet$FS)1Q-D3$c@@KxKKimOa_2ue={WZmyoc8Qs|0LZUJm*m-&EeM~>*_a6_tm7Q0kGw<*D>&5EwL@jo<{;+WyIBRU*z9wsZy-+#`Y-fw9gLrefbk#e$N>?AM+Uwow{Pg8?=~p~|k$k)w9}OL?9=2arx|`*Vmq2lD^rUG_e>~JqSI?u-wQwiA^0IwVTVGEki~UzUx^^{v{Fv%T*OzzKdd6!dI<-F4*n0UjezdW6aF?G>1x{ZcV!efMrZ6$u+3h4VlZR7VbtBTeO&(?Dwi4CuL0!M!DSU2a+l7zbWaz205Zhlnj*ea|gx;@K6L*^n>2xd|+g<aU+eh8qyI=mZowd+fWo~BFSec4UEsk&XYTFNmqsxu^+(GGhu9UcV+?+_S8sWYA<$f#B+pC;xbf;tK+x@%6Zuu=!4xPU*pFZCHdQHzAfA%f|C4KMl`LdYoC2ALY-FSEA=)JQMxQm;KlhLc&%Z3)-Ip3-tZRe}8`fKbZXBM@IME<QYm6{%%-;Zj?>APBZbmC&6_tM>L?T4!emA6)6Y{W|cJciRhk3srpdOhsqGdKsc|1TV7{8!NT7vwdBf2}BF_zN<9{$=$OlA4BRWBl<jX;z6M4E=?<KF?IEj;1_;%QO03J0`?(3fT^os);Ygpng9^V9DrVHDguBVI<*^-_7zjbcVm5h@Zx@pCL*@XQl)!9zgiH%&L>MdiNsH>QbaFd@D#gi`!Z=VP`8^qgL;K)niFS!+*+U$%dqs63RK5FZeCd%@n3bze93e;1vMw%`c6Jdl%%0r#>Q*BP?!-2ARi0;;mm!`sS1|J@o50EWNQkX`o`F0dZGJ*ofXhUn*$y@Dc5}u)E|v<R<2!vB2gFNu;gC^AJ~-XUrbMJ1V)yuy|Me1)>!uu|yu8p(FO-U+e5bQ*X!7rIGJO1`mu1reGQ_Bv@X!up_Nn>9N}?P`e)&AS5S5ViIIj3>(pZP+KWEvy#E(WJwk!!*3nAikSYy@)C<Wy~K`o(tD>z>X31}W{#|P#>E9p<&@@xG#$luQ{kKmdV8Qt)IbWT_5vbSYo-gG@y(4$7Ja6FwMm7ZLy}``y7(U3M!F{7adkQ#I+kA;8uKPg3ReR2nl;B>-sG9n|8GXMi(+~;3UfMu7c2(n2(+9bU6U_*J4I5Hbrq|(tXNs^<Z`OaZ%(f!H9V=8hz)|wVkzz;Yw2`2bnp=4zFaJY)x9RM<%4R$XKILj=@=L}XCg$W+ynm`OoJ&oD;uZ}+7Dr@wq5~kH0Y<9w*m7<=m0p)S)VUW4i<AV(7;(s*ARDsK0GU~Vwh_PJT?G~L$t*q7fufk2SoH2gb&0$PfDTTl19Ux{?Bpn1P{y~Oq(G)h-lAR4JWgw7I*1uw&-`vL{;r6lbo@hAr1-K<AIlvDqzSKSRgBN+3bTOo}NAot*3o<sYY_}QAQmMa~7-!`lkNUk>^O$Y$|s4H+R$KnS?XOtCzy2SHq_cx3n#Y=FyGX9500DG;N+DtOA?HMAwjIhi8bVF9;j(X6S5W-*uoct6EF<H?@XN)Bd6JT|xv)tS7;;Lt!XDF(Klgg&zx-3AR%wSuq~zF2qCfjP^Vf96t^yu&J%p5aF~OCj4Yj?7m$?#{?X(uofNhD|iq=ChMY^k{SOkea3#lb<xHpX2a+J@O5>;-T;24)6>)tC9aEB=~36vaI`j`Tw0<Jy7d6XyuD9D>r`jq6@|BRTE@LgFZ3qT^5_-`dvpx@Np!H#-LWe$7?*Phx)I-3N(#BVC`BZ=4ud|}!+NdQu5}v}*&vLsZuAP3ZnMOGGY2fKh^|xmhWzFpl`5ZIUI2xAsxFHk6l_q9R=w8gmI1NFFQV<ixAx!cY=~2%v$Y`<^CZUoW`vQaRbvRV!MOtHJf%_P!!Ax$0)c(1BCLj(YDhIzaSRKHA;qpq>I^tTVg)qms)3INEQuI#_z)gOiTsi}|JWe$XT)ws7Oo8Go%8BYyr#Ujgivxuty-gnC?|=SCESoWeI~%DmL}%z#`I_`RBKi!#iCIJka&;HB;a&j6EA$%@SI^YXtMlfG#$eivf_xTWN~ZnR5*&b1&^dRQr;Z#OFZAYa8_ubAJkM6*)7)nscUou-+refv5Hzbz8w&Uu7lOyKJYN32>d}nPUMcq;P1qXNM1)s{5VHQ%A2>)i?T)~WyBIxYWLl?=Fnygsqc}h9}TJNA>*SJnQSuRrrD-6h8rH~?G5bv5S_=Ppnv%13%HsPx^^hCQ?6A#7@FJ(>T`n5+z#!AMkRyJ2LiHQjbXeaquoEEt@6tt`deNsY`@=Ip~dfTm1<B>gz7NlSD~23y=-u_c1QFiW(%^f$M#`&{TrrRA|<s=NF<)P73_u^{pFfZ+;ExdxQ8LexQ90p9*XY%VFLP_B#~o3Audo}bZF9H-77dhyn{TcXdJN6cn-wT6Y(j{5))K$B~m{f?{7La$dnU-sIC$F-oTL{OsHH+b(#%p<tNaZk9e;O;F@PkPav&q=&~1Ii(70HyVtc<kYPwH5BfsXyGqcU*<zZ>doC$k?p9Q~l=M6@(&JY*VrV7IRGn8=468(qW+79GU(8*?qyU;pr3M;Z)DlEt!Ni9@;3kt^V`Tu#QsSI1A$6|m0k22^y^T;zs~74u#Hp4wPd{&FDOvU{MB=}v`lLF`@2M+CWK|?fS1BtnSxwR}<iJ?+KnYvoR`T&{iT2&N8UdW@RYyJ4C09jSSh#}#685MrLD>QMvOwD5Bd{Gq?EOL{LC}PbX*CM*<=A4-Sz0tMEZ9*~rJ=F6anaS_f#od995ab)z`@3EI-)&&bf==*hri|{*~p~6IiV%??k_V7Q}0g=|4U{@-;9;Yi|5aCvDc01^4)Fv>7djz=RZE5CgN59{r%xyWUo_+_v$P9S$rx|Fz*(pd#ma5*y@#5k1Z#&vFLWT6FRzG*f{F%29B<ptFJTB<Yc+}{I>jlzI)^^r#I$$@1O09!uUmTVr_A9wEuc=<X?N+>%Q%;E-b9(juPj;4r=ew*2ZYy&Ujz!^gkv$TW_tB{`S$`Eo~f>Qnz>M<IlDcy-Yp4>}%chLizUfbSm_DcJ<yrn7GfLEUaI312fTHVY7c6h-_EtqmkRS+^>_3iBvt5sO088%KP>8!@JP^UUVzA(>|`eT@;JOtz9F1{(5$Mbz9Ij?@PaScI(Z%(^mLbcDxZgsovdZ_Bxm6=V#H?@`itLy)>bpe;Vf1;&d!I89I!No`v^2zdj?kr?IPp^X7iLb~!pZdbbzLO<k=vPLgYDldpxw+OOjJOYL$wllUB+Djei?&ttm}iN$Ct|5~`5UOd`-oIBhvy`QeH`P<3$`OMkwM5iCAHa4Q`tMBnl^|G2@?7oB!s@3uHU-#{;@$$xQ&WzUHwRk7_a53>Vm%92@pB#VOOgwG8zCX2g-&Q8Fhm*(i$pe2l`aYWdym;I_pTA7zuhOmC#e=ELTmG=R5<6WD%zwt)tC7~i+EZrpVP-OPvAfV3oqw-HVnt)86Gs>I;@Qs1<oarHVgI3gclP<by;xi5@5S58t%q)++1ky_l-@7ni|^(7%2fPx>iKYI>tcQU<7DmWQ}5M!)lj??Z|mFBpY7F|+)lRnm>P||oEZ5z{kE~be-gegJylB0MkQ9*ZM|G9KYwN~XBH>5m5G_H{lZ(IHDMb4h3Z1#@+!Z(>%ZJGjP2s#XtmZ4oGfg{WA)wR&%)?rJlFeZ?-kEa*HT9Rsdc{`iJ0ZR#N6SGKYN+G8a=!?%f^lO)P(<GZgFvXYybY@;Iz6`E>(8*v(rHRq<^?Db7J(~!i|@g!sNr{cqee0+kBck>r6eJ9w(Zm)wQdW!qLTBv3&Y`oG72&u5BzA6OZlW;gP<6wAI}2wqHv|t{*+PzniYT#?J4|z{$hJV`6VRwWigsGNrxb<83DPa5<e1KmJ<lln;_ur&rp=aeQig{q62DyAqCuVk@1!c)B=!9zNFU)#GBjIGxiEqvxg0_WWwoKX(vL)z|f>sk`MDLw`Q1ea7CjtB3b|{^fJ(;9T2SEMKflZB0CGTz2)fqmOLzF1}Lf#R{99{Cj`;an9_9C!>1x`Tccm@1l1XspLmX`|r24(q4OG_w;@<nfIT3tkkxrBKL>4zb;>QTCJ(ox*6~8caG2Iem%wNt>^e;wDkJ&e4BVUH?LArZD-#<ZTj!C#rXEdLvu2^zIgX}xH`R=+kB|EMjz`RM-Qp(^NCU*i9UuFa(DN-S>Kw7q!KUPwe*>Be_B1PmUpi%CJM`&T6nY^n7`=mykE6%pVnS)56kThy-@soYc8L(6UD{d=aaVn67Npdo=e*FTkpBzH|mYitHoIVrMLXJcDVJ?Ug`GF3Rm~>!-cb{t$FQy{&uuC_n2%|w!({N#%^WOpNKtam6yBX!SOGDc_VRu*(=v7Q}1h6^O4v0Mmc`9_BpCw^gg?Dz2a*#wO`64^rO+uO#A)xVkL5X(7RmTdfTZTzK5#Mn}?56Gq>%9{oHcnYT_V~`>3?1_X5xLW@ll~7`?Agy_}cQAKCLQ1ki=khn?ihL36V7`BA=mIarxH&z&aDZ=;E~!+zSBOU&$iOhhu_v*cW9+%Us4oowi-l)u`Y*vTX+8#CRL?ZRYorxlvnX<vQ58Aq#k3)7dYiyv>TUUK<xx|F<pX-~Y~)~bn_&uZY|>FsIl>AX5xZ)~?Z&ByKZ=xV8Sm)65q?QFa>u~s=5Z`91iaABh#eS5#JlwVJ0>hI}~xBBOEuUTkcEPhN)MH|M1);Mfs>jzWj*7VwgF_CT_9X+gU$IR)&Ux&|kd#n0qBhWpmXfOAL@qV~>R^C6qJYGw!7oxWl`}Z^DPB=1s-#I)n7FQ1&FFTKC<H?iB<o0H%u)I8am%DvQ?=0*s<c#UH-QGd!!l)#}_4T=}xAa?}bGdS~alR8xUtH)$K6ZTCx;r^3?tT{Us_W-Z)A7n@>LI;1`LWV^J=uJS?mZ`#W69cshXWtFRs}~qTvQudva;EO!q#ajXB8ud$D0tB#2#a3B|Zvw19X=7mWz?Fg5R{?pZE8(W}eHOq)#)$RdUz*?57wX=vmd>t;>f1)67{a{p|_hn$2IG0_`OEJvPe!^?Uq}^n2vBSHwb+0yG8`X>swL3|iyb+$a&EeBX`oVB=rpMtQLDXWb|x@vzd5GAQkt`t(Uf?X2e984aTr_p&57vz(;4zS+ygKjL(W=K6InKS{j)eNVstn;v}#vO{?EIWB2QIqPvs`RSVRMI_%nOOXCu-#u%jYxbp|^WI}D#s4AiJ<rm~f7W-;2~F+*FvxX}HXOoRiFs#<nWX;fuqJ~<<+(P1#tz}UX8Ef*ebl&{gPoKFT+-owWvl<Ko@so3uC8gUg~K;ZeDQz5ISpeN_IELQTDvU7pVRzC<eOhh?bAWcJFGn!$7)Ugj#yc*AWK5$#X(@N$1<st{Me1vY(?GdJR>#(cO)@uGjDd1J<V^6?N-$74zoGljz6S9)B@Bhkn`rg3qW(bp{qu>Et(PusfMRxHkx3w=#z#dcLBaUp^W*w*nxS&*yL`U->(Zfdm@S9Zme%);gQ&Qhv@v<{5-S-c#Jw~BoTmoCEGKWc+vm6Ci3xE!WX0f&#XRQs^dPm2Ks!^H~+O7=)YN){U@Vx2zjWs5p--RQu{$B$XmwOA<C=U<ar=GRHU~JV+r?8>E4-o<y$FEm|X6bUquk%5$!hhQWs|(?KUfzOXIoaR#dfHo#mw~jsz!z@|#e=vrC6mNDl~mf(|;IcN3hFNZWv%_Q{#tjvlL^#9gv1P=vV{m;j0;E3uM1IWv+dOWI4X#FF+nqX(|eK(8^DXvA9Q4`hcD{Y4yg+-X7(Td6RT{=_fW#^%+U!$N(o@J~#zLZM(~C?eCsx+4idOU``Wrj^v>dJKr9kPXMlj@Y}nvKFijXE8=-Dr~LRAU1k==dhlRV-ke}E9Ctl=cpDko>5jeI*%A>xDzexv<9O{zNC<7Ao(Ym(cpwOTC*>c5E<wM*j9Q*@kom)_6P(6jy4ixz27n}8Sj=uWTJ?}<|R`}>tv0%kPv483i>|yfr#m@zxXW&Emq_oay{ik>R#!Y^FE<NA4K;;c{Fc3b|E+Yc3zpM48?!UINL<THjx`g&R;%0=xmmgPNY}NlW!2G`D_%H`OT3c$?7v;_iZClVbkg^rp?|kh?-Qdy+UeV$jgQ#KV56r`@eiRUlL6O*TKPfz-IYOtpbVNAg!AeXD7p95EusLYa(gjY{^{d(Vx}IWyd==+9=J<3LO6e9dya_L_bI}JW+0JnA}dZpDnACQ~qlkMx&rVYo&p~!#3le#nQ*;J;jYbC<qOcut7Gh{p$r3Y?HI4c~C9kYi0^67e*sLAQ%28$vfAW?vuDMLhAATL>@?X!@=^<%^yiq;Lz(@6HzO0J4Oe4orC&R&`t3*f>ugeBUVQ;mI*E&N%876giwy4DFId*-(qLkM@B5An^vOE&A4Vrf&A&`_DNUK4*rQ)>JCCfk_dsRWsRRYdT;_f`Vos|7)C4LDZTU+hDFN2$f8s{Q!t7giH#+JxOA#Dw(XTH0N9)~)y}C7hJcUGtUeN>M7c>^cUZo%lw2bOJ7ByGe5dLH+0>v|!m(wbbuPKFLinvWCq$Y=?4^<ZF*XZqs9$8MLTpQdvIs15S#BIwAqfAIWoLJ$N+9|y4Ku!FaDQ}*&~Yrgx5(&xsKk2dFbPS?p$$D+1MUt{iU%f(PKK$4xlsv=N30DqLI*~EL*RE*As1jGW!rVm+e1hmWpdln$Pb9Q+%mvP+1!!`SGkeP9tjfnkC+f8LmVKhV8V4VtSx=dvZksdEq+%r(#o&CVXKAznz~*V%d3$a0LRKm(YXe#EbS-GW`ZkPW@`)o$z0TA*R*{0AeGO;uXHMx%Yv9FhKGUk8kQadO8t;i@Klz9XY*T9XPl6YQC^pbXkN6S%G$_e#AYuffE2v`9W>Iz80dOmcp3%&NyDJiIH)YKGuTMzG!!Z=gl<JahcFU;$57~AR9v|&^wfls>Pdp+M_Q=2TCTn_gcb6COTP8C)C~}DaW1$~nP7j%4dhF*5fuyG5Kpz_$)lqNL5mk3N{nDi_>X@?U@B>C^rk$7#PVHn!%Zx;)B=G&CvX0=JV-*5F}%P|)FoQ+u-&?%h*qRL|I(1=6Ps5n^_Y{gy5ncRm4lsgQ{to~%DtV+oq-A8qqu-AqK*MbcDu7oWBr<57i&^Eg5T=)Ii$1u`<ZmUU{3|<k}zI2{8jWe`lI%$w+sBv<VHF7sYhJqgY)5IoubHLC{p1Q)bJRIR$EAC#s)Y(M(9_!ve^@yeIJ~WgWS7#(3gLNH)T}?ab0VREoG03-M*>zA$6%VYLz1&fWO`$=yu$(!&kxWN-L2VUFnUA&sDn;$*8e@ddZR|z#hS$sl%Ov)c%+{@|1vX#&*M#+L8r}Nu*%MBpPvn4i`3y+X%Q6f|ggu?7Slm=6poqIR~G0sHxb+F=L&Zkbr*j0z)oK!V*AGP)ND)G|n4++_zb4*6>A4jQdvh3ao=de02!hEudNnde+vl)xyN3Ygm*t%@X*JNgm$fgjPo!Te+{qMTdi@Lhf*SHip=kt`3ME0>%Gx&%&VvhJW6p&`HQx@j-?{ifMXC6JC;pB}IJOk1-nd^kXD`LFONKXLOW*2JI3!I0}x7wgRnw;=ma6`Cmv$u71GsK<|LpqgP+-BZ>RYIQoJsWZ$DN@pOdF01v20D0N=(>`dG#S?3w6vk5j{!fq0nTQ_z~XQ?^-P1LPPH0yL|=>iV}mNWrycsMwf<WXOIgvY;b_HVA6FGPh+i#*z+G>8``|Gu|#CqgBj>cT3THZThAM+{<kMUt1Et9`@+*h7dph?Ix_Q8z;2<t!rIfJAQ}q#~?Ui`pimgdBR_fRz%&q@HS$Sqb1l+lZIfo0T;NJH1J|H<jz508CHcLew$f5`biv7lsj0Am1+@?!#7jlu{Eod*u2P;%?enub{A6!9+bwRYFEC=A*+j^WuQs+RVOyx;TZKp_RX4m~@lm1tQHRViz~b*~g_}Il<^V=mn=B&boVWp~7?_B{Q@aU6n9?xz=%1spmz`3lb4^n;m(9lpS`e(1ZkM+}5|GtlCX?pI^lEZ+j(RF_kW!hf4bW0YvqN50mO|)SB$sa?LD>H??hkY-^U>jgEMeB%5Oi@hQaQ2)}@_3;AA{6%ap)z@xrR|1D2_XkJ2|Fkjiy7mCxnq>_gJ9Lw?#v96$x{zZQPiT6-*J9oqG?DS*c`DsOp9o-ZP94~KGdv{v_O-gtfw<EL;e9vV=K^oZUbr1#^fuC`YXfhCxVK7A!P|rMh?zusk4nRn<=Jow)%TY_t{TU(78v*+F$S|>S0i`{RQ$)K5wcDK71$tvG$TiIx;r}e&fY;sj$yF7V1r@DTQI5&#6<HGB(hXShUAxI-gbNBPG%7^!0?~(#_<=KC%_Ry%Cs4?za$A|a4>IzKkB7-2!Trtf@dC^(A((dSh?xSpuLCOzdnS|^b_6TC1ezhhxPU8#cZ`PCF?7?_Dl;}VmNw7@+U`I$awyjD62g9I=<v0$&u9g^i<Y&1C=!9>l3K5*w-HxTGOB3LXVCydISJYT_fMrH7!H1dC!lhkUvmRwtBHzP`J6dvJ+jz(r>ebPn!EAv#c)LYvMjOwvCUNOAd@?xQRBD-?9SCy3MYvj3pi8CS{fLus<AU21M_EbQ}g|iT$Li6<MSdmFboBirCN#QU<9x>%JFni2()WL1Z>`MtrV?F@$Jmzy%fD6b*Dl1d_pZ1g3j{BH3Ul5s>64j)J$erhSt5=fa|J(liOK(IOlWbNHJiF6w6Hr2rcVHqP24ABVk62gGE4CYZ&=f<6ETG0NS;G;?8jWG5?e_+kl`HRIVjF@^wYVf0`36+L-79j;kOf9_>#;%3DgK3@lB!;u;P)Hq%bfvGa`(XHW&Thh^R!ShdVGVv&s`1Vjuf-u^{`5FLgfkgDBY)IcEtTl#j6tFG&ltep&7?NGLk^njfJ7McwFP3yYf9;C;kxkO#d$ai*O&<?1fpU!wfpAj6VI~oBKClwBcTCsO-`{w22GuW3$qLYx<XfCR#$|#FVUy{X$3q9dIWt%S-Dx7c#3~7V<kGj68IKozY^Ua%=0~0>+o#YxyGgy$1Yf~jipZk_g%-UWy7QF(;efv5IK?8M238b)=)+tqekU<y{rm*^O4YaCG?G@IE%g~t(tzKth2OTPphGB*TY+Gv^^*Y*dtwV5mA)G*x2`*%(`>J}qOi<bEAl{m+jaiUQBpS}zJS|4l2<C-YlslGj^be6DSJV;{rhmiQH}1>zjlWD^t6QwsO1`$yMI1>{zekHeZR=VewgmhJ3^3AHpmJXg{i@U|ovM#im*`b%bsY_K(k-^NhSuzBXuLY;uFMV_myAF+)YmoXE}=!D-E0$sKlz?Hi-tOjC45!Q^c4;Cl~%QH)=Fs4k$BdC5`rfX$%um;p(-KT{uZRrN8L+04smTlqHW0chDPc^9THdrjt(YeNzhhS(Zdcxj&mgT<$TX*(|D;uI#vCpRcrTukrIWt8ybx?ENM+NyY@47OsMi6y;`S=z(;OC_#|lcaJZ!p<1Qd+C?1)cosCDMXfTe;WDyiL%o+^Kx8{kLf!Dx3Oej1WdL6-N1Zpp$i@1cphhf_$s{tD^!cz2akA{f``Mls(raO+Fv=Q)UM+#>Pkpgjxu#_Q@sPr5O%Q<+A)?#~}J%Yzk+B}oW?Ozr$7e|?NKC@9cJ>0?h#KYp-gPlXvF9ZEziACv6>VmzAN2ET-r`dd_utiid5%dnz22;llBNog6i7<2G!zC6Fm<`khuNRPo>?Nj6P+<&k2u?d1%t9a;eL7GAQ@cmppbb|L%`G5Gs+%pXiPZ#b#}Tm@M;i;8GK$OOI3Q31rl9lEM0}L_3t=YA5okO*8)dQLCWP+-R?`zZ4sD2I4r+qgLm&25nH3>)vY*AEexs!G&}f6V4f$*ko!&OWAbi{i8T7~IBSGH+x+Bo97}Vq-KjY|Ug#28<K1;0i0+g)5&l5pkl>A!cBP^0`{0g|&O8vj-($BjUbUX%4g-syJ$DFgK0uzk#;caU(b#yh9<q_k2YVeDWy?{OwBfrqt@!_Vc96iQHUkLMGum<ugN}RpOuNVg*{I$qGKSzGe2Ym^*g`s<~ZToU!k&Ur9&%v`8V><%B68zU9|McPlUN8LOy9d7(hl7D)BJlYIJ`R}(I<yoGJ_u?aWl!=>YA_cc!ERw_r2@I5WF2SbB<Gf26^sDZf!PJpMrb;s({MT{C$IpOX361%lS{_KClFCru!#6NJp)@UOmx`blyTA!^cB09E}+rSA#cEaVFK+$og0`O;G@@o^N%<f{Arz(N@;@|9p`lJL0Cf50ADt^iO}@`=iKPFOFCZ#nik?)W-#e0BX+EaWg|4c{!*%U!M&%bm$WWaF^nbtGR12{V1YW?Bcf?QXE2eP6#(u@3h4gD=lDU7&xa|}#5qZb!{H0_U-Jx+@Lw_hoc3bq7YFJ=dd0{Ip}k`K494N;(C@~1Yxor>zY_Rl(0*`sX^TZle&TTABH>~1mM?XY&yStaIFK2({1{(qOc)w{bDs5sUok%YxMKF4_@O@+*hgS7^ZZwgPe00k#jUflo;t~}Yn7dpNQ9Djc7!f>GW#3URw3QaHd}>dJF%G@tQReGoNEnyz_|Y0dTUdxTQ4vZR&ae?)=TJ;HJYRj9BALgPjZ#K_|QNTRKBrPX!7$q^uXua_JYif`1wGVGHUR#3^4x%iv$+Ss)TT<PBLjB@<`p7fI+9~3PLqKQ--@bS$!{E_zxu%d#p9fG|e~Bs|{!mYbNdC*C4GM0n3b3yl%J<3Rd11UW%JQe56U@VhmEb@~C9{iAaDQenW=Yk0BkhTr#%>#3d1ixxh{UQF;?U2Lc?+RCz>*I4QaIHyWXaZdH7#va1wleQQ5!IHFpRe|(2WXO^uJn+NPEP^fWH%9ZkiRy%1t{O>i1IYZYZ2>$p{KnUkTFu}zYS@A$=Z*h@DY^%elo5>?mznK}MnXjdnHlrVDBJ0APM!&$nu&OG7$)E~?mIJ?4MNp{aXXP^DH3cQWg3C>_P;R3}{z?}Fz90dMHL`p_NF&R?k{F&BqG2&X2hmjNwfky^;sC)Egvx|>1I1hkGm+p!(Z^}iu+`n-J!@?`RIRLo_d*SetgssRa@lAg#xljGgLjmMaex*NbQgLF_~^O66JXTW=wMV_Z2_AI)m70n9+!L!p$J1+D`NRLih2UXWQ6iX{!UoxIK&pC|3SJ2+zXv&#8?tY2eD>M1AScCN0BaM>Q7xAQbE)W<FVVCl}eiyn%zc`sM##vRdP;<EB2y5a|u8T^lUFgebeF?f@@>x(NtN<BMv`9`_95)CA%wpD<JH(Koe=-Ngq+AE0nqpM1<<9bF*RJ-%!vUJb&<1{|$9P#t~kDY6Kj8vEqRbMX?8MIkq#FL3Y8qz;3;cVAySkfP$lDEe+(KS!p5#rbA4A>E80JJ@VDN%T-b9#zK6BeQ;wP7$LD?MZgXn0yH1gotSn&O>WFB>%<uR{|ugjEzTRJgbN{sK<wNUYs4sasv}DAIH-Yo6}4DR3V2^@mVm;b!$8!gSQ&_<cyeI+bg;V7)7v3L)s60bwbkv=V7CzB#-ID-JYw>VkC&_&2ZJdhp@^@cV+l@JX185I91fwmZA9SF^#RvINheBbm>`(y>R3Sy#GERq*-)V71Z4@;w!;d3VM`2>RD8?HNHj#F@`%Jt0`Rb!9`wmncS#4UqU%juqFL2iI>`Y_d9gmIKM4`Qvnt>Z7Ff6(E{Gynp0OWTG?5VOFizqj8c)Ud(ur8XN+ne@h7rqHrEQ^1t8F~s3sYh^{6d~9nHw3gY6{pXJAXCfJlw)NwU93~5aX~#_E^MpIxSY=U9RnJu||s3k^T;?0dau4whO3WM-sn<J7d|Bf7{`>%2R(vW27Q?jIm^#O)oTC&k~qb+Kp+SUs`5J{BS_nh)7uAa4_3ZLL#u~>*!6Qs{;k0g`Fj?WrVf@P{n#Zb_5Vubjc(Pw90|w(pv{VyX;$BT!=|+R9J{GbungZX9po%;bEt!g?&P~dcbe_gCYuGNm)5E7-%+2RWNi|K*1l<Azl{?Rm{Alm2t=nF^b|cu!#BL;ud_%<^<sFi^V<saL+*z#9#9p92C~B_@pxl%MV(4=LT7MMK3dQfNqA)9{jbi`*B~I+&DmaO&@sp=`RSLkTllxffZN41QVX2qthoUd31SYadz3JFdXhpy^7(A0{cKwuS1-&X;7nn2Z=b;x`wwAPtHn4i->sBtmrKDE$ogEBrkbL@Rn9&th+Cbode@|8cOn!z7iSCVVV?c$U8<F<VMH|H0x>0I;5PU(xDK3{9d>(q8Eg8(VN!Ghdf7i7IO}DB+@$UHiVa9?gC(FX)#?6xmV1gJ;{|9)X#d7OE7o3mCLU^6~dlM;UN^m1C_(B3Sw_1v6rIQQ(5exFt&m!JvfOWUFn5vh6Scv_AAGx9tqPbR>96)sk{&DZ4@R+IkAP>y`QW^#DyCsWeiquI)+Xpys|CIj0kT-&|92>^YZ!1gk*gMC;^BImSF7VW>D<#Lwpn{!+)<gyR{~mEwS}e7Y6ERwK{f(AifQz<#B;#LbSI=GlU2#TCl(a`3Sp5?j2z`tO7ecI>U<KU%pz0`Nef0u}Q}O^_L~&0@4SW{s~Q`rO(<yEgi{$0GHBv+MPhf%+>NKb}3o5lJqIvT5LP=$D%O4<bm!LlmDBs?1ZH!j(UPDF~Iv&5Hcj_ilfaWU2`)#bjkL?Y!P%Wsni5|fsJP53QW3~(d5iBK$5L;o}k5avHLK^g7pE+VQFPwrEN6uk(w@b+if(dX@tO9n{Zt%@NzP`CS=qhfnQYA8K*9DRSi6gsK;5kW&)prgnl|~4LH4%ZV@pcCG3J~-4jIfLC5MD-Fn$qHqa%nhe$l8AZKCW0ggzw+0-GafYz?TgzB{xb=@F$)~uXmfCEz6P5}fXe<oR+nCij0gS=AP@m9m+eh3Qgn!?ShtOX?Ayrduy1^5Q&Z!};G!Tv;C*GRnw!f_$s6VEV80>GEBs)7F_CK8BB7InnEF~ty7_cAJtm=XgW6!0jipv}M^053YN)6p9(gf*sKuYfYEj@^=x90Z7HRJf6$Yr>0e6K)M4<y1dTRA$)i&Lnd{DAGfWV+TA|*g3PppWA4T?K0Ov3eZs0Jn~MO=b^SFZt<jqBmN-0Sz*bHbChblVspfZs+QFMx|v!j>A2RP5djKykSxfPQb>X^3*Gn~9d-eGANm?P__$)9Ty$Z#5MAlB*ilOsz}rpjK#WpwTLQomR1DM-1lmyM0LdwphZiInjp*J?(F>CKX>t<=6=J)s7BDAhv32H;$sNGh+$xwLW9t~9z_Y?BwMQ0-DekETz^YH=NN-Q5l#s+RAcwu!u6Kzi2@Ecngpp6>R4asEDLiY37@69@NNpQd^vmvOIz&Y$8&wY}b$!dyjrb4gWQGX>A|Y5`Rmga$ARcwBVIG_@ONiC5j_Wn*e&du~1)8N4UWre00sAx~R>~oNe%QXHj1hBeWl|}OF~(IIX~{b-a+t`?8F@tefPy5e7*TjD4e3|VMYecvS(8)AG__Drzj07<K}{ReK)>3S?%Ad)L4<D)>dV6`><K->q#Qg7iB9mhwVFvHW2or{U7nWvD%t}bu(kppgeV_OF!4RlO+OG=#rV7sCd@4K0Xx5H9cJ!BBhy4PK$Giri}I^DN21juF40m9h0`Jad)0`3kjj4J#8Cz$DczJXaWt01Q7}%0)=T-PQUM2sricHmfqU?Mpv+h7&7I(^+{4vq<fB6dJB;RILnLMd!;6KW)=GtU2v6aR(SlVH_@U>rG|r5@+=*(cBFkwb&&uG?r5lfAP(y&snYo9|;M)Fm_zt9Y8AQkj#-<J(pX!V-vU4v&n9+bpa)-vy5lJpYE!(C2?*@^xek(#P!9f}Re~dqJ9okbU`J*B0c7ADyXmyqR;u@{~MUKR|Q%kC}6Ye4J4wSJ)yZ-=h<1yS%a5f?+z4;nNZnI$y`f!JRoCe2{P%7X?cp4(=1_NRsS`zkC^xcE97_m4@*xQVAP|Oj91=3J0mQgBok#q;vYRH93`~es*g0*5;tRylcHBz$V4&%fQuhl>WrZ9nnKTKPH>cab3rY`Du$%ezGJ#ZDXtQel}NLkVUPcoIX7lULLp5I99<uvgKCN&jB5ME2iP8{?nWFD{aeu6Mapqm!<kIOeHOIJrjui3#P-KbrsinYfC>y09qa!hb?ZW1+4)8H}$C=~Hgh4xsE0FecsPp#eqKLPj}aX(|kfusXLCmLQCwbJA4Po>8YvM~<woM8Yt(d(88yi8%jEZ-!gkceHt3l<^Q0R%5%R|q7DlyL!Hrg=heX!R)obx@Qe9uEJ>WBo9dL?T({rG*38n}Q@sXdHL)BW{~m_5O=^?)7S>4z@Qel~%ylocdpY%8iC9(<ddZh0Z*e8X=%ie<`ti^HjH{ia;c)H6g<S32dC@=^X_JV7HnDUoiY|-wB4&K~*HfP)<V3w@bsKmYjmbKjNKwWKi0XAybZg!&8XmgP?#AD5wQbE5(RqbQX6)a7-g;qslNMG>{XF*j~^aBjN{y5`B*E%GqbWn2gJ|c=@Y0f$!(S;3)%4LX#>O%}t<SA#7z4+;OQ*ywT9hH4?JRtuHuoxZAek{lWY^>&s}RDlyimlRRKiHR$6hSMiBMV^L|3n@)(=Ejfxf---`Wo!YBG$Y;}_gcALaWqQDB`*In$7Q|J|IkH_YNRN4Fc8omU$2<skyXNobqvU-qdz#M_a+#A1nkOW32#3WNas;wJ3gPECu6QmS=D+6nFChjV{ly||=fZK`f<*I^P}#X}=H`^VVT4SP^l!o4#d68Pe}6H9=kI*SipWeX((In1gQ@7FC1v;o7k0}f9PB0`Q?C=Z3YzYt*91EsMQw`8I7<fi%(8N&7ap~q#;U<vPFmF7T)@Z%?IK{FWo3xfMZ=UPQK>9Z74g;sf*OXZfiyCh1dC)fLdoj(PA)@<scK#`o)aT>P7;;G1Bs%LqygL+G2R1wC?o?m2Lo$ati`o3;0>!Z_a_%?@yUEsZM!@p1B70;ol?~%{tz@3q#7J?Xn<MIZzm11C}j0yMb;R+q=L2fsSc1E5Y>iCD-^EQH*RbR_bqn<)MJn@0dJymBp8^`j>>@pbJ<yiZ`F*0TzY!Q%z*)WUN5qRTxxRWa;7i?8bQd<*pjvLSmR>5R)Xrjw2#%>(GIVK2Ted8Xxm;}shg?u>Rf@EdkaSM-oT}*u)$Y|T_>7bT62UFVabBVGHWgQ{E!L-B1(ls4!DH?Nz@9zhr|((dWEHr`M?N4KFEqMP~-DI88H8^@J1LWXkJ-El>+R$)d=yN?qPR9^a_1loiEP;2bUtEPfL}7Qy+}Jg1>=1R^%5z=qQ2(<Zyi$QAAoaG5sENLgfOEsHC~q1)ntT{O%t3FGu~@Mt~@!O2rf_5a6o@Ka+!9DVX^e1CHqZImUy?zNCV4`2TX~>X*%nKbog|{^p+uqo-RyuVuHec%F|@85S>OW*#t(R5wpK7DEY$p{Hz{S5!TnSimcqKAZ?R<|&RpRQSFy^Pxic(MMG=q*T62dfqZZP%Q^MCn(Ms#s2WaN$}@GQkYx5(n&j7gyemhu`kFDUU4V|c7&W;F{4_TI6IPyI_W~`>={F?83cU}QPhDA-eW5-?4WYF^kYf^S3n1QprW`$Aq3GrL*Z2{Objk_E_u&GKBputOn`onXA}`n!Ea{#x3neL30d}j7jiH_=3prdQJRya^cUV*hy{_13$d`&2uCma!p`W+H${>q@<U90go*GHJ@VFb`7EFY@&ji}jL}v*{HM^$c6R77MeS`boC7N&I<3!GNGL4HarUq$m;Ir-2U~{yk!mSI8|QMYm+B9xm!&*k4v^r3VF3aEss0LUX@4ZNM)-pop#N7$1Vok?SQBMn0NUKYB=XENXv{0POIA6_?^ID;QWTn3#Mj&pR80|MpV&7UVRz>?9bIO-EwDQHhtZS;z7%Z5zu+DIBR6x($^oUrxT69M?hfz_TDl(kO;)x`5v}D?FvFqe#ekQWY|9TtU-R83!KLJne^Wu2-;6Tn28d-}baCM)DuKYTgUfylVV4+O{`4OVQ2FWL8Wymk3U(30Tz=lc-d0sWa<?F^7LrQRaa!HcOUn54=j%}nuhSb;1>_qJvBDm57j&oB`G?XQir+$B#YAKug~-VFy@qWl!-}^$@x&2_@dIwV-vBN8IP{{fKkT5eg?!1G^*_RC*p88bf5bx<9WvQPHxl`c60JkH<kF2)x#Rwx+logjhs6TPMI$iIm3r5#y7It`AzPNi+@0B1emX-Y&0J|hkvqU|bh)PXGFVzy39~Hm<&D=I2ih%VQxI6@BsKy;B_P}Ba3%K}$r{)+dOr8aVwNAo)xgIMM;>-On?Hz`4~XM7;LsK4J1Diz|FZIy=1-#8!Q4PQycOWjr@tFODu*SLLnpf9yDEZe*_Rl68ir!LuklhT$_;I2r;*H74X5uO3TJJd-#-Cg%5OmHUJdX)lkMcgIJ3bBc2%w;8$^hMP*e#%M-~c=a-mf#J$75nhnM>!N;+=Q&r8e%1&#-32LzM}TGQtk#d{1Q6840=vH&7oeM1#aRvJ2Nx(E?+au!!GL#S$KF%yLV6R7;FBHOm4dD1S4W;BlB!&%hitX0Aku#XtIpuam6@%aD9#KwI&;?$0WD(qQisnj7Ugq6jC=-oR8385ybArVDwcYqAD?3a*s526tp#xpPHWldkJl4Jq4Xh`gT$edjBwI&;^h<yZ1?1k7Tsf`g=gXkqf?E6p`(cX4PL(>I!Jd#I)mCFEMR{~}jo98A#BeXSBCpnhU9LtcR%%_*{br!qG9e0J<fv;~5QE;O-OH<(U4+{xABC#;Alwr{IQ_44wj<F1K;CUD>hMYLDtQy<vL=pn}2OL*_hx`t9M5P~Bd7KLA)s*)Kf-wW>$PgLZ+?+?H^Tlh2GoMvcAAGHT$+@&}HWw>owzjY}2D?KF%<g*b^f0|$I62DZ$3BFnUv_Q$f_I`^NIX{H>i?={7VZth)G-(MudwWYotgWd-;76f1E)KUlDSJ~vqxh-`boJO0)D@h&7Ne&2D!yh<iP=$!ryq`#68pR?+cJ)1&1tn<?PEtcO3X66nLlEexWZ3{l3?s3=)_JJ8c8eZZtgl^nI<No3lh0v);#YH0sPl)@k~BA{#=46!CcmI^HvATm%ecHPEOISEzQ>1mEo*x_{8!M`EpTp$EKo=tC{)Xa?gzmQs?dMwfX{NbkixtBc)Qy^O#C(NE|f;yN7^E9j97AW#>Nz=0W=yCrZC5XwT~44E{GC7#lw6!XH<Akx@1w4y4dwm9scme!L~;^6beWmOt3Lhe7dWkZ>#L_+Z=OjJ&*l!J^2CL!l+zG)VcioE}%lg^xVKf&1%4eHD^ZX?tVtNGXLN7h%A3f{5HR1R{rWOJD@ru?&P^O6P{4`Nf2IDXxD_1j5j3h7)XbwE4<hU>3(C8U7Evk~qt+g0XN<ESgy7fN+R$qP_Pg6LflqB|AqE;cleyX&9IqX)&#<MUFn^M%-w!m0RY%qYVKq<>GF?dq1v4k%~@qEO~68ebIs%4s@>_0ygGjRIOy{uCj#TfBD|+a*X_d}qq1rml=!kH!~lK=ueoM_<5RF-JS;y;F?sq@4Hf2D(S+xasUUW+S&VnIqev1ESn*Jiza#P*5U;LTMw=!XavG3Dj5#jz3dOn`6;sA$7QOklG*nC-%aHWsk>jvMge($N<-#AWyebha2!`l>E7oN$;HOWO*suxp<ho$Y*i~JBO)#_F^HN7#a5+F_{rkscDcWqSCFiSS}N|2yi4O;bca)4dFUMW>n&^1Cd;b1WxruV#H66j`lCnOg4AYI}p<~7p8L<Mv4F<e_6;0g>F`yA{LxOu5Uc0ih*4%DFaQ0)A+hx>40A9zS@z!o3)~eEoaaGtY*}M8@GhNLH?84L~L!aT}}8Tl-8liG|{))kQTNB{`sKgXd{_hw1OBdLSRrc$DI%sXX(dn!7>2$M>Z6CK)&I4BTdM1TO}+Nby@}IZ6<F)r2$sH$hivW<isziFG)hPbolBw`bX!Pe+4a<C5|+rtRQBAiTc6gF9L&6ILc)=PSg1l;?+_WV<MQmg;tECm1)-{o9_*)l_Y>;8eKk2Tx8uavk;;6&?HAfT9iWoK6APSMf-olME)Bl@-Gh)K^XfxVIo6Dhj>MWh!G*etZ*V;vF?ssq0;Quu~F%UBQu&O#b?WPX9&!;q4rCVW4CP>3(x<B1w@rqt6Y^o06eC!^rIE`<wBvPfw&p8h*N>yycC*{8lGLQB{VadnIQ><nfzXxD2Yo#PXt~(rrJz&&D<6s$um}3KvX{Y)6WBgT{oIoZKz@qE0&1)O2|aOCK41qZIX4ZQ7mh|mVpimBDl~;W;O5|k|cn6!RuwhBE#GQLDCm74(wO>Mlz_CpaMBafY*3pH#UgQwsi=!_v4o=2RR_A06NA5lc7LIjUg!{FgCiR0FUU_M;!jpjA&|P_=U3?v2pE)?!I^$08RnG!oguMHX~8&xe>DJ7<C+ALZ^puz=ZS<Yz~1Dd8)Q;V}k_k-e+v%Fi~4loKfduhy(cG#RR`FmMc~`#=t!QOI+TBZEpf9+a#GGJKY#!0R@UPoDv$)V@3v$c<MhIkJr$-zMkvq@xaTX%3jpg|EEUq#l9WES5s8B{fSbzLx%j-bW<Jl_oaY8GKx#cpE}&nQ=H+!er~~h-n|_We5#;47l!^@5qX2h`cDhTbKd+ew|+xs&Ok2(oA-!A2|}FX1oF>{S+voeUVBB-A}b0g6hfD41M&<vIf7#>sFV&>)bZu-56>d@lm5^0?B(<0j4JwuH1QcE1d6qSNkyRqV$dbKu`*au#cz8BE80rvD1#K0V-69UXd8Rnw+YH1a&zeA5U2vFSAw}j)8?Dl_J<jf5Eg(C-EC=YTx2IGR^<VkowEgIQ*hEF`mWXHFIjmArn{<{!g!J~uMS+Oko1}q`hh%eJ%h{|l85+;wR+tu#{*>qG_d0d4i$6Juw?=uHBSrJy+kOyHtq9cw+lP|S#TrKo6=4@V>Qtu@dF{}e4Ljw;Ao|8pt}`)B49~MK{>k3@hwRO!bbzm=L*Js^vjm0ksJu#)b8blJ4NvAO8bMevKZV$Qf@LcDk3D$xcG+FE}N`)6Di$<rw>KQ3H2i$JUrhNFaC;-9olm+6ZawKdHWtG%EGHdc#_irg?TnSoQjAPiK*+5HypA4EXN#15)I~{BkBN3OHxJJrUfPLI<g)#@!~VOomRIaq~avAMjTP4hL!?8IdFqn1lZ~wauNj55pIPhB~y*M?DJB+E~zRqhJ_kuoDSQiwuWvmcjyd<{XZD=>gY$|jC#54p3qydL8znR>Hm)~TZTWVB_~JrfLTf-+=Fv*gXoQSTrO@jCym1O{`8kUEEnNCtul$WhbiM}ofx38h&{lsER8%UAaDa0Z9JORvtDNhipzH5&>+l<ZD|fKcWVUZGK2>#c8E@t3>sOwk(DxRbZH@o!T_^`yGYtIP%jc#m}IiND4(eW1!)Spf$JMQzXEa7fu5F%jf^0q+Oqf@aYNLK6Q_hsp-4nfA|hkk2LG}k;I<)GsHrExWr?aztSzJW7r6OJ$~+q1$49j^dUX6Hhtncrck~yF_2bZG39te-?oTjIAew-kZl(6uOL)W^)a{x834Y2=aEqlIg;v_^m=1jncZac|yRs2=BSCAu#;IBrx&_Ayz<UxGWKSk(LNR>N(>V_sMno=ktAO$;))3>*O2DTZ<?Q1G7b%>LcqWhwFzg^-C_0MlH$?{wZs&lMxe=>ukP(;FOp^`y{J1gy2!p1?i?Q*YKB?9jyr#_74dc6_lD{WuBpvmtX6k+<Ee6HdGm4_30YZ4o$RB>K(zTA!Dp_N;mJ807Na||nQado2CqVEDD8}uV>a;a|gio!tphQ97v}xW^tU{NGwuSD0I!s24By=t)B;~X7vhjo>SPX@KATCf`aTA;fg>|!q@NEPSOGNmDie;=ZFwo%Yt-o}T)MgUTdDw)G0o*suFpGsPo_do)mE3CX@FvndjE5Bk^4fSOFoB_8&PBSryJB)81CC8;g(d4{-(BXfa2f<s_XKgz^t}t-Sfwq6YtI6wXuwj!-xk~2>9*=<33wog-oxfP`9h>$kuG6Oy9z{twmZ+_=pS-u;r#Mx(iu7<^o1;^KNnZ^wF0dce4999w*rA&0ng5^bC8vR>6vNd6d=!_)ZIdG3OEgO(`^K^`LN-FEtd%9#B7TpM;>6$1HC+p|AH!z!k~QiRD0!EP7r@sqWVH7M~JT~`2%%p7_%#~tZ{}<K}}sq2@a0QHQlh{dkG_<AnpJ*f`Egu(t2J*e-?;y*pkm~Y;()<K$33iPY8@1?7?ICHVF-&lnunAApQ=!^oX-TPyrPQP6lg#Nv=Zl)C&Lja4x!lI8nTCzjfLI9^uXLc!BIjLHf9Se#y|z!u9CZEP-Lp>=qkP%YbO2Oj<OpOX7m4^^w%3wesTQY^0n<4<-N@=#pu|bvG!%;9!<`N;nCGC6XCnJT47Nq7=m1CuLkK!u2qi`GgUP4F}ICASC{vpp^~yNCYL-j~!o@(Ln8&Lgr0Ajz=P0@nh<LZV?o|J&yK*ySlh8tH(0lbcY83T{z4^Tr-XShsV_WT7aUHpA=p)nqcErodvcia9H~{y5uGDS)h^ByPhrK!+{U8AkO)zWf{Z$TH0+wS@_>lJ9%%QC4EAy1C5$#O2xAP-s0X`<ROUZaEM+6{A=tH9oBLlU_%dzDgBX~vTYa*@jBu2#1|IhmAc$KO(z$`9r=_=i$N0}_hnej2^6vqVbdf@5Z(ZyQPbcSAY=q<L!rVNE#O$=el~TzV<|y(v6cq<BG_~=GYPf}P<t}1zmF#3hWbW`zCwH^nqJYWU}Qxtvb@`6Y;9yoxoMqk7BwR!T*u)H6#fp6T%0Xj6*wrkY!uij7f#rW?OP*mp{W@70r$W?lCE6vHgG#w;#8Q6lW@);MNMGHGxEJ)Kg&r<`-vQM%!l`%3h)`=(JKV}$TRywu#Xi}fs6YP&c&|^un77j#|784SYP3kzwBmE1;d-82Rvn6CWRjM9a6Z6v*;CJvl!F^E*bIEjvlJiZDp~3<It(yghp(;ab_DLTnS1*?uNv=3mtAoROR%jjD15?9V+_>KvvG7#}(O%Yza0)jf!Re&FjP#2d$QSzLX)(7GbQanCF!e&N3Z&Y>0qOY)69Dny6-mzVW)?oXM8#Wf0vxV=)8I5Y|QaC6)oS?kkAj#3grXW|du)2%>Otl1r-CCkK%PoEl<*<F;PcpP{B08s3D`#Vy9dqS19QV`CdGE=Go!Sdf;i`%!<H<q31}?6HX&A9)t|F05Y=TLKVQC(14Mx~L2e-_kUU2zL}unI}wPOmfqt?eL0Z4vuSn@N{8MQ6q_c$1){;i+@_H;2)P%_fDaezebaxeUxfcHQ8sdkV5vpDn4OVFBgI-+KQ_bB;=sLp9pe<xMyFMGDj3Lg2OiObF$p9@|>)Scep}3c=?s&APK+`g(<ZqGJk++>D}_X5wn3s&G3wgq2<N~8)14fZI%pe_MkGF`HB;wq3vzbD2zd*%erx7?PZv{<VBWoZ>K#3+CX1wQe<XEVZA}N40%=-2gc2U$~}-v+hC7t)q(QJgeChoUSka(*UAA;C28fXku3$FFoN`nn?D*I>+XY&7c2Rii(BET03i28k~EyL6(R=H3ad>dRQK_*fmcMCXvhE;f)>nRiC0YU#ZCx#mdm71JWlZcc9J3GZ{k}+EuGdX2pthx$R55<eXFH%mU0>i#?bGn@s+2>QtM;TctnXSV?UHSZe=Kkm_C#skvbB#DU{71m?>$7c?!v8M@+j*bSmEwoc5JRFuUlR=`wDHLn{)SVDqYw9R)Zd$sQE;>?h7Fqjd9B##;_c#Ac9YYJ07H{e|afxB$NJ1!6Me2~^qFNJNmSHd{MuqD!0}C^ncWRXrW=tT2OOeU@D9@Xl@C*x6F#NAO+<ogoQ{G8IPa0hq-S3R5xVo1gzGfLIKr>(?kxi}g{~2t81)A#&a^2un*6q6gfEQ7@V#(Y^8P<-S}_m~l?maKN>Z^6U*gf@^Qru+1+-wS6bD?MvBwJ;>W{uio$GZR2UDcU{lWv|DvY1coDF*Vd3OcsC50!k!twk*?T%cIw>41LyX;NAzn)Q#}n=RB{?(Tu311E`Kl?$B2+`IwY%7lD<e}X~fR8Vcu7rz#k$XEY8<k3jVVkHMwdFCgu$wew1q|6b^mD6;0J|j)eVu@wAC)N0MQ)n7dL1CXWd<1FS{-c0v<pN=0QF6N1o#zBYUW@(?ulD+J=}yr`2@{7m{9S7SeM@dpebP@-`eyyP)>L$XIS1K<9BCY>(`RD5aVtIUePTk~wWoSfO*0;%Vm9Mojju3{eu3UF!4kk+38m1Sj8PjSg>OfhQftJx4J1L?vEI6iB~#-JQUL9+@b6&Vnvm<F19XUR8~+1kR@+_UHK*TK&Io|Vi8Jvh&1_s2-a3^I`TMFCToj|NYa?2!*~PuE*QC<)l<LO`OcMjf<$#J*Z1;Y$$~lTt>@b$Wt)%{D~gjXN#-BEOlu`#TA)-$WnqC7+AGl|<HY<pQqe8OqOV7IICP)Mr5zEc^4yB*+n5bE+VQS)ipBsWruUN{#r`EOwE(pb5DG20I1~aqrW@Jzu!@fBMcRWY@GjfDlWLz-lY=P49bdShMbUdZ|m4S5lB!6<8Eh@4JOpgqwih!$a0Z;Q<8JjB&OUfv^s^j~t1O;7W0~Q&I@?JRi*aUHWQ3^emF@4N~!uS{W_z!dXX!>>97^tr&{Z`gCSYl?qRhZjDfGO0uoIYhIuJL7!M!(*jaw1AelgdiuE7?VA#g+PqbkTuKZq`{o89m->q3*{W@1h2^+roFGm+y`-{Hkd<6B5M%cf@ePIrdmALBksN`{WJ{&Qhc;hTA){VEJr4}JnWXJHFLk<5Ool>RCan;(53aa4jQ~d6aKIkJ$Z4#6oFMC$v`X>9pk-H?O4+T(0!iB{x&kB6uxnrjza~T*M@$9bf1HTZ<0qD2urO8zNk|4l2si>;I&(TzFV8VK^@DYZ4l0r0ObbE%gEU0O&`WetVVtywd=&uWQtSH}WzZKud&!44O_U~Ri7Na_X;%q)(;Ri;;@Q?>{{wMKg4MUq%DOo5nvHKYw$;`MlN2rTWI}Arfbw)Av}E4iwpth7^)8ghUZ`y%=h%1V*|+lX56Z7UFvq?~HDn5m9s?ZQGR?^O_GOC*`XXPAYnT&jK{D`T_gmo!tIeh+K;FJGE}8gc!f@l|ORFpkW|2@B2`0@DPBBe-P(w-QE4O28DW9mbNs8ri`<u!-h)R$kcO+fu^$Mi4GfiBzg2)O0>{3<NS`f)@_>jaxbq6f|oy<3c{vU_z5GIlaxBGl2pw{%24U&$F5GlgVb790rha_eL&CZKXsL|L@y^{%cDy?v03z_*7{X)Nj@q?}{FPtAHp%hEtM2`_Cdd`W;L{R7_(8Axz0);OmI{?LvlF`qFxVf_-4*i%1#dB4-3xznGpAZV0NS!<m*q}q^)bd&k<H*-(tgEHuOUa18gp;EF%?7Sz_jKkrjId;p?1L{+eE;DSZgz~)BOX(GHfm-30y8C0NI`$8YF!iOiqYzIqX%?dqhp~zLxkC(^$vjY=r4K+d<xJxg+wiF9oP0T5#7QH4=AvswTrb*TSKC--n_3hnS))q2I)c2EIJ4S%T0)jN0ey;C>bGrR(eYxh;fLX1<{$*TiEZNdO-U$(c>LlgCUy?qDP+ei%MJrNEqekzE2dsMG=0G98meM;N+8S09YEv@#HH<+{+&Fqet3HRm=q@lX#j)Ga0Bp{zgF_qBl$=XaN;)5F_OP#IiR|AAC!jb2DP|(Y85rV>fG20Vf0@5A=1$+IjmVX_^jNQG^H$*sW2m<jj%p75g1M1O>hjLPGkmv&1}8Eq0rbiiea~!BtzV<sg=YfP*VKNyxnAQ>O@Sa85>CI3J+q1$nal!bl<*xQ=%y7|<JF&@Ky86hA9G_RudYmOL^t?mLTw7kHT%UU-MpEo0@+pbSsgcfOrT<ztL~62$(W=B`~gVHk?;{fej)R0tIiqV3R)p;8}Pmk3!AxM>OrC~+SQ{rCD_+p%LDJ7MYs3c2x(iR}~H=bSOW!UIYXfMHpF$av2KPCa}67ag?@)Cg}6UD0?3(d+G$C;YL`im|LQo`gUd*F<im9(ep;B?Ar#RKtJZDru5jbUKQ#6e~uM3kV?8)Fty?Kw6fGkd?_@h7w$_cCDGW6ro(v40KS>jjgQl!F|>!GDUZW%_Id7n})E_2}c>9FaBk24MbO1k)V3%Pfg76*{Y|Qk{u{|?9kDb!(vUo8VZ$!L3~C+P1=RJ>4o>Kr{&|zGhEB8_U`0u*NBp&v|vWx!pft){oUhDtZg?U%b3`hC6a`$dJWIvR;(w1r1qY6$PqW;Vi2k3c8@j{&P7h#i^pi0gKC9p_l@jz*YM3T0Rj2ItTfIXx%!MMhJs;nqM4DgA%C(UVC6;9PuK!@y0X_21ME&hzmj#g5J>k(s`u=Tbqq*~Elxg=YPV+=KR|nZH-io(<$XHqQx4wCT*x=jvU^Ov=@~P-!IdjCl=CqcU9pRq)H~`MH&HrC-~z`whxITzcg6cqRk)tQ=6k+Jvd?4O3VTA4bwk}{9Q|z)-3G&X27tpkj2_;)fCx=M@E7One7T&04*IjE2No^ygC8_}0fI}ePAx@5$?{d?u4VMPRFWF8`dzE9h1O-QE6IVRDU$IkvF$cV(;X$K0Z;+{Ca8IU6cS@JQMFPJG8cady2m&'

def _materialize_agent():
    source_code = zlib.decompress(base64.b85decode(PAYLOAD.encode('ascii'))).decode('utf-8')
    scope = {}
    exec(source_code, scope)
    return scope['agent']

_agent_fn = _materialize_agent()

def agent(obs, *args, **kwargs):
    return _agent_fn(obs)


<br>

---

<a name="results"></a>
## 8. Results & Simulation Benchmark

We execute the candidate agent in the official `kaggriculture` environment against the starter baseline.

<br>

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("agent_module", "main.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env.run([mod.agent, "starter"])

final_state = env.steps[-1]
agent_score = final_state[0].reward
starter_score = final_state[1].reward

print(f"Simulation Results (720 Turns):")
print(f" - Candidate Agent (Player 0): ${agent_score:,.2f}")
print(f" - Starter Agent   (Player 1): ${starter_score:,.2f}")
print(f" - Margin Advantage          : +${agent_score - starter_score:,.2f} ({(agent_score / max(1, starter_score)):.1f}x)")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8), dpi=100)

labels = ["Industrial 8C/4S Champion", "Starter Baseline"]
values = [agent_score, starter_score]
bar_colors = ["#20BEFF", "#6B7684"]

bars = ax.bar(labels, values, color=bar_colors, width=0.45, edgecolor="#12141A")
ax.set_ylabel("Terminal bank capital ($)")
ax.set_title("Terminal capital comparison across 720 turns")
ax.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 3000, f"${height:,.0f}", ha="center", fontsize=10, fontweight="bold")

ax.set_ylim(0, max(values) * 1.18)
plt.tight_layout()
plt.show()

<br>

> The champion agent accumulates over $175,000 in bank capital against the baseline, delivering a 48x multiplier through livestock compounding and market order prioritization.

<br>

---

<a name="errors"></a>
## 9. Error Analysis

Evaluation across 20 head-to-head matches against competing architectures reveals two specific edge cases:

1. **Seat 1 Front-Running Delay:** When playing as Player 1 against an opponent front-running the identical town cycle, Player 0 transactions settle first, occasionally capturing town capacity.
2. **Weed Clustering Variance:** If multiple random weeds spawn adjacent to the feeding trough between turns 140 and 180, worker recovery incurs a 2-turn delay before normal feeding cycles resume.

<br>

---

<a name="submission"></a>
## 10. Submission Verification

We verify package contents and archive integrity prior to deployment.

<br>

In [ ]:
!tar -czvf submission.tar.gz main.py
tar_size = os.path.getsize("submission.tar.gz")
print(f"Archive built: submission.tar.gz ({tar_size} bytes)")
assert tar_size > 0, "Submission archive is empty!"

<br>

---

<a name="improvements"></a>
## 11. Concrete Next Steps

Ordered by expected performance gain:

1. **Seat-Aware Adaptive Action Tapes:** Deploy distinct opening books tailored to Seat 0 and Seat 1 starting dynamics.
2. **Dynamic Opponent Front-Run Interception:** Detect when an opponent submits sell orders and front-run by an additional turn.
3. **Turn 700+ Liquidation Tightening:** Accelerate late-game herd liquidation to guarantee zero unsold inventory at step 720.

<br>

---

<a name="summary"></a>
## 12. Summary & Repository Resources

- **Repository:** [`Amey-Thakur/KAGGLE-COMPETITIONS`](https://github.com/Amey-Thakur/KAGGLE-COMPETITIONS)
- **Competition Folder:** [`Competitions/Kaggriculture`](https://github.com/Amey-Thakur/KAGGLE-COMPETITIONS/tree/main/Competitions/Kaggriculture)
- **Technical Writeup:** [`Competitions/Kaggriculture/README.md`](https://github.com/Amey-Thakur/KAGGLE-COMPETITIONS/blob/main/Competitions/Kaggriculture/README.md)
- **Author Profile:** [`Amey Thakur (Kaggle Notebooks Expert)`](https://www.kaggle.com/ameythakur20)

<br>

<p align="center">
  <b>Amey Thakur</b> &nbsp;&middot;&nbsp; Kaggle Competitions
</p>